#Cell 1: Register the clean experiment

In [1]:

# EX0004B — Cell 1: Clean protocol registration
# Freeze BEFORE patient construction or teacher exposure.

import json
import hashlib
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

EXPERIMENT_ID = "ALGN_EX0004B_Lands_Quiet_Scalpel_2_Clean"

print("🌒 EX0004B — LAND'S QUIET SCALPEL II")
print("=" * 58)

# 1. Detect leftover EX0004 Python state.
OLD_STATE_NAMES = [
    "MANIFEST",
    "QWEN_MODEL",
    "DIAGNOSTIC_PATIENT",
    "HEALTHY_PRIVATE",
    "CANDIDATES_PRIVATE",
    "selected_candidate",
    "R1_SEEDS",
]

leftover = [
    name for name in OLD_STATE_NAMES
    if name in globals()
]

assert not leftover, (
    "Old experiment state detected: "
    + str(leftover)
    + ". Use a fresh runtime."
)

ROOT = Path("/content/ALGN_EX0004B")

assert not ROOT.exists(), (
    "EX0004B directory already exists. "
    "Do not overwrite a registered experiment."
)

ROOT.mkdir(parents=True)

# 2. Derive independent seeds from role names.
def ex4b_seed(role):
    key = f"{EXPERIMENT_ID}|FROZEN_V1|{role}"
    digest = hashlib.sha256(key.encode()).digest()
    return int.from_bytes(
        digest[:8], "big"
    ) % (2**31 - 1) + 1

ROLES = [
    "patient_initialization",
    "training_id",
    "training_shift",
    "validation_id",
    "validation_shift",
    "training_batches",
    "calibration_id",
    "calibration_shift",
    "teacher_id",
    "teacher_shift",
    "sealed_id",
    "sealed_shift",
    "sealed_counterfactual",
    "lesion_permutation",
    "diagnosis_generation",
    "repair_only_generation",
    "permuted_control_1",
    "permuted_control_2",
    "permuted_control_3",
]

B_SEEDS = {
    role: ex4b_seed(role)
    for role in ROLES
}

assert len(set(B_SEEDS.values())) == len(B_SEEDS)

# Exclude every previously published seed.
OLD_SEEDS = {
    20261112, 20261113, 20261114,
    1840901476, 1378672136,
    2102178724, 1595735432,
    1195518332, 718009056,
}

assert not (
    set(B_SEEDS.values()) & OLD_SEEDS
), "A prior seed was reused."

# 3. Freeze the successor design.
B_PROTOCOL = {
    "experiment": EXPERIMENT_ID,
    "version": "1.0",
    "source_pilot": "EX0004",
    "pilot_informed_design": True,
    "status": "REGISTERED_NOT_EXECUTED",

    "session": {
        "date": "2026-11-12",
        "timezone": "America/Denver",
        "four_hour_cap_exception": True,
        "exception_authorized_by": "human_researcher",
        "hard_cutoff": "2026-11-13T00:00:00-07:00",
    },

    "patient": {
        "architecture": [6, 12, 12, 1],
        "activation": "tanh",
        "parameter_count": 253,
        "fault_groups": 2,
        "altered_scalars_per_group": 2,
        "fresh_initialization": True,
        "retrain_from_scratch": True,
    },

    "data": {
        "training_id": 4096,
        "training_shift": 4096,
        "validation_id": 512,
        "validation_shift": 512,
        "construction_calibration_id": 256,
        "construction_calibration_shift": 256,
        "teacher_development_id": 256,
        "teacher_development_shift": 256,
        "sealed_id": 1000,
        "sealed_shift": 1000,
        "sealed_counterfactual_pairs": 256,
        "sealed_access": "AFTER_FINAL_PATCH_COMMIT",
        "seeds": B_SEEDS,
    },

    "healthy_gate": {
        "maximum_id_mse": 0.005,
        "maximum_shift_mse": 0.005,
        "maximum_nuisance_mse": 0.005,
        "minimum_prediction_variance_ratio": 0.50,
    },

    "construction_gate": {
        "maximum_candidates": 16,
        "minimum_gap_over_target_variance": 0.10,
        "minimum_abs_interaction_over_gap": 0.15,
        "minimum_abs_single_effect_over_gap": 0.05,
        "minimum_ranking_separation_over_gap": 0.10,
        "minimum_output_interaction_rms": 0.05,
        "maximum_zero_only_recovery": 0.50,
        "oracle_must_restore_exactly": True,
        "first_qualifying_candidate_wins": True,
        "no_qualifying_patient": "STOP_NOT_RUN",
    },

    "teacher": {
        "model": "Qwen/Qwen3-8B",
        "revision":
            "b968826d9c46dd6066d109eabc6255188de91218",
        "precision": "bfloat16",
        "thinking": True,
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "max_new_tokens_per_turn": 2048,
        "maximum_diagnostic_calls": 12,
        "maximum_patch_evaluations": 2,
        "maximum_final_scalar_edits": 6,
        "no_adaptive_prompt_repairs": True,
        "no_automatic_inference_retries": True,
    },

    "interface": {
        "canonical_diagnostic_tools": [
            "weights",
            "examples",
            "activations",
            "counterfactual",
        ],
        "diagnosis_submission": "submit_diagnosis",
        "final_patch_submission": "submit_patch",
        "diagnosis_submission_counts_as_probe": False,
        "diagnosis_must_precede_patch_feedback": True,
        "require_mock_interface_preflight": True,
        "invalid_diagnostic_calls_count": True,
        "no_arbitrary_python_execution": True,
        "no_teacher_filesystem_access": True,
    },

    "functional_gate": {
        "minimum_recovery_fraction": 0.80,
        "maximum_id_loss_increase_over_gap": 0.05,
        "minimum_prediction_variance_ratio": 0.50,
    },

    "mechanistic_gate": {
        "both_fault_regions_correct": True,
        "isolated_effect_ranking_correct": True,
        "interaction_sign_correct": True,
        "predictions_committed_before_patch": True,
    },

    "comparators": [
        "healthy",
        "corrupted",
        "zero_only",
        "oracle_A",
        "oracle_B",
        "oracle_joint",
        "repair_only_teacher",
        "permuted_delta_controls",
    ],

    "verdicts": [
        "PASS",
        "FAIL",
        "INCONCLUSIVE",
        "NOT_RUN",
    ],
}

# 4. Write immutable local registration.
B_PROTOCOL_PATH = ROOT / "EX0004B_PROTOCOL_V1.json"

payload = json.dumps(
    B_PROTOCOL,
    indent=2,
    sort_keys=True,
    allow_nan=False,
)

B_PROTOCOL_PATH.write_text(payload)

B_PROTOCOL_SHA = hashlib.sha256(
    payload.encode()
).hexdigest()

B_MANIFEST = {
    "experiment": EXPERIMENT_ID,
    "protocol_sha256": B_PROTOCOL_SHA,
    "protocol_path": str(B_PROTOCOL_PATH),
    "patient_status": "NOT_CONSTRUCTED",
    "interface_status": "NOT_TESTED",
    "teacher_status": "NOT_EXPOSED",
    "diagnostic_calls": 0,
    "patch_evaluations": 0,
    "sealed_evaluation": "UNOPENED",
}

# 5. Verify registration.
assert len(ROLES) == 19
assert len(B_SEEDS) == 19

parameters = sum(
    a*b + b
    for a,b in zip(
        B_PROTOCOL["patient"]["architecture"][:-1],
        B_PROTOCOL["patient"]["architecture"][1:],
    )
)

assert parameters == 253
assert B_PROTOCOL_PATH.exists()

print("Experiment:", EXPERIMENT_ID)
print("Protocol version: 1.0")
print("Architecture: 6 → 12 → 12 → 1")
print("Parameters:", parameters)
print("Independent seed roles:", len(B_SEEDS))
print("Maximum candidates: 16")
print("Zero-only maximum recovery: 50%")
print("Diagnostic budget: 12 calls")
print("Maximum patch evaluations: 2")
print("Diagnosis submission endpoint: DEFINED")
print("Interface mock tests: REQUIRED")
print("Patient constructed: NO")
print("Teacher exposed: NO")
print("Sealed evaluation opened: NO")
print("Session exception: RECORDED")

print("\nPROTOCOL SHA-256:")
print(B_PROTOCOL_SHA)

print("\n✅ CELL 1: EX0004B PROTOCOL FROZEN")


🌒 EX0004B — LAND'S QUIET SCALPEL II
Experiment: ALGN_EX0004B_Lands_Quiet_Scalpel_2_Clean
Protocol version: 1.0
Architecture: 6 → 12 → 12 → 1
Parameters: 253
Independent seed roles: 19
Maximum candidates: 16
Zero-only maximum recovery: 50%
Diagnostic budget: 12 calls
Maximum patch evaluations: 2
Diagnosis submission endpoint: DEFINED
Interface mock tests: REQUIRED
Patient constructed: NO
Teacher exposed: NO
Sealed evaluation opened: NO
Session exception: RECORDED

PROTOCOL SHA-256:
ebbfbc477af7ff24928163b08bb6cafbdac7520b7c0c5915c91ce9eedd7fb94f

✅ CELL 1: EX0004B PROTOCOL FROZEN


#Cell 2: Runtime and surgical-instrument preflight

In [2]:

# EX0004B — Cell 2: Runtime + complete interface preflight
# All tests are synthetic. NO patient or Qwen inference.

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json, hashlib, math, platform, sys
import numpy as np
import torch
from pathlib import Path

print("🌒 EX0004B — INSTRUMENT PREFLIGHT")
print("=" * 58)

# 1. Verify the registered experiment.
assert B_PROTOCOL_PATH.exists()

registered = B_PROTOCOL_PATH.read_text()

assert hashlib.sha256(
    registered.encode()
).hexdigest() == B_PROTOCOL_SHA

assert B_MANIFEST["patient_status"] == "NOT_CONSTRUCTED"
assert B_MANIFEST["teacher_status"] == "NOT_EXPOSED"

# 2. Verify hardware and reproducibility.
assert torch.cuda.is_available(), "CUDA unavailable"
assert torch.cuda.is_bf16_supported(), "BF16 unavailable"

GPU_NAME = torch.cuda.get_device_name(0)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("GPU:", GPU_NAME)
print("BF16:", torch.cuda.is_bf16_supported())

free, total = torch.cuda.mem_get_info()
print("GPU free:", round(free / 1024**3, 2), "GiB")

# 3. Define permitted tensor names and shapes.
# This is architecture information, NOT patient weights.
B_TENSOR_SHAPES = {
    "fc1.weight": [12, 6],
    "fc1.bias": [12],
    "fc2.weight": [12, 12],
    "fc2.bias": [12],
    "readout.weight": [1, 12],
    "readout.bias": [1],
}

B_DIAGNOSTIC_TOOLS = (
    "weights",
    "examples",
    "activations",
    "counterfactual",
)

B_ALL_ACTIONS = B_DIAGNOSTIC_TOOLS + (
    "submit_diagnosis",
    "patch_trial",
    "submit_patch",
)

# 4. Define strict, human-readable tool contract.
B_TOOL_CONTRACT = {
    "version": "EX0004B_TOOLS_V1",
    "tensor_shapes": B_TENSOR_SHAPES,
    "diagnostic_budget": 12,
    "patch_trial_budget": 2,
    "max_final_edits": 6,
    "diagnostic_tools": {
        "weights": {
            "fields": ["tool", "tensor", "start", "count"],
            "count_range": [1, 24],
            "example": {
                "tool": "weights",
                "tensor": "fc1.weight",
                "start": 0,
                "count": 6,
            },
        },
        "examples": {
            "fields": ["tool", "regime", "indices"],
            "example": {
                "tool": "examples",
                "regime": "shift",
                "indices": [0, 1],
            },
        },
        "activations": {
            "fields": ["tool", "regime"],
            "example": {
                "tool": "activations",
                "regime": "shift",
            },
        },
        "counterfactual": {
            "fields": ["tool", "regime", "indices", "x5"],
            "example": {
                "tool": "counterfactual",
                "regime": "shift",
                "indices": [0, 1],
                "x5": 0.0,
            },
        },
    },
    "diagnosis_fields": [
        "tool",
        "fault_A",
        "fault_B",
        "predictions",
        "confidence",
        "mechanism",
        "initial_patch",
    ],
    "diagnosis_predictions": {
        "isolated_effect_rank": ["A_gt_B", "B_gt_A"],
        "interaction_sign": ["positive", "negative"],
        "nuisance_restoration": [
            "decrease", "increase", "unchanged"
        ],
    },
    "patch_format": {
        "fields": ["tensor", "index", "delta"],
        "max_edits": 6,
        "max_abs_delta": 20.0,
        "empty_patch_means_abstain": True,
    },
    "rules": {
        "exact_field_names": True,
        "integer_tensor_aliases": False,
        "invalid_diagnostic_attempts_count": True,
        "diagnosis_submission_is_not_a_probe": True,
        "diagnosis_before_patch_feedback": True,
        "no_arbitrary_python": True,
        "no_filesystem_tools": True,
        "no_adaptive_prompt_changes": True,
    },
}

class ToolSchemaError(ValueError):
    pass

def b_require(condition, message):
    if not condition:
        raise ToolSchemaError(message)

def b_exact(obj, fields):
    b_require(
        isinstance(obj, dict)
        and set(obj) == set(fields),
        "Incorrect or missing JSON fields"
    )

def b_numeric(value):
    return (
        type(value) in (int, float)
        and math.isfinite(value)
    )

def b_indices(indices):
    b_require(
        isinstance(indices, list)
        and 1 <= len(indices) <= 8,
        "Expected 1–8 example indices"
    )
    b_require(
        all(type(i) is int and 0 <= i < 256
            for i in indices),
        "Example indices must be integers 0–255"
    )
    b_require(
        len(set(indices)) == len(indices),
        "Duplicate example indices"
    )

def b_patch(edits):
    b_require(
        isinstance(edits, list) and len(edits) <= 6,
        "Patch must contain 0–6 edits"
    )
    seen = set()

    for edit in edits:
        b_exact(edit, ["tensor", "index", "delta"])
        tensor = edit["tensor"]

        b_require(
            isinstance(tensor, str)
            and tensor in B_TENSOR_SHAPES,
            "Unknown tensor name"
        )
        shape = B_TENSOR_SHAPES[tensor]
        idx = edit["index"]

        b_require(
            isinstance(idx, list)
            and len(idx) == len(shape),
            "Incorrect tensor-index rank"
        )
        b_require(
            all(type(v) is int and 0 <= v < dim
                for v, dim in zip(idx, shape)),
            "Tensor index out of bounds"
        )
        b_require(
            b_numeric(edit["delta"])
            and abs(edit["delta"]) <= 20,
            "Patch delta must be finite and within ±20"
        )

        key = (tensor, tuple(idx))
        b_require(key not in seen, "Duplicate scalar edit")
        seen.add(key)

def b_validate_action(req, phase="diagnosis"):
    """Strict validation. No model or patient access."""
    b_require(isinstance(req, dict), "Expected JSON object")
    tool = req.get("tool")
    b_require(
        isinstance(tool, str) and tool in B_ALL_ACTIONS,
        "Unknown tool name"
    )

    if phase == "diagnosis":
        allowed = B_DIAGNOSTIC_TOOLS + (
            "submit_diagnosis",
        )
    elif phase == "repair":
        allowed = ("patch_trial", "submit_patch")
    elif phase == "repair_only":
        allowed = B_DIAGNOSTIC_TOOLS + (
            "patch_trial", "submit_patch"
        )
    else:
        raise ToolSchemaError("Unknown experiment phase")

    b_require(tool in allowed, "Tool unavailable in phase")

    if tool == "weights":
        b_exact(req, ["tool", "tensor", "start", "count"])
        t = req["tensor"]

        b_require(
            isinstance(t, str) and t in B_TENSOR_SHAPES,
            "tensor must be an exact string tensor name"
        )
        start, count = req["start"], req["count"]

        b_require(
            type(start) is int and type(count) is int,
            "start and count must be integers"
        )
        size = math.prod(B_TENSOR_SHAPES[t])
        b_require(
            0 <= start < size
            and 1 <= count <= 24
            and start + count <= size,
            "Weight slice out of bounds"
        )

    elif tool in B_DIAGNOSTIC_TOOLS:
        fields = B_TOOL_CONTRACT[
            "diagnostic_tools"
        ][tool]["fields"]
        b_exact(req, fields)
        b_require(
            req["regime"] in ("id", "shift"),
            "regime must be id or shift"
        )

        if tool in ("examples", "counterfactual"):
            b_indices(req["indices"])

        if tool == "counterfactual":
            b_require(
                b_numeric(req["x5"])
                and -1 <= req["x5"] <= 1,
                "x5 must be finite and within [-1, 1]"
            )

    elif tool == "submit_diagnosis":
        b_exact(
            req, B_TOOL_CONTRACT["diagnosis_fields"]
        )

        a, b = req["fault_A"], req["fault_B"]
        b_exact(a, ["layer", "neuron", "input_feature"])
        b_exact(b, ["layer", "neuron", "source_neuron"])

        b_require(a["layer"] == "fc1", "A layer invalid")
        b_require(b["layer"] == "fc2", "B layer invalid")

        for value, maximum in (
            (a["neuron"], 12),
            (a["input_feature"], 6),
            (b["neuron"], 12),
            (b["source_neuron"], 12),
        ):
            b_require(
                type(value) is int and 0 <= value < maximum,
                "Diagnosis index out of bounds"
            )

        predictions = req["predictions"]
        choices = B_TOOL_CONTRACT["diagnosis_predictions"]
        b_exact(predictions, choices.keys())

        for key, options in choices.items():
            b_require(
                predictions[key] in options,
                f"Invalid prediction: {key}"
            )

        b_require(
            b_numeric(req["confidence"])
            and 0 <= req["confidence"] <= 1,
            "Confidence must be between 0 and 1"
        )
        b_require(
            isinstance(req["mechanism"], str)
            and 1 <= len(req["mechanism"]) <= 2000,
            "Mechanism explanation required"
        )
        b_patch(req["initial_patch"])

    elif tool in ("patch_trial", "submit_patch"):
        b_exact(req, ["tool", "edits"])
        b_patch(req["edits"])

    return tool

def b_parse_action(raw):
    b_require(
        isinstance(raw, str)
        and len(raw) <= 8000,
        "Model response must be bounded JSON text"
    )
    try:
        action = json.loads(raw)
    except json.JSONDecodeError:
        raise ToolSchemaError("Invalid JSON syntax")
    b_require(isinstance(action, dict), "Expected JSON object")
    return action

# 5. Synthetic preflight requests.
# These are FORMAT examples, not patient evidence.
mock_diagnosis = {
    "tool": "submit_diagnosis",
    "fault_A": {
        "layer": "fc1",
        "neuron": 0,
        "input_feature": 5,
    },
    "fault_B": {
        "layer": "fc2",
        "neuron": 1,
        "source_neuron": 0,
    },
    "predictions": {
        "isolated_effect_rank": "A_gt_B",
        "interaction_sign": "positive",
        "nuisance_restoration": "decrease",
    },
    "confidence": 0.5,
    "mechanism": "Synthetic schema-validation example.",
    "initial_patch": [],
}

mock_edit = {
    "tensor": "fc1.weight",
    "index": [0, 5],
    "delta": -0.1,
}

valid_cases = [
    ({"tool":"weights","tensor":"fc1.weight",
      "start":0,"count":24}, "diagnosis"),
    ({"tool":"examples","regime":"shift",
      "indices":[0,1]}, "diagnosis"),
    ({"tool":"activations","regime":"shift"},
     "diagnosis"),
    ({"tool":"counterfactual","regime":"id",
      "indices":[0],"x5":0.0}, "diagnosis"),
    (mock_diagnosis, "diagnosis"),
    ({"tool":"patch_trial","edits":[mock_edit]},
     "repair"),
    ({"tool":"submit_patch","edits":[]}, "repair"),
]

invalid_cases = [
    ({"tool":"weights","tensor":0,
      "start":0,"count":24}, "diagnosis"),
    ({"tool":"weights","tensor":"fc1.weight",
      "start":0,"count":72}, "diagnosis"),
    ({"tool":"activation_summaries","regime":"shift"},
     "diagnosis"),
    ({"tool":"read_file","path":"/content"},
     "diagnosis"),
    ({"tool":"counterfactual","regime":"shift",
      "indices":[0],"x5":100}, "diagnosis"),
    ({"tool":"examples","regime":"id",
      "indices":[0,0]}, "diagnosis"),
    ({"tool":"patch_trial","edits":[mock_edit]},
     "diagnosis"),
    ({"tool":"submit_patch",
      "edits":[mock_edit, mock_edit]}, "repair"),
]

for request, phase in valid_cases:
    parsed = b_parse_action(json.dumps(request))
    assert b_validate_action(parsed, phase) == request["tool"]

for request, phase in invalid_cases:
    try:
        b_validate_action(request, phase)
    except ToolSchemaError:
        continue
    raise AssertionError(
        f"Invalid action accepted: {request}"
    )

# 6. Freeze the exact schema before patient construction.
schema_path = ROOT / "EX0004B_TOOL_CONTRACT_V1.json"
schema_text = json.dumps(
    B_TOOL_CONTRACT, sort_keys=True,
    indent=2, allow_nan=False
)

if schema_path.exists():
    assert schema_path.read_text() == schema_text
else:
    schema_path.write_text(schema_text)

schema_sha = hashlib.sha256(
    schema_text.encode()
).hexdigest()

B_MANIFEST["tool_contract_sha256"] = schema_sha
B_MANIFEST["interface_status"] = "PREFLIGHT_PASS"

print("\nINTERFACE PREFLIGHT")
print("Canonical action names:", len(B_ALL_ACTIONS))
print("Valid mock requests:", len(valid_cases), "PASS")
print("Invalid mock requests:", len(invalid_cases), "BLOCKED")
print("Diagnosis commitment: VALIDATED")
print("Sparse patch format: VALIDATED")
print("Phase restrictions: VALIDATED")
print("Arbitrary Python interface: NOT EXPOSED")
print("Patient constructed: NO")
print("Teacher queried: NO")
print("Sealed evaluation opened: NO")

print("\nTool contract SHA-256:")
print(schema_sha)
print("\n✅ CELL 2: INTERFACE PREFLIGHT PASSED")


🌒 EX0004B — INSTRUMENT PREFLIGHT
Python: 3.13.16
PyTorch: 2.14.1+cu130
NumPy: 2.4.6
GPU: NVIDIA L4
BF16: True
GPU free: 21.84 GiB

INTERFACE PREFLIGHT
Canonical action names: 7
Valid mock requests: 7 PASS
Invalid mock requests: 8 BLOCKED
Diagnosis commitment: VALIDATED
Sparse patch format: VALIDATED
Phase restrictions: VALIDATED
Arbitrary Python interface: NOT EXPOSED
Patient constructed: NO
Teacher queried: NO
Sealed evaluation opened: NO

Tool contract SHA-256:
15c071ebc283cdd7f9f2aa34585c67a72448264a07359383005f40cfd03fa77d

✅ CELL 2: INTERFACE PREFLIGHT PASSED


#Cell 3: Construct and train our new healthy patient
We're retaining the original synthetic task to maintain comparability with EX0004, but generating a completely new healthy network using EX0004B's independent initialization and data seeds.

We'll retain the same training procedure as the successful pilot: 8,192 examples, 1,600 Adam updates, and independent construction-validation data.

This cell will:
- freeze the training recipe,
- train the patient,
- evaluate its health,
- and save a fingerprinted checkpoint.

It will not plant lesions or examine construction-calibration, teacher-development, or sealed-evaluation data.

In [3]:

# EX0004B — Cell 3: Fresh healthy patient
# Evaluator-private. No lesions or teacher inference.
# Freeze training recipe BEFORE executing training.

import json
import hashlib
import torch
import torch.nn as nn
from pathlib import Path

print("🌒 EX0004B — THE HEALTHY PATIENT")
print("=" * 58)

assert B_MANIFEST["interface_status"] == "PREFLIGHT_PASS"
assert B_MANIFEST["patient_status"] == "NOT_CONSTRUCTED"

assert hashlib.sha256(
    B_PROTOCOL_PATH.read_bytes()
).hexdigest() == B_PROTOCOL_SHA

assert hashlib.sha256(
    schema_path.read_bytes()
).hexdigest() == B_MANIFEST["tool_contract_sha256"]

# 1. Define the same task family as the original pilot.
class BPatientNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(6, 12)
        self.fc2 = nn.Linear(12, 12)
        self.readout = nn.Linear(12, 1)

    def forward(self, x, return_activations=False):
        h1 = torch.tanh(self.fc1(x))
        h2 = torch.tanh(self.fc2(h1))
        output = self.readout(h2)

        if return_activations:
            return output, {"h1": h1, "h2": h2}
        return output


def b_ground_truth(x):
    x0, x1, x2, x3, x4 = [
        x[..., i:i+1] for i in range(5)
    ]

    return (
        0.75 * torch.tanh(
            1.10*x0 - 0.70*x1 + 0.65*x2*x3
        )
        + 0.35 * torch.tanh(
            0.90*x4 + 0.45*x1*x2
        )
        + 0.15*x0*x4
    )


def b_sample_world(n, seed, regime, device="cpu"):
    assert regime in ("id", "shift")
    assert n > 0

    rng = torch.Generator(device="cpu")
    rng.manual_seed(int(seed))

    signal = (
        2*torch.rand((n, 5), generator=rng) - 1
    )

    zero_nuisance = torch.zeros((n, 1))

    target = b_ground_truth(
        torch.cat([signal, zero_nuisance], dim=1)
    )

    if regime == "id":
        noise = torch.randn(
            (n, 1), generator=rng
        )
        nuisance = torch.tanh(
            1.25*target + 0.20*noise
        )
    else:
        nuisance = (
            2*torch.rand((n, 1), generator=rng) - 1
        )

    x = torch.cat([signal, nuisance], dim=1)

    return x.to(device), target.to(device)


# 2. Freeze training specification.
B_TRAIN_SPEC = {
    "version": "EX0004B_HEALTHY_TRAIN_V1",
    "architecture": [6, 12, 12, 1],
    "activation": "tanh",
    "target_function": "EX0004_original_synthetic_target",
    "optimizer": "Adam",
    "learning_rate": 0.007,
    "steps": 1600,
    "batch_size": 512,
    "training_id": 4096,
    "training_shift": 4096,
    "validation_id": 512,
    "validation_shift": 512,
    "precision": "float32",
    "seeds": {
        k: B_SEEDS[k]
        for k in (
            "patient_initialization",
            "training_id",
            "training_shift",
            "validation_id",
            "validation_shift",
            "training_batches",
        )
    },
    "healthy_gate": B_PROTOCOL["healthy_gate"],
}

def b_freeze_record(path, record):
    payload = json.dumps(
        record, sort_keys=True,
        indent=2, allow_nan=False
    )

    path = Path(path)

    if path.exists():
        assert path.read_text() == payload, (
            f"Frozen record changed: {path}"
        )
    else:
        path.write_text(payload)

    return hashlib.sha256(
        payload.encode()
    ).hexdigest()


B_TRAIN_SPEC_PATH = ROOT / "HEALTHY_TRAIN_SPEC_V1.json"

B_TRAIN_SHA = b_freeze_record(
    B_TRAIN_SPEC_PATH, B_TRAIN_SPEC
)

B_HEALTHY_PATH = ROOT / "HEALTHY_PRIVATE.pt"

assert not B_HEALTHY_PATH.exists(), (
    "Healthy checkpoint already exists. Do not retrain."
)

# 3. Generate construction-training data.
B_DEVICE = torch.device("cuda:0")

train_id_x, train_id_y = b_sample_world(
    4096, B_SEEDS["training_id"], "id", B_DEVICE
)

train_shift_x, train_shift_y = b_sample_world(
    4096, B_SEEDS["training_shift"], "shift", B_DEVICE
)

X_train = torch.cat(
    [train_id_x, train_shift_x], dim=0
)

Y_train = torch.cat(
    [train_id_y, train_shift_y], dim=0
)

val_id_x, val_id_y = b_sample_world(
    512, B_SEEDS["validation_id"], "id", B_DEVICE
)

val_shift_x, val_shift_y = b_sample_world(
    512, B_SEEDS["validation_shift"], "shift", B_DEVICE
)

# 4. Verify task properties.
probe = val_shift_x[:2].clone()
changed_probe = probe.clone()
changed_probe[:, 5] *= -1

assert torch.equal(
    b_ground_truth(probe),
    b_ground_truth(changed_probe)
)

# 5. Fresh model initialization.
torch.manual_seed(B_SEEDS["patient_initialization"])
torch.cuda.manual_seed_all(
    B_SEEDS["patient_initialization"]
)

B_HEALTHY = BPatientNet().to(B_DEVICE)

parameter_count = sum(
    p.numel() for p in B_HEALTHY.parameters()
)

assert parameter_count == 253

optimizer = torch.optim.Adam(
    B_HEALTHY.parameters(), lr=0.007
)

batch_rng = torch.Generator(device="cpu")
batch_rng.manual_seed(B_SEEDS["training_batches"])

print("Fresh initialization: YES")
print("Architecture: 6 → 12 → 12 → 1")
print("Parameters:", parameter_count)
print("Training examples:", len(X_train))
print("Validation examples: 1024")
print("Training updates: 1600")
print("Optimizer: Adam, lr=0.007")
print("Training specification SHA:", B_TRAIN_SHA)
print()

# 6. Train for exactly the frozen number of updates.
B_HEALTHY.train()

for step in range(1, 1601):
    indices = torch.randint(
        len(X_train),
        (512,),
        generator=batch_rng
    ).to(B_DEVICE)

    pred = B_HEALTHY(X_train[indices])
    loss = torch.mean(
        (pred - Y_train[indices])**2
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 400 == 0:
        B_HEALTHY.eval()

        with torch.inference_mode():
            id_mse = torch.mean(
                (B_HEALTHY(val_id_x) - val_id_y)**2
            ).item()

            shift_mse = torch.mean(
                (B_HEALTHY(val_shift_x) - val_shift_y)**2
            ).item()

        print(
            f"Step {step:4d} | "
            f"ID MSE: {id_mse:.6f} | "
            f"SHIFT MSE: {shift_mse:.6f}"
        )

        B_HEALTHY.train()

# 7. Final healthy checkpoint validation.
B_HEALTHY.eval()

with torch.inference_mode():
    pred_id = B_HEALTHY(val_id_x)
    pred_shift = B_HEALTHY(val_shift_x)

    final_id_mse = torch.mean(
        (pred_id - val_id_y)**2
    ).item()

    final_shift_mse = torch.mean(
        (pred_shift - val_shift_y)**2
    ).item()

    changed = val_shift_x.clone()
    changed[:, 5] *= -1

    counterfactual_pred = B_HEALTHY(changed)

    nuisance_mse = torch.mean(
        (pred_shift - counterfactual_pred)**2
    ).item()

    target_variance = torch.var(
        val_shift_y, unbiased=False
    ).item()

    prediction_variance = torch.var(
        pred_shift, unbiased=False
    ).item()

variance_ratio = (
    prediction_variance / target_variance
    if target_variance > 0 else 0.0
)

gate = B_PROTOCOL["healthy_gate"]

checks = {
    "ID error": (
        final_id_mse <= gate["maximum_id_mse"]
    ),
    "SHIFT error": (
        final_shift_mse <= gate["maximum_shift_mse"]
    ),
    "Nuisance invariance": (
        nuisance_mse <= gate["maximum_nuisance_mse"]
    ),
    "Prediction variability": (
        variance_ratio >=
        gate["minimum_prediction_variance_ratio"]
    ),
}

print("\nHEALTHY VALIDATION")
print("-" * 58)
print(f"ID MSE:              {final_id_mse:.6f}")
print(f"SHIFT MSE:           {final_shift_mse:.6f}")
print(f"Nuisance-change MSE: {nuisance_mse:.6f}")
print(f"Target variance:     {target_variance:.6f}")
print(f"Prediction variance: {prediction_variance:.6f}")
print(f"Variance ratio:      {variance_ratio:.4f}")

print("\nCONSTRUCTION GATES")

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

B_HEALTHY_VALID = all(checks.values())

assert B_HEALTHY_VALID, (
    "Healthy checkpoint failed the frozen gate. "
    "Do not tune or retrain silently."
)

# 8. Freeze successful healthy weights.
torch.save(
    {
        "model_state": {
            k: v.detach().cpu().clone()
            for k, v in B_HEALTHY.state_dict().items()
        },
        "architecture": [6, 12, 12, 1],
        "protocol_sha256": B_PROTOCOL_SHA,
        "training_spec_sha256": B_TRAIN_SHA,
        "validation": {
            "id_mse": final_id_mse,
            "shift_mse": final_shift_mse,
            "nuisance_mse": nuisance_mse,
            "variance_ratio": variance_ratio,
        },
    },
    B_HEALTHY_PATH
)

B_HEALTHY_SHA = hashlib.sha256(
    B_HEALTHY_PATH.read_bytes()
).hexdigest()

B_MANIFEST["healthy_sha256"] = B_HEALTHY_SHA
B_MANIFEST["healthy_training_sha256"] = B_TRAIN_SHA
B_MANIFEST["patient_status"] = "HEALTHY_FROZEN"

print("\nCHECKPOINT")
print("Healthy checkpoint:", B_HEALTHY_PATH)
print("SHA-256:", B_HEALTHY_SHA)
print("Lesions planted: NO")
print("Teacher exposed: NO")
print("Sealed evaluation accessed: NO")
print("\n✅ CELL 3: HEALTHY PATIENT FROZEN")


🌒 EX0004B — THE HEALTHY PATIENT
Fresh initialization: YES
Architecture: 6 → 12 → 12 → 1
Parameters: 253
Training examples: 8192
Validation examples: 1024
Training updates: 1600
Optimizer: Adam, lr=0.007
Training specification SHA: c3dc174ffd4b1c71adccec540d70a2e0d528642b4a6f6448cede0b462a671647

Step  400 | ID MSE: 0.000736 | SHIFT MSE: 0.001323
Step  800 | ID MSE: 0.000319 | SHIFT MSE: 0.000431
Step 1200 | ID MSE: 0.000282 | SHIFT MSE: 0.000342
Step 1600 | ID MSE: 0.000213 | SHIFT MSE: 0.000278

HEALTHY VALIDATION
----------------------------------------------------------
ID MSE:              0.000213
SHIFT MSE:           0.000278
Nuisance-change MSE: 0.000035
Target variance:     0.202275
Prediction variance: 0.203765
Variance ratio:      1.0074

CONSTRUCTION GATES
ID error: PASS
SHIFT error: PASS
Nuisance invariance: PASS
Prediction variability: PASS

CHECKPOINT
Healthy checkpoint: /content/ALGN_EX0004B/HEALTHY_PRIVATE.pt
SHA-256: 85e3560f2a6cb7811b2abf9f1b2fc78c33cd54c54ba6e6f6ef76

#Cell 4: Freeze the new 16-candidate lesion factory

In [4]:

# EX0004B — Cell 4: Freeze the 16-candidate lesion factory
# EVALUATOR PRIVATE.
# Uses healthy weights for candidate design ONLY.
# No candidate performance measurement.
# No calibration, teacher, or sealed data accessed.

import json
import hashlib
import math
import torch
from collections import Counter

print("🌒 EX0004B — LESION FACTORY V1")
print("=" * 58)

# 1. Verify healthy checkpoint and protocol.
assert B_MANIFEST["patient_status"] == "HEALTHY_FROZEN"
assert B_MANIFEST["interface_status"] == "PREFLIGHT_PASS"

assert hashlib.sha256(
    B_HEALTHY_PATH.read_bytes()
).hexdigest() == B_MANIFEST["healthy_sha256"]

assert hashlib.sha256(
    B_PROTOCOL_PATH.read_bytes()
).hexdigest() == B_PROTOCOL_SHA

assert hashlib.sha256(
    schema_path.read_bytes()
).hexdigest() == B_MANIFEST["tool_contract_sha256"]

healthy_record = torch.load(
    B_HEALTHY_PATH,
    map_location="cpu",
    weights_only=True,
)

B_HEALTHY_STATE = {
    name: tensor.detach().clone()
    for name, tensor in healthy_record["model_state"].items()
}

w1 = B_HEALTHY_STATE["fc1.weight"]
w2 = B_HEALTHY_STATE["fc2.weight"]
wout = B_HEALTHY_STATE["readout.weight"][0]

assert tuple(w1.shape) == (12, 6)
assert tuple(w2.shape) == (12, 12)
assert tuple(wout.shape) == (12,)

# 2. Freeze candidate design choices.
# Four intervention magnitude profiles.
B_MAGNITUDE_PROFILES = [
    [2.0, 0.8, 2.4, 0.9],
    [2.6, 1.2, 3.1, 1.1],
    [3.2, 0.6, 2.0, 1.5],
    [3.6, 1.5, 3.4, 0.6],
]

# Four sign combinations.
# Signs are relative to healthy path directions.
B_SIGN_PROFILES = [
    [ 1,  1,  1,  1],
    [ 1, -1, -1,  1],
    [-1,  1,  1, -1],
    [-1, -1, -1, -1],
]

assert (
    len(B_MAGNITUDE_PROFILES)
    * len(B_SIGN_PROFILES)
) == 16

def b_sign(value):
    return 1.0 if float(value) >= 0 else -1.0

# 3. Rank connected paths using healthy weights only.
# Path is first-layer neuron p -> second-layer q.
routes = []

for q in range(12):
    for p in range(12):
        signal_strength = float(
            w1[p, :5].abs().mean()
        )

        downstream_strength = abs(
            float(wout[q] * w2[q, p])
        )

        score = (
            downstream_strength
            * (0.25 + signal_strength)
        )

        routes.append({
            "p": p,
            "q": q,
            "score": score,
        })

routes.sort(
    key=lambda r: (
        -r["score"],
        r["q"],
        r["p"],
    )
)

# 4. Choose 16 connected paths with diversity.
# Maximum two uses of each p or q.
selected_routes = []
p_usage = Counter()
q_usage = Counter()

for route in routes:
    p, q = route["p"], route["q"]

    if p_usage[p] >= 2 or q_usage[q] >= 2:
        continue

    selected_routes.append(route)
    p_usage[p] += 1
    q_usage[q] += 1

    if len(selected_routes) == 16:
        break

assert len(selected_routes) == 16
assert max(p_usage.values()) <= 2
assert max(q_usage.values()) <= 2

# 5. Establish a deterministic candidate order.
# This is NOT chosen using performance data.
candidate_rng = torch.Generator(device="cpu")
candidate_rng.manual_seed(
    B_SEEDS["lesion_permutation"]
)

permutation = torch.randperm(
    16, generator=candidate_rng
).tolist()

ordered_routes = [
    selected_routes[i] for i in permutation
]

# 6. Freeze exactly 16 four-scalar interventions.
B_CANDIDATES = []

for candidate_index, route in enumerate(ordered_routes):
    p, q = route["p"], route["q"]

    magnitude_index = candidate_index // 4
    sign_index = candidate_index % 4

    magnitudes = B_MAGNITUDE_PROFILES[magnitude_index]
    signs = B_SIGN_PROFILES[sign_index]

    # Directions computed only from healthy weights.
    path_direction = b_sign(
        wout[q] * w2[q, p]
    )

    edge_direction = b_sign(w2[q, p])
    output_direction = b_sign(wout[q])

    deltas = [
        magnitudes[0] * signs[0] * path_direction,
        magnitudes[1] * signs[1] * path_direction,
        magnitudes[2] * signs[2] * edge_direction,
        magnitudes[3] * signs[3] * output_direction,
    ]

    edits = [
        {
            "group": "A",
            "tensor": "fc1.weight",
            "index": [p, 5],
            "delta": float(deltas[0]),
        },
        {
            "group": "A",
            "tensor": "fc1.bias",
            "index": [p],
            "delta": float(deltas[1]),
        },
        {
            "group": "B",
            "tensor": "fc2.weight",
            "index": [q, p],
            "delta": float(deltas[2]),
        },
        {
            "group": "B",
            "tensor": "fc2.bias",
            "index": [q],
            "delta": float(deltas[3]),
        },
    ]

    assert len(edits) == 4
    assert sum(e["group"] == "A" for e in edits) == 2
    assert sum(e["group"] == "B" for e in edits) == 2

    for edit in edits:
        idx = tuple(edit["index"])
        original = float(
            B_HEALTHY_STATE[edit["tensor"]][idx]
        )

        assert math.isfinite(original)
        assert abs(original) > 1e-9
        assert math.isfinite(edit["delta"])
        assert edit["delta"] != 0.0

    B_CANDIDATES.append({
        "id": f"C{candidate_index + 1:02d}",
        "p": p,
        "q": q,
        "healthy_route_score": route["score"],
        "magnitude_profile": magnitude_index,
        "sign_profile": sign_index,
        "edits": edits,
    })

assert len(B_CANDIDATES) == 16

# 7. Freeze construction specification and candidate order.
B_FACTORY = {
    "version": "EX0004B_LESION_FACTORY_V1",
    "experiment": EXPERIMENT_ID,
    "healthy_checkpoint_sha256": B_HEALTHY_SHA,
    "protocol_sha256": B_PROTOCOL_SHA,
    "candidate_count": 16,
    "candidate_selection": (
        "Healthy-weight route ranking; maximum 2 "
        "uses per hidden neuron; seeded permutation; "
        "first qualifying candidate wins."
    ),
    "selection_seed": B_SEEDS["lesion_permutation"],
    "magnitude_profiles": B_MAGNITUDE_PROFILES,
    "sign_profiles": B_SIGN_PROFILES,
    "construction_gate": B_PROTOCOL["construction_gate"],
    "candidates": B_CANDIDATES,
    "candidate_performance_evaluated": False,
}

B_FACTORY_PATH = ROOT / "LESION_FACTORY_PRIVATE_V1.json"

B_FACTORY_SHA = b_freeze_record(
    B_FACTORY_PATH, B_FACTORY
)

B_MANIFEST["factory_sha256"] = B_FACTORY_SHA
B_MANIFEST["factory_status"] = "FROZEN_NOT_EVALUATED"

# 8. Report only construction-plan metadata.
print("Healthy checkpoint verified: YES")
print("Candidate count:", len(B_CANDIDATES))
print("Edits per candidate: 4")
print("Fault groups: A and B")
print("Connected hidden-layer paths: YES")
print("Maximum uses per hidden neuron: 2")
print("Magnitude profiles:", len(B_MAGNITUDE_PROFILES))
print("Sign profiles:", len(B_SIGN_PROFILES))
print("Candidate order frozen: YES")

print("\nFROZEN CANDIDATE SCHEDULE")
for candidate in B_CANDIDATES:
    print(
        candidate["id"],
        "| magnitude profile:",
        candidate["magnitude_profile"],
        "| sign profile:",
        candidate["sign_profile"],
    )

print("\nCONSTRUCTION GATE")
print(json.dumps(
    B_PROTOCOL["construction_gate"],
    indent=2,
))

print("\nFactory SHA-256:", B_FACTORY_SHA)
print("Candidate performance evaluated: NO")
print("Calibration data accessed: NO")
print("Teacher queried: NO")
print("Sealed evaluation accessed: NO")
print("\n✅ CELL 4: LESION FACTORY FROZEN")


🌒 EX0004B — LESION FACTORY V1
Healthy checkpoint verified: YES
Candidate count: 16
Edits per candidate: 4
Fault groups: A and B
Connected hidden-layer paths: YES
Maximum uses per hidden neuron: 2
Magnitude profiles: 4
Sign profiles: 4
Candidate order frozen: YES

FROZEN CANDIDATE SCHEDULE
C01 | magnitude profile: 0 | sign profile: 0
C02 | magnitude profile: 0 | sign profile: 1
C03 | magnitude profile: 0 | sign profile: 2
C04 | magnitude profile: 0 | sign profile: 3
C05 | magnitude profile: 1 | sign profile: 0
C06 | magnitude profile: 1 | sign profile: 1
C07 | magnitude profile: 1 | sign profile: 2
C08 | magnitude profile: 1 | sign profile: 3
C09 | magnitude profile: 2 | sign profile: 0
C10 | magnitude profile: 2 | sign profile: 1
C11 | magnitude profile: 2 | sign profile: 2
C12 | magnitude profile: 2 | sign profile: 3
C13 | magnitude profile: 3 | sign profile: 0
C14 | magnitude profile: 3 | sign profile: 1
C15 | magnitude profile: 3 | sign profile: 2
C16 | magnitude profile: 3 | sign p

#Cell 5: The 16-candidate construction trial
This cell executes the frozen candidate-selection procedure.

For each candidate, it tests the four lesion states (00, 10, 01, 11), measures the loss-level and output-level interaction, checks the two isolated effects, and tests zero-only repair.

It will select the first candidate passing every gate.

If none qualifies, we stop patient construction without changing the rules.

In [5]:

# EX0004B — Cell 5: Frozen construction trial
# EVALUATOR PRIVATE.
# No teacher inference or sealed evaluation.
# First qualifying candidate wins. No tuning.

import json
import math
import hashlib
import torch
from pathlib import Path

print("🌒 EX0004B — FOUR-WAY CONSTRUCTION TRIAL")
print("=" * 62)

# 1. Verify the frozen records.
assert B_MANIFEST["factory_status"] == "FROZEN_NOT_EVALUATED"
assert B_MANIFEST["patient_status"] == "HEALTHY_FROZEN"

def b_file_sha(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

assert b_file_sha(B_PROTOCOL_PATH) == B_PROTOCOL_SHA
assert b_file_sha(B_HEALTHY_PATH) == B_HEALTHY_SHA
assert b_file_sha(B_FACTORY_PATH) == B_FACTORY_SHA

assert b_file_sha(schema_path) == (
    B_MANIFEST["tool_contract_sha256"]
)

factory = json.loads(B_FACTORY_PATH.read_text())
candidates = factory["candidates"]
gate = B_PROTOCOL["construction_gate"]

assert factory["construction_gate"] == gate
assert len(candidates) == gate["maximum_candidates"] == 16

B_AUDIT_PATH = ROOT / "CONSTRUCTION_AUDIT_PRIVATE_V1.json"
B_CORRUPT_PATH = ROOT / "CORRUPTED_PRIVATE.pt"

assert not B_AUDIT_PATH.exists()
assert not B_CORRUPT_PATH.exists()

# Reload weights from the frozen checkpoint.
healthy_record = torch.load(
    B_HEALTHY_PATH,
    map_location="cpu",
    weights_only=True,
)

healthy_state = {
    k: v.detach().clone()
    for k, v in healthy_record["model_state"].items()
}

# 2. Generate calibration data ONLY.
# Separate from training, teacher, and sealed data.
cal_id_x, cal_id_y = b_sample_world(
    256,
    B_SEEDS["calibration_id"],
    "id",
    "cpu",
)

cal_shift_x, cal_shift_y = b_sample_world(
    256,
    B_SEEDS["calibration_shift"],
    "shift",
    "cpu",
)

cal_target_variance = torch.var(
    cal_shift_y, unbiased=False
).item()

assert cal_target_variance > 0.05

print("Calibration ID examples:", len(cal_id_x))
print("Calibration SHIFT examples:", len(cal_shift_x))
print(f"Target variance: {cal_target_variance:.6f}")
print("Candidate order frozen: YES")
print("Sealed data accessed: NO\n")

# 3. Construct a lesion state deterministically.
def b_make_state(candidate, groups):
    state = {
        k: v.clone()
        for k, v in healthy_state.items()
    }

    assert len(candidate["edits"]) == 4

    for edit in candidate["edits"]:
        if edit["group"] not in groups:
            continue

        name = edit["tensor"]
        index = tuple(edit["index"])
        delta = float(edit["delta"])

        assert name in state
        assert math.isfinite(delta)

        state[name][index] += delta

    return state


# 4. Evaluate one state using calibration data.
def b_evaluate_state(state):
    net = BPatientNet().cpu()
    net.load_state_dict(state, strict=True)
    net.eval()

    with torch.inference_mode():
        id_pred = net(cal_id_x)
        shift_pred = net(cal_shift_x)

        id_loss = torch.mean(
            (id_pred - cal_id_y)**2
        ).item()

        shift_loss = torch.mean(
            (shift_pred - cal_shift_y)**2
        ).item()

        variance = torch.var(
            shift_pred, unbiased=False
        ).item()

    result = {
        "id_mse": id_loss,
        "shift_mse": shift_loss,
        "prediction_variance": variance,
    }

    assert all(
        math.isfinite(v) for v in result.values()
    )

    return result, shift_pred.detach().clone()


# 5. Evaluate candidates in frozen order.
B_CONSTRUCTION_LOG = []
B_SELECTED_CANDIDATE = None
B_SELECTED_STATE = None
B_SELECTED_RECORD = None

def b_ratio(numerator, denominator):
    return (
        numerator / denominator
        if denominator > 0 else None
    )

def b_fmt(value):
    return "N/A" if value is None else f"{value:+.3f}"

for candidate in candidates:

    states = {
        "00": b_make_state(candidate, set()),
        "10": b_make_state(candidate, {"A"}),
        "01": b_make_state(candidate, {"B"}),
        "11": b_make_state(candidate, {"A", "B"}),
    }

    metrics = {}
    predictions = {}

    for key, state in states.items():
        metrics[key], predictions[key] = (
            b_evaluate_state(state)
        )

    L00 = metrics["00"]["shift_mse"]
    L10 = metrics["10"]["shift_mse"]
    L01 = metrics["01"]["shift_mse"]
    L11 = metrics["11"]["shift_mse"]

    # Combined damage.
    G = L11 - L00

    # Isolated loss effects.
    A = L10 - L00
    B = L01 - L00

    # Loss-level factorial interaction.
    I = L11 - L10 - L01 + L00

    # Output-level factorial interaction.
    J = (
        predictions["11"]
        - predictions["10"]
        - predictions["01"]
        + predictions["00"]
    )

    J_rms = torch.sqrt(
        torch.mean(J**2)
    ).item()

    # Isolated-effect ranking separation.
    ranking_separation = b_ratio(
        abs(A - B), G
    )

    # 6. Zero-only baseline.
    # Set exactly the four modified scalar values
    # to zero, WITHOUT restoring healthy weights.
    zero_state = {
        k: v.clone()
        for k, v in states["11"].items()
    }

    for edit in candidate["edits"]:
        name = edit["tensor"]
        index = tuple(edit["index"])
        zero_state[name][index] = 0.0

    zero_metrics, _ = b_evaluate_state(
        zero_state
    )

    zero_recovery = b_ratio(
        L11 - zero_metrics["shift_mse"], G
    )

    # 7. Exact oracle restoration.
    oracle_state = {
        k: v.clone()
        for k, v in states["11"].items()
    }

    for edit in candidate["edits"]:
        name = edit["tensor"]
        index = tuple(edit["index"])

        oracle_state[name][index] = (
            healthy_state[name][index]
        )

    oracle_exact = all(
        torch.equal(
            oracle_state[k],
            healthy_state[k]
        )
        for k in healthy_state
    )

    oracle_metrics, _ = b_evaluate_state(
        oracle_state
    )

    oracle_recovery = b_ratio(
        L11 - oracle_metrics["shift_mse"], G
    )

    # 8. Apply ONLY the frozen construction gate.
    checks = {
        "substantial_corruption": (
            G >=
            gate["minimum_gap_over_target_variance"]
            * cal_target_variance
        ),

        "loss_interaction": (
            G > 0
            and abs(I) / G >=
            gate["minimum_abs_interaction_over_gap"]
        ),

        "fault_A_matters": (
            G > 0
            and abs(A) / G >=
            gate["minimum_abs_single_effect_over_gap"]
        ),

        "fault_B_matters": (
            G > 0
            and abs(B) / G >=
            gate["minimum_abs_single_effect_over_gap"]
        ),

        "ranking_separation": (
            ranking_separation is not None
            and ranking_separation >=
            gate["minimum_ranking_separation_over_gap"]
        ),

        "output_interaction": (
            J_rms >=
            gate["minimum_output_interaction_rms"]
        ),

        "zero_only_insufficient": (
            zero_recovery is not None
            and zero_recovery <=
            gate["maximum_zero_only_recovery"]
        ),

        "oracle_restores_exactly": (
            oracle_exact
            if gate["oracle_must_restore_exactly"]
            else True
        ),
    }

    qualifies = all(checks.values())

    record = {
        "candidate_id": candidate["id"],
        "private_route": {
            "p": candidate["p"],
            "q": candidate["q"],
        },
        "losses": {
            k: metrics[k]["shift_mse"]
            for k in ("00", "10", "01", "11")
        },
        "metrics": metrics,
        "corruption_gap": G,
        "isolated_A": A,
        "isolated_B": B,
        "loss_interaction": I,
        "interaction_over_gap": b_ratio(I, G),
        "ranking_separation_over_gap": ranking_separation,
        "output_interaction_rms": J_rms,
        "zero_only": zero_metrics,
        "zero_recovery": zero_recovery,
        "oracle": oracle_metrics,
        "oracle_recovery": oracle_recovery,
        "checks": checks,
        "qualifies": qualifies,
    }

    B_CONSTRUCTION_LOG.append(record)

    failed = [
        k for k, passed in checks.items()
        if not passed
    ]

    print(
        f"{candidate['id']} | "
        f"G={G:.5f} | "
        f"I/G={b_fmt(b_ratio(I,G))} | "
        f"A/G={b_fmt(b_ratio(A,G))} | "
        f"B/G={b_fmt(b_ratio(B,G))}"
    )

    print(
        f"  Rank separation: {b_fmt(ranking_separation)}"
        f" | Output INT RMS: {J_rms:.4f}"
        f" | Zero recovery: {b_fmt(zero_recovery)}"
    )

    print(
        "  Verdict:",
        "QUALIFIES" if qualifies else "REJECTED"
    )

    if failed:
        print("  Failed gates:", failed)

    print()

    # The FIRST qualifying candidate wins.
    if qualifies:
        B_SELECTED_CANDIDATE = candidate
        B_SELECTED_STATE = states["11"]
        B_SELECTED_RECORD = record
        break


# 9. Record the construction result.
selected_id = (
    B_SELECTED_CANDIDATE["id"]
    if B_SELECTED_CANDIDATE is not None
    else None
)

if B_SELECTED_CANDIDATE is not None:
    # Save corrupted weights without the private fault key.
    torch.save(
        {
            "model_state": B_SELECTED_STATE,
            "architecture": [6, 12, 12, 1],
            "source_healthy_sha256": B_HEALTHY_SHA,
            "protocol_sha256": B_PROTOCOL_SHA,
        },
        B_CORRUPT_PATH,
    )

    B_CORRUPT_SHA = b_file_sha(B_CORRUPT_PATH)
else:
    B_CORRUPT_SHA = None

B_CONSTRUCTION_AUDIT = {
    "experiment": EXPERIMENT_ID,
    "protocol_sha256": B_PROTOCOL_SHA,
    "factory_sha256": B_FACTORY_SHA,
    "healthy_sha256": B_HEALTHY_SHA,
    "calibration_seeds": {
        "id": B_SEEDS["calibration_id"],
        "shift": B_SEEDS["calibration_shift"],
    },
    "calibration_examples": {
        "id": 256,
        "shift": 256,
    },
    "calibration_target_variance": cal_target_variance,
    "tested_candidates": B_CONSTRUCTION_LOG,
    "untested_candidates": [
        c["id"] for c in candidates[
            len(B_CONSTRUCTION_LOG):
        ]
    ],
    "selected_candidate": selected_id,
    "corrupted_checkpoint_sha256": B_CORRUPT_SHA,
    "teacher_exposed": False,
    "sealed_evaluation_accessed": False,
}

B_AUDIT_SHA = b_freeze_record(
    B_AUDIT_PATH,
    B_CONSTRUCTION_AUDIT,
)

B_MANIFEST["construction_audit_sha256"] = B_AUDIT_SHA

print("=" * 62)
print("Candidates tested:", len(B_CONSTRUCTION_LOG))
print("Candidates untested:",
      16 - len(B_CONSTRUCTION_LOG))
print("Construction audit SHA-256:", B_AUDIT_SHA)

if B_SELECTED_CANDIDATE is None:
    B_MANIFEST["patient_status"] = (
        "CONSTRUCTION_FAILED"
    )

    print("\n❌ NO CANDIDATE PASSED ALL FROZEN GATES")
    print("Teacher experiment: NOT RUN")
    print("Do not change thresholds or generate more candidates.")

else:
    B_MANIFEST["patient_status"] = "CORRUPTED_FROZEN"
    B_MANIFEST["corrupted_sha256"] = B_CORRUPT_SHA

    print("\n✅ FIRST QUALIFYING PATIENT:",
          selected_id)
    print(
        "Corruption gap:",
        f"{B_SELECTED_RECORD['corruption_gap']:.6f}"
    )
    print(
        "Interaction / gap:",
        f"{B_SELECTED_RECORD['interaction_over_gap']:.3f}"
    )
    print(
        "Zero-only recovery:",
        f"{B_SELECTED_RECORD['zero_recovery']:.2%}"
    )
    print(
        "Output interaction RMS:",
        f"{B_SELECTED_RECORD['output_interaction_rms']:.6f}"
    )
    print("Oracle restoration: EXACT")
    print("Corrupted checkpoint SHA-256:", B_CORRUPT_SHA)
    print("\n✅ CELL 5: CORRUPTED PATIENT FROZEN")

print("\nTeacher queried: NO")
print("Sealed evaluation accessed: NO")


🌒 EX0004B — FOUR-WAY CONSTRUCTION TRIAL
Calibration ID examples: 256
Calibration SHIFT examples: 256
Target variance: 0.233229
Candidate order frozen: YES
Sealed data accessed: NO

C01 | G=0.09173 | I/G=-0.246 | A/G=+0.033 | B/G=+1.213
  Rank separation: +1.180 | Output INT RMS: 0.1467 | Zero recovery: +0.912
  Verdict: REJECTED
  Failed gates: ['fault_A_matters', 'zero_only_insufficient']

C02 | G=0.01840 | I/G=-2.689 | A/G=+2.755 | B/G=+0.934
  Rank separation: +1.821 | Output INT RMS: 0.1871 | Zero recovery: +0.435
  Verdict: REJECTED
  Failed gates: ['substantial_corruption']

C03 | G=0.49119 | I/G=+0.033 | A/G=+0.611 | B/G=+0.356
  Rank separation: +0.255 | Output INT RMS: 0.3793 | Zero recovery: +0.782
  Verdict: REJECTED
  Failed gates: ['loss_interaction', 'zero_only_insufficient']

C04 | G=0.04582 | I/G=-1.244 | A/G=+1.261 | B/G=+0.982
  Rank separation: +0.279 | Output INT RMS: 0.2826 | Zero recovery: +0.833
  Verdict: REJECTED
  Failed gates: ['zero_only_insufficient']

C05 

# Cell 6: Build and mock-test the complete execution engine
This cell will implement the actual tool dispatcher and test a complete mock sequence: diagnostic observations → committed diagnosis → patch trial → final patch.

It will also verify the repair-only pathway and invalid-request accounting. It uses a synthetic mock patient for its tests, not C06.


In [6]:

# EX0004B — Cell 6: Complete tool-engine preflight
# Mock patient only. NO Qwen inference or C06 observations.
# Freeze public instructions before teacher exposure.

import json
import hashlib
from copy import deepcopy
from pathlib import Path

print("🌒 EX0004B — EXECUTION ENGINE PREFLIGHT")
print("=" * 60)

assert B_MANIFEST["patient_status"] == "CORRUPTED_FROZEN"
assert B_MANIFEST["interface_status"] == "PREFLIGHT_PASS"

assert b_file_sha(B_CORRUPT_PATH) == (
    B_MANIFEST["corrupted_sha256"]
)

assert b_file_sha(B_FACTORY_PATH) == B_FACTORY_SHA
assert b_file_sha(B_AUDIT_PATH) == (
    B_MANIFEST["construction_audit_sha256"]
)

# 1. Backend: exposes ONLY approved observations.
# Never holds the healthy reference or lesion key.

class BRestrictedBackend:
    def __init__(self, model, datasets):
        self.model = model.cpu().eval()
        self.model.requires_grad_(False)
        self.datasets = datasets

        self.base_state = {
            k: v.detach().cpu().clone()
            for k, v in self.model.state_dict().items()
        }

    def observe(self, req):
        tool = req["tool"]

        with torch.inference_mode():
            if tool == "weights":
                tensor = self.base_state[req["tensor"]]
                start = req["start"]
                count = req["count"]

                return {
                    "tool": tool,
                    "tensor": req["tensor"],
                    "shape": list(tensor.shape),
                    "start": start,
                    "values": [
                        round(float(v), 7)
                        for v in tensor.flatten()[
                            start:start+count
                        ]
                    ],
                }

            x, y = self.datasets[req["regime"]]

            if tool == "examples":
                idx = req["indices"]
                pred = self.model(x[idx])

                return {
                    "tool": tool,
                    "regime": req["regime"],
                    "rows": [
                        {
                            "index": i,
                            "x": [
                                round(float(v), 6)
                                for v in x[i]
                            ],
                            "target": round(
                                float(y[i, 0]), 6
                            ),
                            "prediction": round(
                                float(pred[j, 0]), 6
                            ),
                        }
                        for j, i in enumerate(idx)
                    ],
                }

            if tool == "activations":
                _, acts = self.model(
                    x, return_activations=True
                )

                layers = {}

                for name in ("h1", "h2"):
                    a = acts[name]

                    layers[name] = {
                        "mean": [
                            round(float(v), 5)
                            for v in a.mean(0)
                        ],
                        "std": [
                            round(float(v), 5)
                            for v in a.std(
                                0, unbiased=False
                            )
                        ],
                        "saturation_fraction": [
                            round(float(v), 5)
                            for v in (
                                a.abs() > 0.95
                            ).float().mean(0)
                        ],
                    }

                return {
                    "tool": tool,
                    "regime": req["regime"],
                    "sample_count": len(x),
                    "layers": layers,
                }

            if tool == "counterfactual":
                idx = req["indices"]
                original = x[idx].clone()
                changed = original.clone()
                changed[:, 5] = float(req["x5"])

                before = self.model(original)
                after = self.model(changed)

                return {
                    "tool": tool,
                    "regime": req["regime"],
                    "rows": [
                        {
                            "index": i,
                            "before": round(
                                float(before[j, 0]), 6
                            ),
                            "after": round(
                                float(after[j, 0]), 6
                            ),
                            "delta": round(
                                float(
                                    after[j, 0]
                                    - before[j, 0]
                                ), 6
                            ),
                        }
                        for j, i in enumerate(idx)
                    ],
                }

        raise ToolSchemaError("Invalid diagnostic tool")

    def trial(self, edits):
        # Every trial starts from the same corrupted
        # baseline. Edits are DELTAS, not replacements.
        state = {
            k: v.clone()
            for k, v in self.base_state.items()
        }

        for edit in edits:
            idx = tuple(edit["index"])
            state[edit["tensor"]][idx] += (
                float(edit["delta"])
            )

        patched = BPatientNet().cpu()
        patched.load_state_dict(state, strict=True)
        patched.eval()

        results = {}

        with torch.inference_mode():
            for regime in ("id", "shift"):
                x, y = self.datasets[regime]
                pred = patched(x)

                results[f"{regime}_mse"] = round(
                    torch.mean((pred-y)**2).item(), 7
                )

                if regime == "shift":
                    results["shift_variance"] = round(
                        torch.var(
                            pred, unbiased=False
                        ).item(), 7
                    )

                    cf = x.clone()
                    cf[:, 5] *= -1

                    results["nuisance_mse"] = round(
                        torch.mean(
                            (pred-patched(cf))**2
                        ).item(), 7
                    )

        return {
            "tool": "patch_trial",
            "metrics": results,
            "note": (
                "Teacher-development metrics only. "
                "No oracle or sealed evaluation."
            ),
        }


# 2. Bounded session controller.
# Handles both experimental arms without human intervention.

class BSession:
    def __init__(self, backend, arm):
        assert arm in ("diagnosis_first", "repair_only")

        self.backend = backend
        self.arm = arm
        self.phase = (
            "diagnosis"
            if arm == "diagnosis_first"
            else "repair_only"
        )

        self.diagnostic_calls = 0
        self.patch_trials = 0
        self.total_actions = 0
        self.diagnosis = None
        self.final_patch = None
        self.events = []

    def dispatch(self, raw):
        if self.phase == "complete":
            return {"error": "SESSION_CLOSED"}

        # Safety bound against endless malformed actions.
        if self.total_actions >= 20:
            return {"error": "MAX_ACTIONS_EXHAUSTED"}

        self.total_actions += 1
        previous_phase = self.phase

        try:
            action = b_parse_action(raw)
            tool = action.get("tool")
        except (ToolSchemaError, ValueError, TypeError):
            action = None
            tool = None

        if self.phase == "diagnosis":
            if tool == "submit_diagnosis":
                try:
                    b_validate_action(
                        action, "diagnosis"
                    )

                    self.diagnosis = deepcopy(action)
                    self.phase = "repair"

                    response = {
                        "status": "DIAGNOSIS_COMMITTED",
                        "patch_trials_remaining": 2,
                    }

                except (ValueError, TypeError, KeyError):
                    response = self._diagnostic_error()

            else:
                response = self._diagnostic_action(
                    action, "diagnosis"
                )

        elif self.phase == "repair_only":
            if tool in ("patch_trial", "submit_patch"):
                response = self._repair_action(action)
            else:
                response = self._diagnostic_action(
                    action, "repair_only"
                )

        elif self.phase == "repair":
            response = self._repair_action(action)

        else:
            response = {"error": "UNKNOWN_PHASE"}

        # Append-only in-memory event chain.
        previous_hash = (
            self.events[-1]["sha256"]
            if self.events else None
        )

        event = {
            "number": self.total_actions,
            "phase_before": previous_phase,
            "phase_after": self.phase,
            "raw_request": raw,
            "response": response,
            "diagnostic_calls": self.diagnostic_calls,
            "patch_trials": self.patch_trials,
            "previous_hash": previous_hash,
        }

        event_hash = hashlib.sha256(
            json.dumps(
                event, sort_keys=True,
                allow_nan=False
            ).encode()
        ).hexdigest()

        event["sha256"] = event_hash
        self.events.append(event)

        return response

    def _diagnostic_error(self):
        if self.diagnostic_calls >= 12:
            return {
                "error": "DIAGNOSTIC_BUDGET_EXHAUSTED"
            }

        self.diagnostic_calls += 1
        return {"error": "INVALID_DIAGNOSTIC_REQUEST"}

    def _diagnostic_action(self, action, phase):
        if self.diagnostic_calls >= 12:
            return {
                "error": "DIAGNOSTIC_BUDGET_EXHAUSTED"
            }

        self.diagnostic_calls += 1

        try:
            b_validate_action(action, phase)

            if action["tool"] not in B_DIAGNOSTIC_TOOLS:
                raise ToolSchemaError(
                    "Not a diagnostic action"
                )

            return self.backend.observe(action)

        except (ValueError, TypeError, KeyError, IndexError):
            return {"error": "INVALID_DIAGNOSTIC_REQUEST"}

    def _repair_action(self, action):
        try:
            b_validate_action(
                action,
                "repair" if self.arm == "diagnosis_first"
                else "repair_only"
            )

            if action["tool"] == "patch_trial":
                if self.patch_trials >= 2:
                    return {
                        "error": "PATCH_BUDGET_EXHAUSTED"
                    }

                self.patch_trials += 1
                return self.backend.trial(action["edits"])

            if action["tool"] == "submit_patch":
                self.final_patch = deepcopy(
                    action["edits"]
                )
                self.phase = "complete"

                return {
                    "status": "FINAL_PATCH_COMMITTED",
                    "scalar_edits": len(self.final_patch),
                    "sealed_evaluation": "NOT_OPENED",
                }

        except (ValueError, TypeError, KeyError, IndexError):
            return {"error": "INVALID_PATCH_ACTION"}

        return {"error": "INVALID_PATCH_ACTION"}


# 3. Build an entirely synthetic mock patient.
# No C06 values are used in these tests.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(314159)
    mock_model = BPatientNet().cpu()

mock_data = {
    "id": (
        torch.zeros((256, 6)),
        torch.zeros((256, 1)),
    ),
    "shift": (
        torch.ones((256, 6))*0.1,
        torch.zeros((256, 1)),
    ),
}

mock_backend = BRestrictedBackend(
    mock_model, mock_data
)

# 4. Test the complete diagnosis-first pathway.
mock_session = BSession(
    mock_backend, "diagnosis_first"
)

for tool in B_DIAGNOSTIC_TOOLS:
    example = B_TOOL_CONTRACT[
        "diagnostic_tools"
    ][tool]["example"]

    response = mock_session.dispatch(
        json.dumps(example)
    )

    assert "error" not in response

assert mock_session.diagnostic_calls == 4

response = mock_session.dispatch(
    json.dumps(mock_diagnosis)
)

assert response["status"] == "DIAGNOSIS_COMMITTED"
assert mock_session.phase == "repair"
assert mock_session.diagnostic_calls == 4

response = mock_session.dispatch(
    json.dumps({
        "tool": "patch_trial",
        "edits": [mock_edit],
    })
)

assert response["tool"] == "patch_trial"
assert mock_session.patch_trials == 1

response = mock_session.dispatch(
    json.dumps({
        "tool": "submit_patch",
        "edits": [mock_edit],
    })
)

assert response["status"] == "FINAL_PATCH_COMMITTED"
assert mock_session.phase == "complete"

# 5. Test the repair-only pathway.
repair_mock = BSession(
    mock_backend, "repair_only"
)

assert "error" not in repair_mock.dispatch(
    json.dumps({
        "tool": "activations",
        "regime": "shift",
    })
)

assert "error" not in repair_mock.dispatch(
    json.dumps({
        "tool": "patch_trial",
        "edits": [],
    })
)

assert repair_mock.dispatch(
    json.dumps({
        "tool": "submit_patch",
        "edits": [],
    })
)["status"] == "FINAL_PATCH_COMMITTED"

# 6. Verify invalid diagnostic calls consume budget.
invalid_mock = BSession(
    mock_backend, "diagnosis_first"
)

for _ in range(12):
    result = invalid_mock.dispatch(
        '{"tool":"activation_summaries","regime":"shift"}'
    )
    assert result["error"] == "INVALID_DIAGNOSTIC_REQUEST"

assert invalid_mock.diagnostic_calls == 12

result = invalid_mock.dispatch(
    '{"tool":"activations","regime":"shift"}'
)

assert result["error"] == "DIAGNOSTIC_BUDGET_EXHAUSTED"

# Even after 12 calls, valid diagnosis submission remains
# possible, and is not counted as a diagnostic observation.
result = invalid_mock.dispatch(
    json.dumps(mock_diagnosis)
)

assert result["status"] == "DIAGNOSIS_COMMITTED"
assert invalid_mock.diagnostic_calls == 12

# 7. Freeze public, internally consistent instructions.
B_PUBLIC_GUIDE = {
    "version": "EX0004B_PUBLIC_GUIDE_V1",
    "architecture": [6, 12, 12, 1],
    "task": (
        "Continuous prediction; x5 is a nuisance input."
    ),
    "patient": (
        "Two planted fault groups in different hidden "
        "layers; four altered scalar parameters total."
    ),
    "diagnostic_tools": B_TOOL_CONTRACT[
        "diagnostic_tools"
    ],
    "tensor_shapes": B_TENSOR_SHAPES,
    "diagnosis_fields": B_TOOL_CONTRACT[
        "diagnosis_fields"
    ],
    "diagnosis_predictions": B_TOOL_CONTRACT[
        "diagnosis_predictions"
    ],
    "patch_format": B_TOOL_CONTRACT["patch_format"],
    "repair_actions": {
        "patch_trial": {
            "tool": "patch_trial",
            "edits": [mock_edit],
        },
        "submit_patch": {
            "tool": "submit_patch",
            "edits": [],
        },
    },
    "rules": {
        "one_json_action_per_turn": True,
        "diagnostic_calls_max": 12,
        "patch_trials_max": 2,
        "scalar_edits_max": 6,
        "diagnosis_before_patch_feedback": True,
        "patch_deltas_relative_to_corrupted_baseline": True,
        "diagnosis_submission_costs_zero_probes": True,
        "no_healthy_weights_or_oracle_access": True,
        "no_arbitrary_python_or_filesystem_access": True,
    },
}

B_PUBLIC_GUIDE_PATH = (
    ROOT / "PUBLIC_TEACHER_GUIDE_V1.json"
)

B_PUBLIC_GUIDE_SHA = b_freeze_record(
    B_PUBLIC_GUIDE_PATH, B_PUBLIC_GUIDE
)

B_EXECUTION_RULES = {
    "version": "EX0004B_EXECUTOR_V1",
    "maximum_total_actions_per_arm": 20,
    "invalid_diagnostic_requests_count": True,
    "invalid_patch_requests_do_not_consume_evaluations": True,
    "final_patch_is_not_evaluated_on_submission": True,
    "session_event_hash_chain": True,
    "diagnosis_is_irrevocable": True,
    "all_patch_trials_start_from_corrupted_baseline": True,
}

B_EXECUTION_SHA = b_freeze_record(
    ROOT / "EXECUTION_RULES_V1.json",
    B_EXECUTION_RULES
)

B_MANIFEST["execution_status"] = "MOCK_PREFLIGHT_PASS"
B_MANIFEST["public_guide_sha256"] = B_PUBLIC_GUIDE_SHA
B_MANIFEST["execution_rules_sha256"] = B_EXECUTION_SHA

print("Diagnostic tools exercised: 4 / 4")
print("Diagnosis submission: PASS")
print("Patch trial: PASS")
print("Final patch commitment: PASS")
print("Repair-only arm: PASS")
print("Invalid call counting: PASS")
print("Diagnosis allowed after budget exhaustion: PASS")
print("Maximum total actions per arm: 20")
print("Actual patient probed: NO")
print("Qwen invoked: NO")
print("Sealed evaluation accessed: NO")

print("\nPUBLIC GUIDE SHA-256:")
print(B_PUBLIC_GUIDE_SHA)

print("\nEXECUTION RULES SHA-256:")
print(B_EXECUTION_SHA)

print("\n✅ CELL 6: MOCK EXECUTION PREFLIGHT PASSED")


🌒 EX0004B — EXECUTION ENGINE PREFLIGHT
Diagnostic tools exercised: 4 / 4
Diagnosis submission: PASS
Patch trial: PASS
Final patch commitment: PASS
Repair-only arm: PASS
Invalid call counting: PASS
Diagnosis allowed after budget exhaustion: PASS
Maximum total actions per arm: 20
Actual patient probed: NO
Qwen invoked: NO
Sealed evaluation accessed: NO

PUBLIC GUIDE SHA-256:
9a9ec913a9836d28e1c645e90165ff05a46a1a27fc38a6f75062556ed5bd6907

EXECUTION RULES SHA-256:
475461b7f20c3afcb33fdb60f9582b54a0a26fbd25f5b89bbc0181d79e90d6ae

✅ CELL 6: MOCK EXECUTION PREFLIGHT PASSED


#Cell 7: Load Qwen and prepare C06's restricted examination room
This cell does four things:
- verifies the frozen experimental artifacts,
- creates the teacher-development datasets,
- loads the exact Qwen3-8B revision on the L4,
- and connects our corrupted patient to the restricted backend

In [7]:

# EX0004B — Cell 7: Load frozen teacher and private backend
# NO inference, diagnostic requests, or sealed evaluation.

import sys
import subprocess
import importlib.metadata as metadata
import json
import hashlib
from pathlib import Path
from packaging.version import Version

print("🌒 EX0004B — THE SURGEON RETURNS")
print("=" * 60)

# 1. Verify frozen experimental artifacts.
assert B_MANIFEST["patient_status"] == "CORRUPTED_FROZEN"
assert B_MANIFEST["execution_status"] == "MOCK_PREFLIGHT_PASS"
assert B_MANIFEST["teacher_status"] == "NOT_EXPOSED"

assert b_file_sha(B_PROTOCOL_PATH) == B_PROTOCOL_SHA
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA
assert b_file_sha(B_HEALTHY_PATH) == B_HEALTHY_SHA

assert b_file_sha(B_PUBLIC_GUIDE_PATH) == (
    B_MANIFEST["public_guide_sha256"]
)

assert b_file_sha(
    ROOT / "EXECUTION_RULES_V1.json"
) == B_MANIFEST["execution_rules_sha256"]

B_TEACHER_RECEIPT_PATH = (
    ROOT / "TEACHER_LOAD_RECEIPT_V1.json"
)

assert not B_TEACHER_RECEIPT_PATH.exists(), (
    "Teacher receipt already exists. "
    "Do not silently repeat the setup."
)

# 2. Prepare fresh teacher-development data.
# Calibration and sealed seeds are not used.
B_TEACHER_DATA = {
    "id": b_sample_world(
        256, B_SEEDS["teacher_id"], "id", "cpu"
    ),
    "shift": b_sample_world(
        256, B_SEEDS["teacher_shift"], "shift", "cpu"
    ),
}

def b_dataset_fingerprint(dataset):
    h = hashlib.sha256()

    for regime in ("id", "shift"):
        for tensor in dataset[regime]:
            t = tensor.detach().cpu().contiguous()
            h.update(str(tuple(t.shape)).encode())
            h.update(str(t.dtype).encode())
            h.update(t.numpy().tobytes())

    return h.hexdigest()

B_TEACHER_DATA_SHA = b_dataset_fingerprint(
    B_TEACHER_DATA
)

assert B_TEACHER_DATA["id"][0].shape == (256, 6)
assert B_TEACHER_DATA["shift"][0].shape == (256, 6)

print("Teacher ID examples: 256")
print("Teacher SHIFT examples: 256")
print("Teacher data fingerprint:", B_TEACHER_DATA_SHA)
print("Construction-calibration data reused: NO")

# 3. Load the corrupted patient into a CPU-only backend.
# No diagnostic observations are executed here.
corrupt_record = torch.load(
    B_CORRUPT_PATH,
    map_location="cpu",
    weights_only=True,
)

B_DIAGNOSTIC_PATIENT = BPatientNet().cpu()

B_DIAGNOSTIC_PATIENT.load_state_dict(
    corrupt_record["model_state"],
    strict=True,
)

B_DIAGNOSTIC_PATIENT.eval()
B_DIAGNOSTIC_PATIENT.requires_grad_(False)

B_LIVE_BACKEND = BRestrictedBackend(
    B_DIAGNOSTIC_PATIENT,
    B_TEACHER_DATA,
)

# Verify its weights exactly match the frozen patient.
assert all(
    torch.equal(
        B_LIVE_BACKEND.base_state[name],
        corrupt_record["model_state"][name]
    )
    for name in corrupt_record["model_state"]
)

print("Corrupted checkpoint verified: YES")
print("Restricted backend: CPU")
print("Diagnostic requests executed: 0")

# 4. Free training resources from GPU.
# The healthy checkpoint on disk remains unchanged.
B_HEALTHY.cpu()

torch.cuda.empty_cache()

assert torch.cuda.is_available()
assert torch.cuda.is_bf16_supported()

free_before, total_gpu = torch.cuda.mem_get_info()

print("\nGPU:", torch.cuda.get_device_name(0))
print(
    "Available memory:",
    f"{free_before/1024**3:.2f} GiB"
)

assert free_before >= 18 * 1024**3, (
    "Insufficient memory for frozen BF16 teacher. "
    "Stop instead of silently changing precision."
)

# 5. Ensure compatible inference dependencies.
requirements = {
    "transformers": ("4.55.0", "5.0.0"),
    "accelerate": ("1.1.0", "2.0.0"),
    "huggingface_hub": ("0.33.0", "1.0.0"),
}

needs_install = False

for package, (minimum, maximum) in requirements.items():
    try:
        current = Version(metadata.version(package))

        if not (
            Version(minimum) <= current < Version(maximum)
        ):
            needs_install = True

    except metadata.PackageNotFoundError:
        needs_install = True

if needs_install:
    print("\nInstalling compatible inference dependencies...")

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.55,<5",
        "accelerate>=1.1,<2",
        "huggingface_hub>=0.33,<1",
    ])

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

# 6. Use the exact previously frozen Qwen revision.
B_MODEL_ID = B_PROTOCOL["teacher"]["model"]
B_MODEL_REVISION = B_PROTOCOL["teacher"]["revision"]

assert B_MODEL_ID == "Qwen/Qwen3-8B"
assert B_MODEL_REVISION == (
    "b968826d9c46dd6066d109eabc6255188de91218"
)

print("\nLoading:", B_MODEL_ID)
print("Pinned revision:", B_MODEL_REVISION)
print("Transformers:", transformers.__version__)

B_QWEN_TOKENIZER = AutoTokenizer.from_pretrained(
    B_MODEL_ID,
    revision=B_MODEL_REVISION,
    trust_remote_code=False,
)

B_QWEN_MODEL = AutoModelForCausalLM.from_pretrained(
    B_MODEL_ID,
    revision=B_MODEL_REVISION,
    dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=False,
)

B_QWEN_MODEL.eval()
B_QWEN_MODEL.requires_grad_(False)

# 7. Verify model placement and tokenizer.
parameter_count = sum(
    p.numel() for p in B_QWEN_MODEL.parameters()
)

model_devices = {
    str(p.device)
    for p in B_QWEN_MODEL.parameters()
}

assert 8_000_000_000 < parameter_count < 8_500_000_000
assert model_devices == {"cuda:0"}

# Tokenizer-only smoke test. No generation.
smoke_prompt = B_QWEN_TOKENIZER.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Tokenizer compatibility test."
        },
        {
            "role": "user",
            "content": "No response required."
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

assert isinstance(smoke_prompt, str)
assert len(smoke_prompt) > 0

# 8. Freeze runtime configuration.
B_GENERATION = {
    "enable_thinking": True,
    "do_sample": True,
    "temperature": B_PROTOCOL["teacher"]["temperature"],
    "top_p": B_PROTOCOL["teacher"]["top_p"],
    "top_k": B_PROTOCOL["teacher"]["top_k"],
    "max_new_tokens": B_PROTOCOL[
        "teacher"
    ]["max_new_tokens_per_turn"],
    "diagnosis_seed": B_SEEDS[
        "diagnosis_generation"
    ],
    "repair_only_seed": B_SEEDS[
        "repair_only_generation"
    ],
}

B_TEACHER_RUNTIME = {
    "model_id": B_MODEL_ID,
    "revision": B_MODEL_REVISION,
    "precision": "bfloat16",
    "device": "cuda:0",
    "attention": "sdpa",
    "parameter_count": parameter_count,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "teacher_data_sha256": B_TEACHER_DATA_SHA,
    "corrupted_patient_sha256": B_CORRUPT_SHA,
    "public_guide_sha256": B_PUBLIC_GUIDE_SHA,
    "generation": B_GENERATION,
    "diagnostic_calls": 0,
    "patch_evaluations": 0,
    "status": "LOADED_NOT_EXPOSED",
}

B_RUNTIME_SHA = b_freeze_record(
    B_TEACHER_RECEIPT_PATH,
    B_TEACHER_RUNTIME,
)

B_MANIFEST["teacher_runtime_sha256"] = B_RUNTIME_SHA
B_MANIFEST["teacher_status"] = "LOADED_NOT_EXPOSED"

# 9. Report runtime state.
free_after, _ = torch.cuda.mem_get_info()

print("\nTEACHER VERIFICATION")
print("-" * 60)
print("Qwen parameters:", f"{parameter_count:,}")
print("Model devices:", model_devices)
print("Precision: BF16")
print("Tokenizer smoke test: PASS")
print(
    "GPU memory used:",
    f"{(total_gpu-free_after)/1024**3:.2f} GiB"
)
print(
    "GPU memory remaining:",
    f"{free_after/1024**3:.2f} GiB"
)

print("\nEXPERIMENT BOUNDARIES")
print("Private fault key in teacher prompt: NO")
print("Healthy weights in teacher prompt: NO")
print("Teacher patient observations: 0")
print("Patch evaluations: 0")
print("Sealed evaluation generated: NO")

print("\nRuntime fingerprint:")
print(B_RUNTIME_SHA)

print("\n✅ CELL 7: TEACHER LOADED, PATIENT ISOLATED")


🌒 EX0004B — THE SURGEON RETURNS
Teacher ID examples: 256
Teacher SHIFT examples: 256
Teacher data fingerprint: 6ca8302d9d15e41ce872e7b57ee9adf1b6bdb127c3148b0f4f0f239d32dd3c0b
Construction-calibration data reused: NO
Corrupted checkpoint verified: YES
Restricted backend: CPU
Diagnostic requests executed: 0

GPU: NVIDIA L4
Available memory: 21.69 GiB

Installing compatible inference dependencies...

Loading: Qwen/Qwen3-8B
Pinned revision: b968826d9c46dd6066d109eabc6255188de91218
Transformers: 4.57.6


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

... (7 outputs left out)

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


TEACHER VERIFICATION
------------------------------------------------------------
Qwen parameters: 8,190,735,360
Model devices: {'cuda:0'}
Precision: BF16
Tokenizer smoke test: PASS
GPU memory used: 15.61 GiB
GPU memory remaining: 6.43 GiB

EXPERIMENT BOUNDARIES
Private fault key in teacher prompt: NO
Healthy weights in teacher prompt: NO
Teacher patient observations: 0
Patch evaluations: 0
Sealed evaluation generated: NO

Runtime fingerprint:
1b4720723c99cba63d808018a3bcd0a1756c52d283def5614d81022bdb14dbd1

✅ CELL 7: TEACHER LOADED, PATIENT ISOLATED


#Cell 8: Freeze and preflight the autonomous Qwen loop
This is our final mock-only cell. It establishes the actual prompts for both experimental arms, defines the Qwen generation adapter, and tests the entire conversation loop with scripted responses.

In [8]:

# EX0004B — Cell 8: Autonomous-loop preflight
# Mock responses ONLY. No Qwen generation or C06 probes.

import os
import json
import hashlib
import marshal
from copy import deepcopy
from pathlib import Path

print("🌒 EX0004B — AUTONOMOUS LOOP PREFLIGHT")
print("=" * 60)

assert B_MANIFEST["teacher_status"] == "LOADED_NOT_EXPOSED"
assert B_MANIFEST["execution_status"] == "MOCK_PREFLIGHT_PASS"
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA
assert b_file_sha(B_PUBLIC_GUIDE_PATH) == B_PUBLIC_GUIDE_SHA
assert b_file_sha(B_TEACHER_RECEIPT_PATH) == B_RUNTIME_SHA

# 1. Complete neutral JSON example for diagnosis submission.
# These are FORMAT examples, not evidence or recommendations.
B_DIAGNOSIS_EXAMPLE = {
    "tool": "submit_diagnosis",
    "fault_A": {
        "layer": "fc1",
        "neuron": 0,
        "input_feature": 0
    },
    "fault_B": {
        "layer": "fc2",
        "neuron": 0,
        "source_neuron": 0
    },
    "predictions": {
        "isolated_effect_rank": "A_gt_B",
        "interaction_sign": "positive",
        "nuisance_restoration": "unchanged"
    },
    "confidence": 0.5,
    "mechanism": "FORMAT EXAMPLE ONLY. Not a diagnosis.",
    "initial_patch": []
}

B_NEUTRAL_EDIT = {
    "tensor": "readout.bias",
    "index": [0],
    "delta": 0.01
}

assert b_validate_action(
    B_DIAGNOSIS_EXAMPLE, "diagnosis"
) == "submit_diagnosis"

assert b_validate_action({
    "tool": "patch_trial",
    "edits": [B_NEUTRAL_EDIT]
}, "repair") == "patch_trial"

# 2. Arm-specific instructions, frozen BEFORE live exposure.
B_SYSTEM_PROMPTS = {
    "diagnosis_first": (
        "You are a blinded neural-network investigator. "
        "Your task is to diagnose two planted fault regions, "
        "predict their isolated and interacting causal effects, "
        "and then attempt a sparse weight repair. "
        "Use only the permitted JSON actions. "
        "Submit your diagnosis before any patch trial. "
        "After submitting it, use patch_trial or submit_patch. "
        "You may submit a diagnosis before exhausting probes. "
        "When probes are exhausted, submit your diagnosis. "
        "Uncertainty is permitted; do not invent observations. "
        "Output exactly ONE JSON object per turn, without "
        "Markdown fences or additional explanatory text. "
        "Do not request arbitrary Python or filesystem access."
    ),
    "repair_only": (
        "You are a neural-network weight-repair investigator. "
        "Your task is to improve a corrupted network using "
        "bounded diagnostic observations and sparse weight edits. "
        "You do not need to submit a mechanistic diagnosis. "
        "You may use patch_trial or submit_patch immediately. "
        "All patch deltas are relative to the original corrupted "
        "checkpoint, not to a previous trial. "
        "Use only permitted JSON actions. "
        "Output exactly ONE JSON object per turn, without "
        "Markdown fences or additional explanatory text. "
        "Do not request arbitrary Python or filesystem access."
    )
}

def b_public_prompt_for_arm(arm):
    guide = deepcopy(B_PUBLIC_GUIDE)

    if arm == "diagnosis_first":
        guide["diagnosis_submission_example"] = (
            B_DIAGNOSIS_EXAMPLE
        )
        guide["allowed_actions"] = [
            *B_DIAGNOSTIC_TOOLS,
            "submit_diagnosis",
            "patch_trial",
            "submit_patch"
        ]
        guide["phase_rule"] = (
            "patch_trial and submit_patch are unavailable "
            "until submit_diagnosis is accepted."
        )
    else:
        guide.pop("diagnosis_fields", None)
        guide.pop("diagnosis_predictions", None)
        guide["rules"]["diagnosis_before_patch_feedback"] = False
        guide["allowed_actions"] = [
            *B_DIAGNOSTIC_TOOLS,
            "patch_trial",
            "submit_patch"
        ]
        guide["phase_rule"] = (
            "No diagnosis submission is required or allowed."
        )

    guide["patch_trial_example"] = {
        "tool": "patch_trial",
        "edits": [B_NEUTRAL_EDIT]
    }
    guide["final_patch_example"] = {
        "tool": "submit_patch",
        "edits": []
    }
    guide["example_warning"] = (
        "Examples demonstrate syntax only. They are not "
        "observations, diagnoses or recommended repairs."
    )
    return guide


B_INITIAL_MESSAGES = {}

for arm in ("diagnosis_first", "repair_only"):
    B_INITIAL_MESSAGES[arm] = [
        {
            "role": "system",
            "content": B_SYSTEM_PROMPTS[arm]
        },
        {
            "role": "user",
            "content": (
                "FROZEN TOOL AND TASK CONTRACT:\n"
                + json.dumps(
                    b_public_prompt_for_arm(arm),
                    indent=2
                )
                + "\n\nChoose your first permitted JSON action."
            )
        }
    ]

B_PROMPT_RECORD = {
    "version": "EX0004B_PROMPTS_V1",
    "initial_messages": B_INITIAL_MESSAGES,
    "model_revision": B_MODEL_REVISION,
    "generation": B_GENERATION,
}

B_PROMPT_PATH = ROOT / "FROZEN_TEACHER_PROMPTS_V1.json"
B_PROMPT_SHA = b_freeze_record(
    B_PROMPT_PATH, B_PROMPT_RECORD
)

# Tokenizer preflight without model generation.
end_think_id = B_QWEN_TOKENIZER.convert_tokens_to_ids(
    "</think>"
)

assert isinstance(end_think_id, int)
assert end_think_id >= 0

for arm, messages in B_INITIAL_MESSAGES.items():
    rendered = B_QWEN_TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )
    assert isinstance(rendered, str)
    assert len(rendered) > 100
    assert "C06" not in rendered
    assert "HEALTHY_PRIVATE.pt" not in rendered

print("Frozen public prompts: 2 arms")
print("Complete diagnosis JSON example: PASS")
print("Canonical tool names: PASS")
print("Qwen chat-template rendering: PASS")

# 3. Deterministic per-turn sampling seeds.
# Do not reuse the identical seed for every turn.
def b_turn_seed(arm, turn_number):
    base = (
        B_SEEDS["diagnosis_generation"]
        if arm == "diagnosis_first"
        else B_SEEDS["repair_only_generation"]
    )
    material = f"{base}|{arm}|turn={turn_number}"
    digest = hashlib.sha256(material.encode()).digest()
    return int.from_bytes(digest[:8], "big") % (2**31 - 1) + 1


# 4. Actual Qwen generation adapter.
# DEFINED HERE; NEVER CALLED IN THIS PREFLIGHT.
def b_live_decision(messages, arm, turn_number):
    prompt = B_QWEN_TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    prompt_hash = hashlib.sha256(
        prompt.encode()
    ).hexdigest()

    inputs = B_QWEN_TOKENIZER(
        [prompt],
        return_tensors="pt"
    ).to("cuda:0")

    seed = b_turn_seed(arm, turn_number)

    with torch.random.fork_rng(devices=[0]):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        with torch.inference_mode():
            output = B_QWEN_MODEL.generate(
                **inputs,
                max_new_tokens=B_GENERATION["max_new_tokens"],
                do_sample=B_GENERATION["do_sample"],
                temperature=B_GENERATION["temperature"],
                top_p=B_GENERATION["top_p"],
                top_k=B_GENERATION["top_k"],
                pad_token_id=B_QWEN_TOKENIZER.eos_token_id
            )

    new_tokens = output[0][
        inputs["input_ids"].shape[1]:
    ].tolist()

    completed = end_think_id in new_tokens

    if completed:
        split = (
            len(new_tokens) - 1
            - new_tokens[::-1].index(end_think_id)
        )
        final_text = B_QWEN_TOKENIZER.decode(
            new_tokens[split + 1:],
            skip_special_tokens=True
        ).strip()
    else:
        final_text = ""

    return {
        "text": final_text,
        "thinking_completed": completed,
        "generated_tokens": len(new_tokens),
        "generation_seed": seed,
        "prompt_sha256": prompt_hash,
        "completion_sha256": hashlib.sha256(
            B_QWEN_TOKENIZER.decode(
                new_tokens,
                skip_special_tokens=False
            ).encode()
        ).hexdigest(),
    }


# 5. Append-only, hash-chained event journal.
def b_append_journal(path, kind, body):
    path = Path(path)

    previous_hash = None
    if path.exists():
        lines = path.read_text().splitlines()
        if lines:
            previous_hash = json.loads(
                lines[-1]
            )["sha256"]

    record = {
        "kind": kind,
        "body": body,
        "previous_hash": previous_hash
    }

    payload = json.dumps(
        record, sort_keys=True, allow_nan=False
    )

    record["sha256"] = hashlib.sha256(
        payload.encode()
    ).hexdigest()

    with path.open("a") as stream:
        stream.write(
            json.dumps(record, sort_keys=True) + "\n"
        )
        stream.flush()
        os.fsync(stream.fileno())


def b_verify_journal(path):
    events = [
        json.loads(line)
        for line in Path(path).read_text().splitlines()
    ]

    previous_hash = None
    for event in events:
        assert event["previous_hash"] == previous_hash
        unsigned = dict(event)
        stored = unsigned.pop("sha256")

        calculated = hashlib.sha256(
            json.dumps(
                unsigned,
                sort_keys=True,
                allow_nan=False
            ).encode()
        ).hexdigest()

        assert stored == calculated
        previous_hash = stored

    return len(events)


# 6. One fixed runner for BOTH teacher arms.
# No human prompt edits between its actions.
def b_run_arm(arm, backend, decide, output_dir):
    output_dir = Path(output_dir)
    assert not output_dir.exists(), (
        "Arm already has records. Never silently restart."
    )
    output_dir.mkdir(parents=True)

    journal = output_dir / "events.jsonl"
    transcript = output_dir / "transcript.json"

    session = BSession(backend, arm)
    messages = deepcopy(B_INITIAL_MESSAGES[arm])

    status = "ACTION_LIMIT_REACHED"

    for turn in range(1, 21):
        # Record intent BEFORE generation.
        b_append_journal(journal, "TURN_START", {
            "turn": turn,
            "arm": arm,
            "phase": session.phase,
            "diagnostic_calls": session.diagnostic_calls,
            "patch_trials": session.patch_trials,
        })

        decision = decide(messages, arm, turn)

        # Persist output BEFORE dispatching any tool.
        b_append_journal(journal, "GENERATION", {
            "turn": turn,
            "decision": decision,
        })

        if not decision["thinking_completed"]:
            status = "INCOMPLETE_GENERATION"
            break

        # One submitted action, one dispatch.
        response = session.dispatch(decision["text"])

        b_append_journal(journal, "TOOL_RESULT", {
            "turn": turn,
            "response": response,
            "phase_after": session.phase,
            "diagnostic_calls": session.diagnostic_calls,
            "patch_trials": session.patch_trials,
            "session_event_hash": session.events[-1]["sha256"],
        })

        messages.append({
            "role": "assistant",
            "content": decision["text"]
        })

        # Fixed feedback template; no adaptive coaching.
        messages.append({
            "role": "user",
            "content": (
                "TOOL RESPONSE:\n"
                + json.dumps(response, sort_keys=True)
                + "\nCURRENT STATE:\n"
                + json.dumps({
                    "phase": session.phase,
                    "diagnostic_calls_remaining":
                        max(0, 12-session.diagnostic_calls),
                    "patch_trials_remaining":
                        max(0, 2-session.patch_trials),
                    "total_actions_remaining":
                        max(0, 20-session.total_actions)
                }, sort_keys=True)
                + "\nChoose one permitted JSON action."
            )
        })

        # Save complete current conversation after each action.
        snapshot = {
            "arm": arm,
            "turn": turn,
            "messages": messages,
            "phase": session.phase,
            "diagnostic_calls": session.diagnostic_calls,
            "patch_trials": session.patch_trials,
            "diagnosis": session.diagnosis,
            "final_patch": session.final_patch,
        }

        temporary = transcript.with_suffix(".tmp")
        temporary.write_text(
            json.dumps(snapshot, indent=2, allow_nan=False)
        )
        temporary.replace(transcript)

        if session.phase == "complete":
            status = "COMPLETED"
            break

    summary = {
        "arm": arm,
        "status": status,
        "diagnostic_calls": session.diagnostic_calls,
        "patch_trials": session.patch_trials,
        "total_actions": session.total_actions,
        "diagnosis_committed": session.diagnosis is not None,
        "final_patch_committed": session.final_patch is not None,
        "diagnosis": session.diagnosis,
        "final_patch": session.final_patch,
        "sealed_evaluation": "UNOPENED",
    }

    (output_dir / "summary.json").write_text(
        json.dumps(summary, indent=2, allow_nan=False)
    )

    b_append_journal(journal, "FINAL_STATUS", summary)
    b_verify_journal(journal)

    return summary


# 7. Mock a complete response-generation sequence.
# No actual Qwen output and no C06 patient observations.
B_MOCK_ACTIONS = {
    "diagnosis_first": [
        {
            "tool": "weights",
            "tensor": "fc1.bias",
            "start": 0,
            "count": 6
        },
        {
            "tool": "activations",
            "regime": "shift"
        },
        B_DIAGNOSIS_EXAMPLE,
        {
            "tool": "patch_trial",
            "edits": [B_NEUTRAL_EDIT]
        },
        {
            "tool": "submit_patch",
            "edits": []
        },
    ],
    "repair_only": [
        {
            "tool": "examples",
            "regime": "id",
            "indices": [0, 1]
        },
        {
            "tool": "patch_trial",
            "edits": [B_NEUTRAL_EDIT]
        },
        {
            "tool": "submit_patch",
            "edits": []
        },
    ]
}

def b_mock_decision(messages, arm, turn):
    raw = json.dumps(B_MOCK_ACTIONS[arm][turn-1])
    return {
        "text": raw,
        "thinking_completed": True,
        "generated_tokens": 0,
        "generation_seed": b_turn_seed(arm, turn),
        "prompt_sha256": hashlib.sha256(
            json.dumps(messages, sort_keys=True).encode()
        ).hexdigest(),
        "completion_sha256": hashlib.sha256(
            raw.encode()
        ).hexdigest(),
    }


B_MOCK_ROOT = ROOT / "MOCK_AUTONOMOUS_PREFLIGHT_V1"
assert not B_MOCK_ROOT.exists()

mock_diagnosis_result = b_run_arm(
    "diagnosis_first",
    mock_backend,
    b_mock_decision,
    B_MOCK_ROOT / "diagnosis_first"
)

mock_repair_result = b_run_arm(
    "repair_only",
    mock_backend,
    b_mock_decision,
    B_MOCK_ROOT / "repair_only"
)

assert mock_diagnosis_result["status"] == "COMPLETED"
assert mock_diagnosis_result["diagnostic_calls"] == 2
assert mock_diagnosis_result["patch_trials"] == 1
assert mock_diagnosis_result["diagnosis_committed"]
assert mock_diagnosis_result["final_patch_committed"]

assert mock_repair_result["status"] == "COMPLETED"
assert mock_repair_result["diagnostic_calls"] == 1
assert mock_repair_result["patch_trials"] == 1
assert mock_repair_result["final_patch_committed"]

# 8. Freeze runner configuration and implementation fingerprint.
B_RUNNER_CODE_SHA = hashlib.sha256(
    marshal.dumps(b_run_arm.__code__)
    + marshal.dumps(b_live_decision.__code__)
).hexdigest()

B_LOOP_SPEC = {
    "version": "EX0004B_AUTONOMOUS_LOOP_V1",
    "prompt_sha256": B_PROMPT_SHA,
    "code_sha256": B_RUNNER_CODE_SHA,
    "maximum_actions_per_arm": 20,
    "max_diagnostic_calls": 12,
    "max_patch_trials": 2,
    "per_turn_seed_derivation": "SHA256(base|arm|turn)",
    "incomplete_generation_policy": "STOP_NO_RETRY",
    "event_journal": "APPEND_ONLY_HASH_CHAIN",
    "feedback": "FIXED_TEMPLATE",
    "mock_diagnosis_status": mock_diagnosis_result["status"],
    "mock_repair_status": mock_repair_result["status"],
    "teacher_exposed": False,
}

B_LOOP_SPEC_PATH = ROOT / "AUTONOMOUS_LOOP_SPEC_V1.json"
B_LOOP_SHA = b_freeze_record(
    B_LOOP_SPEC_PATH, B_LOOP_SPEC
)

B_MANIFEST["loop_status"] = "FULL_PREFLIGHT_PASS"
B_MANIFEST["loop_sha256"] = B_LOOP_SHA
B_MANIFEST["runner_code_sha256"] = B_RUNNER_CODE_SHA

print("\nAUTONOMOUS PREFLIGHT RESULTS")
print("-" * 60)
print("Diagnosis-first mock: PASS")
print("Repair-only mock: PASS")
print("Complete diagnosis JSON: PASS")
print("Diagnosis before patch feedback: PASS")
print("Final patch commitment: PASS")
print("Per-turn event journaling: PASS")
print("Journal hash-chain verification: PASS")
print("Complete transcript persistence: PASS")
print("Frozen per-turn seed derivation: PASS")
print("Qwen tokenizer compatibility: PASS")
print("Actual Qwen generations: 0")
print("Actual C06 observations: 0")
print("Sealed evaluation accessed: NO")

print("\nFROZEN PROMPT SHA-256:")
print(B_PROMPT_SHA)
print("\nFROZEN LOOP SHA-256:")
print(B_LOOP_SHA)

print("\n✅ CELL 8: AUTONOMOUS LOOP PREFLIGHT PASSED")


🌒 EX0004B — AUTONOMOUS LOOP PREFLIGHT
Frozen public prompts: 2 arms
Complete diagnosis JSON example: PASS
Canonical tool names: PASS
Qwen chat-template rendering: PASS

AUTONOMOUS PREFLIGHT RESULTS
------------------------------------------------------------
Diagnosis-first mock: PASS
Repair-only mock: PASS
Complete diagnosis JSON: PASS
Diagnosis before patch feedback: PASS
Final patch commitment: PASS
Per-turn event journaling: PASS
Journal hash-chain verification: PASS
Complete transcript persistence: PASS
Frozen per-turn seed derivation: PASS
Qwen tokenizer compatibility: PASS
Actual Qwen generations: 0
Actual C06 observations: 0
Sealed evaluation accessed: NO

FROZEN PROMPT SHA-256:
22d9d04a32c4b32f3a63f16bcf01de6c14ee16a8e5016012a717fdfa00a93747

FROZEN LOOP SHA-256:
6adf16a9353165115468c64f302e4717ba2f36b0663cea6d1a6b2c314588a41b

✅ CELL 8: AUTONOMOUS LOOP PREFLIGHT PASSED


#Cell 9: The surgeon's autonomous investigation
This cell verifies:
- the complete frozen evidence chain,
- registers the invocation,
- and executes the already-tested autonomous runner exactly once.

It will automatically:
- perform the diagnostic investigation,
- accept a diagnosis commitment,
- permit up to two patch trials,
- and record any final patch

It will not open the sealed evaluation

In [10]:

# EX0004B — Cell 9: Live diagnosis-first investigation
# ONE autonomous run. No retries or prompt modifications.
# Sealed evaluation remains unopened.

import json
import hashlib
import marshal
import traceback
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

print("🌒 EX0004B — DIAGNOSIS-FIRST LIVE TRIAL")
print("=" * 62)

# 1. Reverify the frozen experiment.
assert B_MANIFEST["patient_status"] == "CORRUPTED_FROZEN"
assert B_MANIFEST["teacher_status"] == "LOADED_NOT_EXPOSED"
assert B_MANIFEST["loop_status"] == "FULL_PREFLIGHT_PASS"

frozen_files = [
    (B_PROTOCOL_PATH, B_PROTOCOL_SHA),
    (B_HEALTHY_PATH, B_HEALTHY_SHA),
    (B_FACTORY_PATH, B_FACTORY_SHA),
    (B_CORRUPT_PATH, B_CORRUPT_SHA),
    (
        B_AUDIT_PATH,
        B_MANIFEST["construction_audit_sha256"]
    ),
    (
        schema_path,
        B_MANIFEST["tool_contract_sha256"]
    ),
    (B_PUBLIC_GUIDE_PATH, B_PUBLIC_GUIDE_SHA),
    (B_TEACHER_RECEIPT_PATH, B_RUNTIME_SHA),
    (B_PROMPT_PATH, B_PROMPT_SHA),
    (B_LOOP_SPEC_PATH, B_LOOP_SHA),
]

for path, expected in frozen_files:
    assert b_file_sha(path) == expected, (
        f"Frozen artifact mismatch: {path}"
    )

current_code_sha = hashlib.sha256(
    marshal.dumps(b_run_arm.__code__)
    + marshal.dumps(b_live_decision.__code__)
).hexdigest()

assert current_code_sha == B_RUNNER_CODE_SHA

assert b_dataset_fingerprint(
    B_TEACHER_DATA
) == B_TEACHER_DATA_SHA

assert B_LIVE_BACKEND.model.training is False

# Ensure the active backend contains frozen corrupted
# weights, not healthy weights or a modified trial patch.
corrupted_record = torch.load(
    B_CORRUPT_PATH,
    map_location="cpu",
    weights_only=True
)

assert all(
    torch.equal(
        B_LIVE_BACKEND.base_state[name],
        corrupted_record["model_state"][name]
    )
    for name in corrupted_record["model_state"]
)

# 2. Verify model and prompt configuration.
assert B_MODEL_ID == "Qwen/Qwen3-8B"
assert B_MODEL_REVISION == B_PROTOCOL["teacher"]["revision"]
assert B_QWEN_MODEL.training is False
assert all(
    p.device.type == "cuda"
    for p in B_QWEN_MODEL.parameters()
)

assert (
    B_INITIAL_MESSAGES
    == json.loads(B_PROMPT_PATH.read_text())[
        "initial_messages"
    ]
)

# 3. Enforce the registered session launch cutoff.
mountain_now = datetime.now(
    ZoneInfo("America/Denver")
)

hard_cutoff = datetime.fromisoformat(
    B_PROTOCOL["session"]["hard_cutoff"]
)

assert mountain_now < hard_cutoff, (
    "Registered session cutoff has passed. "
    "Do not launch a new live run."
)

# 4. Establish unique, non-overwritable output paths.
B_DIAGNOSIS_LIVE_DIR = (
    ROOT / "LIVE_DIAGNOSIS_FIRST_V1"
)

B_DIAGNOSIS_INVOCATION_PATH = (
    ROOT / "LIVE_DIAGNOSIS_FIRST_INVOCATION_V1.json"
)

assert not B_DIAGNOSIS_LIVE_DIR.exists(), (
    "Diagnosis-first run already has records. "
    "Do not repeat or overwrite it."
)

assert not B_DIAGNOSIS_INVOCATION_PATH.exists()

# 5. Freeze the invocation BEFORE first model exposure.
B_DIAGNOSIS_INVOCATION = {
    "experiment": EXPERIMENT_ID,
    "arm": "diagnosis_first",
    "run": 1,
    "status_at_invocation": "NOT_YET_EXPOSED",
    "patient_sha256": B_CORRUPT_SHA,
    "teacher_data_sha256": B_TEACHER_DATA_SHA,
    "model_id": B_MODEL_ID,
    "model_revision": B_MODEL_REVISION,
    "runtime_sha256": B_RUNTIME_SHA,
    "prompt_sha256": B_PROMPT_SHA,
    "tool_contract_sha256":
        B_MANIFEST["tool_contract_sha256"],
    "execution_rules_sha256":
        B_MANIFEST["execution_rules_sha256"],
    "loop_sha256": B_LOOP_SHA,
    "runner_code_sha256": B_RUNNER_CODE_SHA,
    "max_diagnostic_calls": 12,
    "max_patch_trials": 2,
    "max_total_actions": 20,
    "prompt_revisions_allowed": False,
    "automatic_retries_allowed": False,
    "sealed_evaluation": "UNOPENED",
}

B_DIAGNOSIS_INVOCATION_SHA = b_freeze_record(
    B_DIAGNOSIS_INVOCATION_PATH,
    B_DIAGNOSIS_INVOCATION
)

B_MANIFEST["teacher_status"] = (
    "DIAGNOSIS_FIRST_IN_PROGRESS"
)

print("Patient checkpoint: VERIFIED")
print("Teacher dataset: VERIFIED")
print("Model revision: VERIFIED")
print("Prompt and interface: VERIFIED")
print("Autonomous runner: VERIFIED")
print("Diagnostic budget: 12")
print("Patch trial budget: 2")
print("Maximum actions: 20")
print("Sealed evaluation: UNOPENED")
print()
print("Invocation SHA-256:")
print(B_DIAGNOSIS_INVOCATION_SHA)

print("\nRunning the frozen autonomous investigation...")
print("No human guidance will enter the conversation.\n")

# 6. Run the frozen controller exactly once.
# Every action is saved in its own hash-chained journal.
try:
    B_DIAGNOSIS_LIVE_RESULT = b_run_arm(
        arm="diagnosis_first",
        backend=B_LIVE_BACKEND,
        decide=b_live_decision,
        output_dir=B_DIAGNOSIS_LIVE_DIR,
    )

except Exception as exc:
    B_MANIFEST["teacher_status"] = (
        "DIAGNOSIS_FIRST_INTERRUPTED"
    )

    crash_record = {
        "experiment": EXPERIMENT_ID,
        "arm": "diagnosis_first",
        "status": "INTERRUPTED_NO_RETRY",
        "exception_type": type(exc).__name__,
        "exception_message": str(exc)[:2000],
        "invocation_sha256":
            B_DIAGNOSIS_INVOCATION_SHA,
        "sealed_evaluation": "UNOPENED",
    }

    crash_path = (
        ROOT / "LIVE_DIAGNOSIS_FIRST_CRASH.json"
    )

    if not crash_path.exists():
        crash_path.write_text(
            json.dumps(
                crash_record,
                indent=2,
                allow_nan=False
            )
        )

    print("\n❌ LIVE RUN INTERRUPTED")
    print("Do not rerun Cell 9.")
    print("Preserve the journal and crash record.")
    raise

# 7. Independently verify saved outcomes.
journal_path = (
    B_DIAGNOSIS_LIVE_DIR / "events.jsonl"
)

summary_path = (
    B_DIAGNOSIS_LIVE_DIR / "summary.json"
)

assert journal_path.exists()
assert summary_path.exists()

event_count = b_verify_journal(journal_path)

saved_summary = json.loads(
    summary_path.read_text()
)

assert saved_summary == B_DIAGNOSIS_LIVE_RESULT

assert 0 <= saved_summary["diagnostic_calls"] <= 12
assert 0 <= saved_summary["patch_trials"] <= 2
assert 0 <= saved_summary["total_actions"] <= 20

if saved_summary["status"] == "COMPLETED":
    assert saved_summary["final_patch_committed"]

    if saved_summary["diagnosis_committed"]:
        assert saved_summary["diagnosis"] is not None

B_MANIFEST["teacher_status"] = (
    "DIAGNOSIS_FIRST_"
    + saved_summary["status"]
)

B_MANIFEST["diagnostic_calls"] = (
    saved_summary["diagnostic_calls"]
)

B_MANIFEST["diagnosis_patch_trials"] = (
    saved_summary["patch_trials"]
)

# 8. Report only the result of this arm.
# Do not reveal the private fault key or score it yet.
print("\nDIAGNOSIS-FIRST TRIAL RESULT")
print("-" * 62)

print("Status:", saved_summary["status"])
print(
    "Diagnostic calls:",
    saved_summary["diagnostic_calls"],
    "/ 12"
)
print(
    "Patch trials:",
    saved_summary["patch_trials"],
    "/ 2"
)
print(
    "Total actions:",
    saved_summary["total_actions"],
    "/ 20"
)
print(
    "Diagnosis committed:",
    saved_summary["diagnosis_committed"]
)
print(
    "Final patch committed:",
    saved_summary["final_patch_committed"]
)

if saved_summary["final_patch_committed"]:
    print(
        "Final scalar edits:",
        len(saved_summary["final_patch"])
    )

print("Journal events verified:", event_count)
print("Sealed evaluation accessed: NO")

print("\nSaved evidence directory:")
print(B_DIAGNOSIS_LIVE_DIR)

print("\n✅ CELL 9: LIVE TRIAL RECORDED")


🌒 EX0004B — DIAGNOSIS-FIRST LIVE TRIAL


AssertionError: 

#Cell 9A: Diagnose the fingerprint mismatch

In [11]:

# EX0004B — Cell 9A: Runner fingerprint investigation
# READ ONLY. No new experiment or model generation.

import json
import hashlib
import marshal
import inspect
from pathlib import Path

print("🌒 EX0004B — PRE-EXPOSURE INTEGRITY CHECK")
print("=" * 60)

def code_fingerprint():
    return hashlib.sha256(
        marshal.dumps(b_run_arm.__code__)
        + marshal.dumps(b_live_decision.__code__)
    ).hexdigest()

stored = B_RUNNER_CODE_SHA
current_samples = [
    code_fingerprint()
    for _ in range(5)
]

loop_spec = json.loads(
    B_LOOP_SPEC_PATH.read_text()
)

invocation_path = (
    ROOT / "LIVE_DIAGNOSIS_FIRST_INVOCATION_V1.json"
)
live_dir = ROOT / "LIVE_DIAGNOSIS_FIRST_V1"

print("ORIGINAL FROZEN SHA:")
print(stored)

print("\nCURRENT SHA:")
print(current_samples[0])

print("\nCROSS-CHECKS")
print(
    "Current hash repeatable:",
    len(set(current_samples)) == 1
)
print(
    "Frozen manifest agrees:",
    B_MANIFEST["runner_code_sha256"] == stored
)
print(
    "Frozen loop specification agrees:",
    loop_spec["code_sha256"] == stored
)
print(
    "Function hash unchanged:",
    current_samples[0] == stored
)

print("\nFUNCTION IDENTITY")
for fn in (b_run_arm, b_live_decision):
    code = fn.__code__
    print(f"\n{fn.__name__}:")
    print("  Signature:", inspect.signature(fn))
    print("  File:", code.co_filename)
    print("  First line:", code.co_firstlineno)
    print(
        "  Code SHA:",
        hashlib.sha256(
            marshal.dumps(code)
        ).hexdigest()
    )

print("\nEXPOSURE BOUNDARY")
print("Invocation exists:", invocation_path.exists())
print("Live directory exists:", live_dir.exists())
print("Teacher state:", B_MANIFEST["teacher_status"])
print(
    "Frozen loop specification intact:",
    b_file_sha(B_LOOP_SPEC_PATH) == B_LOOP_SHA
)
print(
    "Public prompts intact:",
    b_file_sha(B_PROMPT_PATH) == B_PROMPT_SHA
)
print(
    "Corrupted patient intact:",
    b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA
)

assert not invocation_path.exists(), (
    "Invocation already exists: stop and investigate."
)
assert not live_dir.exists(), (
    "Live evidence already exists: stop and investigate."
)
assert B_MANIFEST["teacher_status"] == "LOADED_NOT_EXPOSED"

print("\n✅ CELL 9A: INVESTIGATION COMPLETE")
print("No experimental action performed.")


🌒 EX0004B — PRE-EXPOSURE INTEGRITY CHECK
ORIGINAL FROZEN SHA:
84b9e2778b6ed0b5187bf221603f2a5ded5d128b0caf4c18e40e7ac5454831f4

CURRENT SHA:
0ea514e09101ac8ffd8b832b2f24ce30b1d85bf5a309238c8e579f729dc6c673

CROSS-CHECKS
Current hash repeatable: True
Frozen manifest agrees: True
Frozen loop specification agrees: True
Function hash unchanged: False

FUNCTION IDENTITY

b_run_arm:
  Signature: (arm, backend, decide, output_dir)
  File: /tmp/ipykernel_844/1206323533.py
  First line: 338
  Code SHA: 342718fa9470c077e156b5a1e7891916132d39142f93337a9c73b07fe6159820

b_live_decision:
  Signature: (messages, arm, turn_number)
  File: /tmp/ipykernel_844/1206323533.py
  First line: 209
  Code SHA: 5e8b36b16ff1eb3407c2509d5e0f0a4f030f9cfd72223c58480ce1249c293805

EXPOSURE BOUNDARY
Invocation exists: False
Live directory exists: False
Teacher state: LOADED_NOT_EXPOSED
Frozen loop specification intact: True
Public prompts intact: True
Corrupted patient intact: True

✅ CELL 9A: INVESTIGATION COMPLETE


#Cell 9B: Verify original source and register amendment A1
This cell reconstructs the two functions from the original Cell 8 source, compares their bytecode and constants against the live functions, reruns the mock pathways, and saves a separate amendment receipt.

It does not load a new model or generate Qwen responses.

In [12]:

# EX0004B — Cell 9B: Source-verified integrity amendment A1
# No live Qwen generation. No C06 diagnostic observations.
# Do not overwrite the original loop specification.

import ast
import types
import json
import hashlib
import marshal
from pathlib import Path

print("🌒 EX0004B — SOURCE INTEGRITY REBASELINE")
print("=" * 62)

# 1. Preserve the pre-exposure boundary.
assert B_MANIFEST["teacher_status"] == "LOADED_NOT_EXPOSED"
assert not (
    ROOT / "LIVE_DIAGNOSIS_FIRST_V1"
).exists()
assert not (
    ROOT / "LIVE_DIAGNOSIS_FIRST_INVOCATION_V1.json"
).exists()

assert b_file_sha(B_PROMPT_PATH) == B_PROMPT_SHA
assert b_file_sha(B_LOOP_SPEC_PATH) == B_LOOP_SHA
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA

# 2. Recover the original Cell 8 source from IPython history.
history = get_ipython().history_manager.input_hist_raw

matches = [
    (i, src)
    for i, src in enumerate(history)
    if (
        "def b_live_decision(" in src
        and "def b_run_arm(" in src
        and "B_RUNNER_CODE_SHA =" in src
        and "B_MOCK_ACTIONS =" in src
    )
]

assert matches, (
    "Original Cell 8 source unavailable. "
    "Stop rather than accepting an unverified rebaseline."
)

# Multiple identical executions are permissible.
source_hashes = {
    hashlib.sha256(src.encode()).hexdigest()
    for _, src in matches
}

assert len(source_hashes) == 1, (
    "Conflicting versions of Cell 8 found in history."
)

history_index, original_source = matches[0]

source_sha = hashlib.sha256(
    original_source.encode()
).hexdigest()

# 3. Compile original source WITHOUT executing it.
source_tree = ast.parse(original_source)

source_functions = {
    node.name: node
    for node in source_tree.body
    if isinstance(node, ast.FunctionDef)
}

module_code = compile(
    original_source,
    filename="<EX0004B_ORIGINAL_CELL8>",
    mode="exec",
    dont_inherit=True,
)

compiled_functions = {
    item.co_name: item
    for item in module_code.co_consts
    if isinstance(item, types.CodeType)
}

# 4. Canonical structural fingerprint.
# Excludes runtime filename and mutable interpreter caches.
def b_code_structure(co):
    def encode_const(value):
        if isinstance(value, types.CodeType):
            return {
                "type": "code",
                "value": b_code_structure(value),
            }
        if isinstance(value, tuple):
            return {
                "type": "tuple",
                "items": [
                    encode_const(v) for v in value
                ],
            }
        return {
            "type": type(value).__name__,
            "repr": repr(value),
        }

    return {
        "bytecode": co.co_code.hex(),
        "exception_table": co.co_exceptiontable.hex(),
        "constants": [
            encode_const(c) for c in co.co_consts
        ],
        "names": list(co.co_names),
        "varnames": list(co.co_varnames),
        "freevars": list(co.co_freevars),
        "cellvars": list(co.co_cellvars),
        "flags": co.co_flags,
        "argcount": co.co_argcount,
        "posonlyargcount": co.co_posonlyargcount,
        "kwonlyargcount": co.co_kwonlyargcount,
        "stacksize": co.co_stacksize,
    }

def b_structural_sha(co):
    canonical = json.dumps(
        b_code_structure(co),
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(
        canonical.encode()
    ).hexdigest()

live_functions = {
    "b_run_arm": b_run_arm,
    "b_live_decision": b_live_decision,
}

function_results = {}

for name, fn in live_functions.items():
    assert name in source_functions
    assert name in compiled_functions

    original_code = compiled_functions[name]
    live_code = fn.__code__

    original_sha = b_structural_sha(original_code)
    live_sha = b_structural_sha(live_code)

    assert (
        live_code.co_firstlineno
        == source_functions[name].lineno
    ), f"{name}: unexpected source location"

    assert original_sha == live_sha, (
        f"{name}: executable structure differs "
        "from original Cell 8. STOP."
    )

    function_results[name] = original_sha

    print(f"{name}: STRUCTURAL MATCH")
    print(f"  SHA-256: {live_sha}")

# 5. Preserve both marshal fingerprints.
original_marshal = B_RUNNER_CODE_SHA

current_marshal = hashlib.sha256(
    marshal.dumps(b_run_arm.__code__)
    + marshal.dumps(b_live_decision.__code__)
).hexdigest()

assert original_marshal == (
    B_MANIFEST["runner_code_sha256"]
)

# 6. Reexecute the two MOCK pathways only.
# No Qwen calls, no C06 observations.
A1_MOCK_ROOT = ROOT / "MOCK_AUTONOMOUS_A1"

assert not A1_MOCK_ROOT.exists()

mock_first = b_run_arm(
    "diagnosis_first",
    mock_backend,
    b_mock_decision,
    A1_MOCK_ROOT / "diagnosis_first",
)

mock_second = b_run_arm(
    "repair_only",
    mock_backend,
    b_mock_decision,
    A1_MOCK_ROOT / "repair_only",
)

assert mock_first["status"] == "COMPLETED"
assert mock_second["status"] == "COMPLETED"
assert mock_first["diagnosis_committed"]
assert mock_first["final_patch_committed"]
assert mock_second["final_patch_committed"]

assert b_verify_journal(
    A1_MOCK_ROOT / "diagnosis_first/events.jsonl"
) > 0

assert b_verify_journal(
    A1_MOCK_ROOT / "repair_only/events.jsonl"
) > 0

# 7. Preserve original source as an audit artifact.
A1_SOURCE_PATH = ROOT / "CELL8_EXECUTOR_SOURCE_A1.py"

assert not A1_SOURCE_PATH.exists()
A1_SOURCE_PATH.write_text(original_source)

assert b_file_sha(A1_SOURCE_PATH) == source_sha

# 8. Freeze a separate pre-exposure amendment.
# Original protocol and loop specification remain intact.
A1_RECORD = {
    "experiment": EXPERIMENT_ID,
    "amendment": "A1_PRE_EXPOSURE_CODE_INTEGRITY",
    "reason": (
        "marshal-based code fingerprint mismatch "
        "before first live teacher exposure"
    ),
    "original_loop_sha256": B_LOOP_SHA,
    "original_marshal_sha256": original_marshal,
    "observed_marshal_sha256": current_marshal,
    "original_cell_history_index": history_index,
    "original_source_sha256": source_sha,
    "verified_structural_sha256": function_results,
    "verification_method": (
        "Original notebook source compiled without "
        "execution and compared recursively against "
        "live function bytecode, constants, names, "
        "arguments and exception tables."
    ),
    "mock_diagnosis_first": mock_first["status"],
    "mock_repair_only": mock_second["status"],
    "scientific_protocol_changed": False,
    "teacher_prompts_changed": False,
    "patient_changed": False,
    "function_implementations_changed": False,
    "live_teacher_calls": 0,
    "sealed_evaluation": "UNOPENED",
    "authorization": (
        "Use source-verified structural fingerprints "
        "in place of the original marshal equality "
        "assertion for live launch."
    ),
}

A1_PATH = ROOT / "CODE_INTEGRITY_AMENDMENT_A1.json"

assert not A1_PATH.exists()

A1_SHA = b_freeze_record(
    A1_PATH, A1_RECORD
)

B_MANIFEST["integrity_amendment_sha256"] = A1_SHA
B_MANIFEST["integrity_status"] = (
    "SOURCE_VERIFIED_PRE_EXPOSURE_A1"
)

print("\nINTEGRITY VERDICT")
print("-" * 62)
print("Original Cell 8 source recovered: YES")
print("Live functions structurally match source: YES")
print("Diagnosis-first mock replay: PASS")
print("Repair-only mock replay: PASS")
print("Original frozen loop preserved: YES")
print("Teacher prompts modified: NO")
print("Patient modified: NO")
print("Live Qwen calls: 0")
print("Sealed evaluation accessed: NO")

print("\nOriginal source SHA-256:", source_sha)
print("A1 amendment SHA-256:", A1_SHA)
print("\n✅ CELL 9B: SOURCE INTEGRITY VERIFIED")


🌒 EX0004B — SOURCE INTEGRITY REBASELINE


AssertionError: Conflicting versions of Cell 8 found in history.

#Cell 9C: Verify the active functions against their source
This is a replacement for Cell 9B. Do not rerun Cell 9B.

In [13]:
# EX0004B — Cell 9C: Source-aware integrity verification
# PRE-EXPOSURE ONLY. No Qwen generation or C06 probes.

import ast
import types
import json
import hashlib
import marshal
from pathlib import Path

print("🌒 EX0004B — EXECUTABLE SOURCE VERIFICATION")
print("=" * 62)

# 1. Confirm the experimental boundary.
assert B_MANIFEST["teacher_status"] == "LOADED_NOT_EXPOSED"
assert B_MANIFEST["loop_status"] == "FULL_PREFLIGHT_PASS"

assert not (
    ROOT / "LIVE_DIAGNOSIS_FIRST_V1"
).exists()

assert not (
    ROOT / "LIVE_DIAGNOSIS_FIRST_INVOCATION_V1.json"
).exists()

assert b_file_sha(B_PROMPT_PATH) == B_PROMPT_SHA
assert b_file_sha(B_LOOP_SPEC_PATH) == B_LOOP_SHA
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA

# 2. Build a stable structural representation.
# Ignore filenames and line-number metadata.
def b9c_const(value):
    if isinstance(value, types.CodeType):
        return ["code", b9c_structure(value)]

    if isinstance(value, tuple):
        return [
            "tuple",
            [b9c_const(v) for v in value]
        ]

    if isinstance(value, frozenset):
        return [
            "frozenset",
            sorted(repr(v) for v in value)
        ]

    return [type(value).__name__, repr(value)]


def b9c_structure(code):
    return {
        "bytecode": code.co_code.hex(),
        "exception_table": code.co_exceptiontable.hex(),
        "constants": [
            b9c_const(v) for v in code.co_consts
        ],
        "names": list(code.co_names),
        "varnames": list(code.co_varnames),
        "freevars": list(code.co_freevars),
        "cellvars": list(code.co_cellvars),
        "flags": code.co_flags,
        "argcount": code.co_argcount,
        "posonlyargcount": code.co_posonlyargcount,
        "kwonlyargcount": code.co_kwonlyargcount,
        "stacksize": code.co_stacksize,
    }


def b9c_sha(code):
    payload = json.dumps(
        b9c_structure(code),
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(
        payload.encode()
    ).hexdigest()


live = {
    "b_run_arm": b_run_arm.__code__,
    "b_live_decision": b_live_decision.__code__,
}

live_hashes = {
    name: b9c_sha(code)
    for name, code in live.items()
}

# 3. Search all history entries instead of demanding
# that every matching text entry be identical.
history = get_ipython().history_manager.input_hist_raw

candidates_seen = 0
verified_matches = []

for history_index, source in enumerate(history):
    if not (
        "def b_run_arm(" in source
        and "def b_live_decision(" in source
        and "B_RUNNER_CODE_SHA =" in source
    ):
        continue

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    top_level_functions = {
        node.name
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
    }

    if not set(live).issubset(top_level_functions):
        continue

    candidates_seen += 1

    compiled = compile(
        source,
        f"<EX0004B_HISTORY_{history_index}>",
        "exec",
        dont_inherit=True,
    )

    compiled_functions = {
        obj.co_name: obj
        for obj in compiled.co_consts
        if isinstance(obj, types.CodeType)
    }

    if not set(live).issubset(compiled_functions):
        continue

    source_hashes = {
        name: b9c_sha(compiled_functions[name])
        for name in live
    }

    structures_match = source_hashes == live_hashes

    positions_match = all(
        compiled_functions[name].co_firstlineno
        == live[name].co_firstlineno
        for name in live
    )

    if structures_match and positions_match:
        verified_matches.append({
            "index": history_index,
            "source": source,
            "source_sha256": hashlib.sha256(
                source.encode()
            ).hexdigest(),
        })

print("Candidate source entries:", candidates_seen)
print("Executable matches:", len(verified_matches))

assert verified_matches, (
    "No historical source matches both live functions. "
    "STOP: do not override the integrity guard."
)

# Multiple matching versions are permitted.
# Select the most recent exact executable match.
selected = verified_matches[-1]

original_source = selected["source"]
source_sha = selected["source_sha256"]

print("Selected history entry:", selected["index"])

for name, fingerprint in live_hashes.items():
    print(f"{name}: VERIFIED")
    print("  Structural SHA-256:", fingerprint)

# 4. Repeat complete mock execution with the
# verified functions. No C06 patient observations.
A1_MOCK_ROOT = ROOT / "MOCK_AUTONOMOUS_A1"

assert not A1_MOCK_ROOT.exists()

mock_first = b_run_arm(
    "diagnosis_first",
    mock_backend,
    b_mock_decision,
    A1_MOCK_ROOT / "diagnosis_first",
)

mock_second = b_run_arm(
    "repair_only",
    mock_backend,
    b_mock_decision,
    A1_MOCK_ROOT / "repair_only",
)

assert mock_first["status"] == "COMPLETED"
assert mock_second["status"] == "COMPLETED"
assert mock_first["diagnosis_committed"]
assert mock_first["final_patch_committed"]
assert mock_second["final_patch_committed"]

assert b_verify_journal(
    A1_MOCK_ROOT / "diagnosis_first/events.jsonl"
) > 0

assert b_verify_journal(
    A1_MOCK_ROOT / "repair_only/events.jsonl"
) > 0

# 5. Archive source without modifying any
# original experimental artifact.
A1_SOURCE_PATH = ROOT / "CELL8_VERIFIED_SOURCE_A1.py"
A1_PATH = ROOT / "CODE_INTEGRITY_AMENDMENT_A1.json"

assert not A1_SOURCE_PATH.exists()
assert not A1_PATH.exists()

A1_SOURCE_PATH.write_text(original_source)

assert b_file_sha(A1_SOURCE_PATH) == source_sha

current_marshal = hashlib.sha256(
    marshal.dumps(b_run_arm.__code__)
    + marshal.dumps(b_live_decision.__code__)
).hexdigest()

A1_RECORD = {
    "experiment": EXPERIMENT_ID,
    "amendment": "A1_PRE_EXPOSURE_CODE_INTEGRITY",
    "reason": (
        "Original marshal fingerprint mismatch; "
        "source-history ambiguity resolved by "
        "executable structural comparison."
    ),
    "original_loop_sha256": B_LOOP_SHA,
    "original_marshal_sha256": B_RUNNER_CODE_SHA,
    "observed_marshal_sha256": current_marshal,
    "matching_history_entries": [
        {
            "index": entry["index"],
            "source_sha256": entry["source_sha256"],
        }
        for entry in verified_matches
    ],
    "selected_source_sha256": source_sha,
    "structural_sha256": live_hashes,
    "mock_diagnosis_first": mock_first["status"],
    "mock_repair_only": mock_second["status"],
    "protocol_changed": False,
    "patient_changed": False,
    "prompts_changed": False,
    "function_implementations_changed": False,
    "live_teacher_calls": 0,
    "sealed_evaluation": "UNOPENED",
}

A1_SHA = b_freeze_record(A1_PATH, A1_RECORD)

B_MANIFEST["integrity_amendment_sha256"] = A1_SHA
B_MANIFEST["integrity_status"] = (
    "SOURCE_VERIFIED_PRE_EXPOSURE_A1"
)

print("\nINTEGRITY VERDICT")
print("-" * 62)
print("Both functions match original source: YES")
print("Diagnosis-first mock: PASS")
print("Repair-only mock: PASS")
print("Original loop specification: PRESERVED")
print("Teacher prompts: UNCHANGED")
print("Patient C06: UNCHANGED")
print("Qwen diagnostic calls: 0")
print("Sealed evaluation: UNOPENED")

print("\nVerified source SHA-256:")
print(source_sha)

print("\nA1 amendment SHA-256:")
print(A1_SHA)

print("\n✅ CELL 9C: EXECUTABLE SOURCE VERIFIED")

🌒 EX0004B — EXECUTABLE SOURCE VERIFICATION
Candidate source entries: 1
Executable matches: 0


AssertionError: No historical source matches both live functions. STOP: do not override the integrity guard.

#Cell 9D: Identify the exact mismatch
This compares the functions field by field and checks whether IPython can recover the original function definitions.

In [14]:

# EX0004B — Cell 9D: Pinpoint code-object differences
# READ ONLY. No Qwen generation, patient probes, or amendments.

import ast
import inspect
import textwrap
import hashlib
import json
import types
from pathlib import Path

print("🌒 EX0004B — CODE FINGERPRINT FORENSICS")
print("=" * 62)

assert B_MANIFEST["teacher_status"] == "LOADED_NOT_EXPOSED"
assert not (ROOT / "LIVE_DIAGNOSIS_FIRST_V1").exists()
assert not (
    ROOT / "LIVE_DIAGNOSIS_FIRST_INVOCATION_V1.json"
).exists()

assert b_file_sha(B_PROMPT_PATH) == B_PROMPT_SHA
assert b_file_sha(B_LOOP_SPEC_PATH) == B_LOOP_SHA
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA

# Recover genuine source entries by AST, not text matching.
matches = []

for i, source in enumerate(
    get_ipython().history_manager.input_hist_raw
):
    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    definitions = {
        node.name: node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
    }

    if {"b_run_arm", "b_live_decision"} <= set(definitions):
        matches.append((i, source, definitions))

print("Genuine source entries:", len(matches))

assert matches, "Original function source not found."

live_functions = {
    "b_run_arm": b_run_arm,
    "b_live_decision": b_live_decision,
}

for history_id, source, definitions in matches:
    compiled = compile(
        source,
        f"<history_{history_id}>",
        "exec",
        dont_inherit=True,
    )

    compiled_codes = {
        item.co_name: item
        for item in compiled.co_consts
        if isinstance(item, types.CodeType)
    }

    print(f"\nSOURCE ENTRY: {history_id}")

    for name, fn in live_functions.items():
        assert name in compiled_codes

        live_code = fn.__code__
        source_code = compiled_codes[name]

        # Defined in Cell 9C, before its failed assertion.
        live_fields = b9c_structure(live_code)
        source_fields = b9c_structure(source_code)

        differences = [
            key
            for key in live_fields
            if live_fields[key] != source_fields[key]
        ]

        print(f"\n{name}")
        print("  Live first line:", live_code.co_firstlineno)
        print("  Source first line:", source_code.co_firstlineno)
        print("  Different fields:", differences or "NONE")

        print(
            "  Bytecode identical:",
            live_code.co_code == source_code.co_code
        )

        print(
            "  Names identical:",
            live_code.co_names == source_code.co_names
        )

        print(
            "  Constants structurally identical:",
            live_fields["constants"] ==
            source_fields["constants"]
        )

        # Inspect the currently loaded function's source.
        try:
            recovered = inspect.getsource(fn)
            recovered_tree = ast.parse(
                textwrap.dedent(recovered)
            )

            expected_ast = ast.dump(
                definitions[name],
                include_attributes=False,
            )

            recovered_ast = ast.dump(
                recovered_tree.body[0],
                include_attributes=False,
            )

            print(
                "  Live-source AST matches history:",
                expected_ast == recovered_ast
            )

        except (OSError, TypeError, SyntaxError) as exc:
            print(
                "  Live-source recovery:",
                type(exc).__name__
            )

        # Display only compact fingerprints of differences.
        for field in differences:
            left = json.dumps(
                live_fields[field],
                sort_keys=True,
            )
            right = json.dumps(
                source_fields[field],
                sort_keys=True,
            )

            print(f"  DIFFERENCE: {field}")
            print(
                "    Live SHA:",
                hashlib.sha256(left.encode()).hexdigest()[:16]
            )
            print(
                "    Source SHA:",
                hashlib.sha256(right.encode()).hexdigest()[:16]
            )

print("\nBOUNDARY")
print("Live teacher calls: 0")
print("Patient C06: UNCHANGED")
print("Frozen prompts: UNCHANGED")
print("Sealed evaluation: UNOPENED")
print("Amendment written: NO")
print("\n✅ CELL 9D: DIFFERENCES IDENTIFIED")


🌒 EX0004B — CODE FINGERPRINT FORENSICS
Genuine source entries: 1

SOURCE ENTRY: 8

b_run_arm
  Live first line: 338
  Source first line: 339
  Different fields: ['bytecode']
  Bytecode identical: False
  Names identical: True
  Constants structurally identical: True
  Live-source AST matches history: True
  DIFFERENCE: bytecode
    Live SHA: e5b6c1ae31bb8374
    Source SHA: 7f782cf52fdcad79

b_live_decision
  Live first line: 209
  Source first line: 210
  Different fields: ['bytecode', 'exception_table']
  Bytecode identical: False
  Names identical: True
  Constants structurally identical: True
  Live-source AST matches history: True
  DIFFERENCE: bytecode
    Live SHA: 3352bec95b0a0139
    Source SHA: 03f17b436cc3b0ca
  DIFFERENCE: exception_table
    Live SHA: 1da90fae2ef877b7
    Source SHA: b52840c5e64a63cc

BOUNDARY
Live teacher calls: 0
Patient C06: UNCHANGED
Frozen prompts: UNCHANGED
Sealed evaluation: UNOPENED
Amendment written: NO

✅ CELL 9D: DIFFERENCES IDENTIFIED


#Cell 9E: Register A1 and launch the original autonomous trial

In [15]:

# EX0004B — Cell 9E: A1-qualified live launch
# Preserve original protocol, prompts, patient, and runner.
# No binary-equivalence claim.
# Exactly ONE diagnosis-first invocation.

import ast
import inspect
import textwrap
import json
import hashlib
import marshal
import traceback
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

print("🌒 EX0004B — A1-QUALIFIED LIVE LAUNCH")
print("=" * 62)

# 1. Verify no previous live exposure.
assert B_MANIFEST["teacher_status"] == "LOADED_NOT_EXPOSED"
assert B_MANIFEST["loop_status"] == "FULL_PREFLIGHT_PASS"

B_DIAGNOSIS_LIVE_DIR = ROOT / "LIVE_DIAGNOSIS_FIRST_V1"
B_DIAGNOSIS_INVOCATION_PATH = (
    ROOT / "LIVE_DIAGNOSIS_FIRST_INVOCATION_V1.json"
)

assert not B_DIAGNOSIS_LIVE_DIR.exists()
assert not B_DIAGNOSIS_INVOCATION_PATH.exists()

# 2. Verify the entire frozen artifact chain.
frozen_files = [
    (B_PROTOCOL_PATH, B_PROTOCOL_SHA),
    (B_HEALTHY_PATH, B_HEALTHY_SHA),
    (B_FACTORY_PATH, B_FACTORY_SHA),
    (B_CORRUPT_PATH, B_CORRUPT_SHA),
    (B_AUDIT_PATH, B_MANIFEST[
        "construction_audit_sha256"
    ]),
    (schema_path, B_MANIFEST[
        "tool_contract_sha256"
    ]),
    (B_PUBLIC_GUIDE_PATH, B_PUBLIC_GUIDE_SHA),
    (B_TEACHER_RECEIPT_PATH, B_RUNTIME_SHA),
    (B_PROMPT_PATH, B_PROMPT_SHA),
    (B_LOOP_SPEC_PATH, B_LOOP_SHA),
]

for path, expected in frozen_files:
    assert b_file_sha(path) == expected, str(path)

assert b_dataset_fingerprint(
    B_TEACHER_DATA
) == B_TEACHER_DATA_SHA

assert all(
    torch.equal(
        B_LIVE_BACKEND.base_state[k],
        torch.load(
            B_CORRUPT_PATH,
            map_location="cpu",
            weights_only=True,
        )["model_state"][k],
    )
    for k in B_LIVE_BACKEND.base_state
)

assert B_QWEN_MODEL.training is False
assert B_MODEL_REVISION == B_PROTOCOL["teacher"]["revision"]

# 3. Recover the original Cell 8 source.
source_candidates = []

for i, source in enumerate(
    get_ipython().history_manager.input_hist_raw
):
    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    definitions = {
        node.name: node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
    }

    if {"b_run_arm", "b_live_decision"} <= set(definitions):
        source_candidates.append(
            (i, source, definitions)
        )

assert len(source_candidates) == 1, (
    "Ambiguous original executor source. STOP."
)

history_index, original_source, definitions = (
    source_candidates[0]
)

# 4. Compare live functions with original source AST.
functions = {
    "b_run_arm": b_run_arm,
    "b_live_decision": b_live_decision,
}

source_ast_hashes = {}

for name, fn in functions.items():
    live_source = textwrap.dedent(
        inspect.getsource(fn)
    )

    live_tree = ast.parse(live_source)

    live_node = live_tree.body[0]
    original_node = definitions[name]

    live_ast = ast.dump(
        live_node,
        include_attributes=False
    )

    original_ast = ast.dump(
        original_node,
        include_attributes=False
    )

    assert live_ast == original_ast, (
        f"{name}: source AST mismatch. STOP."
    )

    source_ast_hashes[name] = hashlib.sha256(
        original_ast.encode()
    ).hexdigest()

print("Original Cell 8 source: RECOVERED")
print("Live runner source AST: MATCH")
print("Live generation source AST: MATCH")

# 5. Repeat both mock pathways using the live functions.
# Only the synthetic mock backend is used.
A1_MOCK_ROOT = ROOT / "MOCK_PREEXPOSURE_A1"

assert not A1_MOCK_ROOT.exists()

mock_diagnosis = b_run_arm(
    "diagnosis_first",
    mock_backend,
    b_mock_decision,
    A1_MOCK_ROOT / "diagnosis_first",
)

mock_repair = b_run_arm(
    "repair_only",
    mock_backend,
    b_mock_decision,
    A1_MOCK_ROOT / "repair_only",
)

assert mock_diagnosis["status"] == "COMPLETED"
assert mock_diagnosis["diagnosis_committed"]
assert mock_diagnosis["final_patch_committed"]

assert mock_repair["status"] == "COMPLETED"
assert mock_repair["final_patch_committed"]

assert b_verify_journal(
    A1_MOCK_ROOT / "diagnosis_first/events.jsonl"
) > 0

assert b_verify_journal(
    A1_MOCK_ROOT / "repair_only/events.jsonl"
) > 0

print("Diagnosis-first mock replay: PASS")
print("Repair-only mock replay: PASS")
print("Journal integrity: PASS")

# 6. Record the binary mismatch transparently.
observed_marshal_sha = hashlib.sha256(
    marshal.dumps(b_run_arm.__code__)
    + marshal.dumps(b_live_decision.__code__)
).hexdigest()

assert observed_marshal_sha != B_RUNNER_CODE_SHA, (
    "Binary discrepancy unexpectedly disappeared. "
    "Investigate instead of filing an inaccurate amendment."
)

A1_SOURCE_PATH = ROOT / "CELL8_SOURCE_A1.py"
A1_PATH = ROOT / "CODE_INTEGRITY_AMENDMENT_A1.json"

assert not A1_SOURCE_PATH.exists()
assert not A1_PATH.exists()

A1_SOURCE_PATH.write_text(original_source)

original_source_sha = b_file_sha(
    A1_SOURCE_PATH
)

A1_RECORD = {
    "experiment": EXPERIMENT_ID,
    "amendment": "A1_PRE_EXPOSURE",
    "reason": (
        "Original marshal-based executable fingerprint "
        "differs from the current notebook function fingerprint."
    ),
    "original_loop_sha256": B_LOOP_SHA,
    "original_runner_marshal_sha256": B_RUNNER_CODE_SHA,
    "observed_runner_marshal_sha256": observed_marshal_sha,
    "original_source_history_entry": history_index,
    "original_source_sha256": original_source_sha,
    "verified_source_ast_hashes": source_ast_hashes,
    "source_ast_matches": True,
    "binary_equivalence_established": False,
    "binary_difference_cause": "UNRESOLVED",
    "mock_diagnosis_first": mock_diagnosis["status"],
    "mock_repair_only": mock_repair["status"],
    "decision": (
        "Proceed with source-level verification and "
        "passing behavioral mock tests. Preserve residual "
        "binary-fingerprint uncertainty in final report."
    ),
    "scientific_protocol_changed": False,
    "teacher_prompts_changed": False,
    "patient_changed": False,
    "runner_implementation_edited": False,
    "diagnostic_budget_changed": False,
    "live_teacher_calls_before_amendment": 0,
    "sealed_evaluation": "UNOPENED",
}

A1_SHA = b_freeze_record(
    A1_PATH,
    A1_RECORD
)

B_MANIFEST["integrity_amendment_sha256"] = A1_SHA
B_MANIFEST["integrity_status"] = (
    "SOURCE_VERIFIED_A1_BINARY_UNRESOLVED"
)

print("\nA1 AMENDMENT")
print("Source-level match: VERIFIED")
print("Behavioral mock replay: PASS")
print("Binary discrepancy: DOCUMENTED, UNRESOLVED")
print("Patient changed: NO")
print("Prompts changed: NO")
print("Runner changed: NO")
print("Teacher observations: 0")
print("A1 SHA-256:", A1_SHA)

# 7. Verify the session's registered launch cutoff.
now = datetime.now(
    ZoneInfo("America/Denver")
)

cutoff = datetime.fromisoformat(
    B_PROTOCOL["session"]["hard_cutoff"]
)

assert now < cutoff, (
    "Registered midnight cutoff passed. "
    "Do not start another live invocation."
)

# 8. Freeze the qualified invocation BEFORE model exposure.
invocation = {
    "experiment": EXPERIMENT_ID,
    "arm": "diagnosis_first",
    "run": 1,
    "integrity_amendment": "A1_PRE_EXPOSURE",
    "integrity_amendment_sha256": A1_SHA,
    "integrity_qualification": (
        "Source AST verified; executable binary "
        "fingerprint discrepancy unresolved."
    ),
    "patient_sha256": B_CORRUPT_SHA,
    "teacher_data_sha256": B_TEACHER_DATA_SHA,
    "model_id": B_MODEL_ID,
    "model_revision": B_MODEL_REVISION,
    "runtime_sha256": B_RUNTIME_SHA,
    "prompt_sha256": B_PROMPT_SHA,
    "loop_sha256": B_LOOP_SHA,
    "tool_contract_sha256": B_MANIFEST[
        "tool_contract_sha256"
    ],
    "maximum_diagnostic_calls": 12,
    "maximum_patch_trials": 2,
    "maximum_total_actions": 20,
    "prompt_revisions_allowed": False,
    "automatic_retries_allowed": False,
    "sealed_evaluation": "UNOPENED",
}

B_DIAGNOSIS_INVOCATION_SHA = b_freeze_record(
    B_DIAGNOSIS_INVOCATION_PATH,
    invocation
)

B_MANIFEST["teacher_status"] = (
    "DIAGNOSIS_FIRST_IN_PROGRESS"
)

print("\nLIVE INVOCATION")
print("Invocation SHA-256:", B_DIAGNOSIS_INVOCATION_SHA)
print("C06: VERIFIED")
print("Qwen: LOADED")
print("Diagnostic budget: 12")
print("Patch trial budget: 2")
print("Sealed evaluation: UNOPENED")

print("\n🌒 Beginning autonomous diagnosis-first trial...\n")

# 9. Invoke the ORIGINAL frozen runner exactly once.
try:
    B_DIAGNOSIS_LIVE_RESULT = b_run_arm(
        arm="diagnosis_first",
        backend=B_LIVE_BACKEND,
        decide=b_live_decision,
        output_dir=B_DIAGNOSIS_LIVE_DIR,
    )

except Exception as exc:
    B_MANIFEST["teacher_status"] = (
        "DIAGNOSIS_FIRST_INTERRUPTED"
    )

    crash_path = (
        ROOT / "LIVE_DIAGNOSIS_FIRST_CRASH.json"
    )

    crash_record = {
        "experiment": EXPERIMENT_ID,
        "status": "INTERRUPTED_NO_RETRY",
        "exception_type": type(exc).__name__,
        "message": str(exc)[:2000],
        "traceback": traceback.format_exc(limit=8),
        "invocation_sha256": B_DIAGNOSIS_INVOCATION_SHA,
        "sealed_evaluation": "UNOPENED",
    }

    if not crash_path.exists():
        crash_path.write_text(
            json.dumps(
                crash_record,
                indent=2,
                allow_nan=False,
            )
        )

    print("\n❌ LIVE RUN INTERRUPTED")
    print("Do not rerun Cell 9E.")
    print("Preserve the invocation and event journal.")
    raise

# 10. Verify the completed or bounded outcome.
journal = B_DIAGNOSIS_LIVE_DIR / "events.jsonl"
summary_path = B_DIAGNOSIS_LIVE_DIR / "summary.json"

event_count = b_verify_journal(journal)

saved = json.loads(summary_path.read_text())

assert saved == B_DIAGNOSIS_LIVE_RESULT
assert 0 <= saved["diagnostic_calls"] <= 12
assert 0 <= saved["patch_trials"] <= 2
assert 0 <= saved["total_actions"] <= 20

B_MANIFEST["teacher_status"] = (
    "DIAGNOSIS_FIRST_" + saved["status"]
)

B_MANIFEST["diagnostic_calls"] = (
    saved["diagnostic_calls"]
)

B_MANIFEST["diagnosis_patch_trials"] = (
    saved["patch_trials"]
)

print("\nDIAGNOSIS-FIRST RESULT")
print("=" * 62)
print("Status:", saved["status"])
print("Diagnostic calls:", saved["diagnostic_calls"], "/ 12")
print("Patch trials:", saved["patch_trials"], "/ 2")
print("Total actions:", saved["total_actions"], "/ 20")
print("Diagnosis committed:", saved["diagnosis_committed"])
print("Final patch committed:", saved["final_patch_committed"])

if saved["final_patch_committed"]:
    print("Final scalar edits:", len(saved["final_patch"]))

print("Journal events verified:", event_count)
print("Sealed evaluation accessed: NO")
print("A1 integrity qualification: PRESERVED")
print("\n✅ CELL 9E: LIVE TRIAL OUTCOME RECORDED")


🌒 EX0004B — A1-QUALIFIED LIVE LAUNCH
Original Cell 8 source: RECOVERED
Live runner source AST: MATCH
Live generation source AST: MATCH
Diagnosis-first mock replay: PASS
Repair-only mock replay: PASS
Journal integrity: PASS

A1 AMENDMENT
Source-level match: VERIFIED
Behavioral mock replay: PASS
Binary discrepancy: DOCUMENTED, UNRESOLVED
Patient changed: NO
Prompts changed: NO
Runner changed: NO
Teacher observations: 0
A1 SHA-256: 385b796540ce20223b65fb91589525a93fe47515467a657d953bc612d5bc81e8

LIVE INVOCATION
Invocation SHA-256: 19dd321c6883b765650754762c0e79b500cbac13648d65229b9cbc7b243d9b5d
C06: VERIFIED
Qwen: LOADED
Diagnostic budget: 12
Patch trial budget: 2
Sealed evaluation: UNOPENED

🌒 Beginning autonomous diagnosis-first trial...


DIAGNOSIS-FIRST RESULT
Status: COMPLETED
Diagnostic calls: 0 / 12
Patch trials: 2 / 2
Total actions: 4 / 20
Diagnosis committed: True
Final patch committed: True
Final scalar edits: 0
Journal events verified: 13
Sealed evaluation accessed: NO
A1 inte

#Cell 10: Run the repair-only control
One uninterrupted control trial · Same C06 patient · Independent generation seed

This cell verifies the completed diagnosis-first record and the original frozen experimental configuration, then launches the repair-only arm.

It uses the same Qwen model, diagnostic tools, 12-call budget, two patch trials, and C06 checkpoint.

The only experimental difference is the previously frozen repair-only instruction condition.

Qwen is not required to submit a mechanistic diagnosis.

In [16]:

# EX0004B — Cell 10: Live repair-only control
# Frozen experiment. No prompt edits or adaptive retries.
# No sealed evaluation or private rubric access.

import ast
import inspect
import textwrap
import json
import hashlib
import traceback
import torch
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

print("🌒 EX0004B — REPAIR-ONLY CONTROL")
print("=" * 62)

# 1. Verify the diagnosis-first trial is final.
assert B_MANIFEST["teacher_status"] == (
    "DIAGNOSIS_FIRST_COMPLETED"
)

first_dir = ROOT / "LIVE_DIAGNOSIS_FIRST_V1"
first_summary_path = first_dir / "summary.json"
first_journal_path = first_dir / "events.jsonl"

assert first_summary_path.exists()
assert first_journal_path.exists()

first_summary = json.loads(
    first_summary_path.read_text()
)

assert first_summary["status"] == "COMPLETED"
assert first_summary["diagnosis_committed"]
assert first_summary["final_patch_committed"]
assert first_summary["diagnostic_calls"] == 0
assert first_summary["patch_trials"] == 2
assert len(first_summary["final_patch"]) == 0

assert b_verify_journal(first_journal_path) == 13

first_summary_sha = b_file_sha(first_summary_path)

# 2. Verify frozen experimental artifacts.
checks = [
    (B_PROTOCOL_PATH, B_PROTOCOL_SHA),
    (B_HEALTHY_PATH, B_HEALTHY_SHA),
    (B_FACTORY_PATH, B_FACTORY_SHA),
    (B_CORRUPT_PATH, B_CORRUPT_SHA),
    (B_AUDIT_PATH, B_MANIFEST[
        "construction_audit_sha256"
    ]),
    (schema_path, B_MANIFEST[
        "tool_contract_sha256"
    ]),
    (B_PUBLIC_GUIDE_PATH, B_PUBLIC_GUIDE_SHA),
    (B_TEACHER_RECEIPT_PATH, B_RUNTIME_SHA),
    (B_PROMPT_PATH, B_PROMPT_SHA),
    (B_LOOP_SPEC_PATH, B_LOOP_SHA),
]

for path, expected in checks:
    assert b_file_sha(path) == expected, (
        f"Frozen artifact mismatch: {path}"
    )

assert b_dataset_fingerprint(
    B_TEACHER_DATA
) == B_TEACHER_DATA_SHA

# 3. Verify the A1 amendment and original source.
A1_PATH = ROOT / "CODE_INTEGRITY_AMENDMENT_A1.json"
A1_SOURCE_PATH = ROOT / "CELL8_SOURCE_A1.py"

assert b_file_sha(A1_PATH) == (
    B_MANIFEST["integrity_amendment_sha256"]
)

a1 = json.loads(A1_PATH.read_text())

assert a1["source_ast_matches"] is True
assert a1["binary_equivalence_established"] is False
assert a1["live_teacher_calls_before_amendment"] == 0

assert b_file_sha(A1_SOURCE_PATH) == (
    a1["original_source_sha256"]
)

source_tree = ast.parse(
    A1_SOURCE_PATH.read_text()
)

source_functions = {
    node.name: node
    for node in source_tree.body
    if isinstance(node, ast.FunctionDef)
}

for name, fn in (
    ("b_run_arm", b_run_arm),
    ("b_live_decision", b_live_decision),
):
    live_source = textwrap.dedent(
        inspect.getsource(fn)
    )

    live_node = ast.parse(
        live_source
    ).body[0]

    expected_node = source_functions[name]

    live_ast = ast.dump(
        live_node, include_attributes=False
    )

    expected_ast = ast.dump(
        expected_node, include_attributes=False
    )

    assert live_ast == expected_ast, (
        f"{name}: original executable source changed"
    )

    assert hashlib.sha256(
        live_ast.encode()
    ).hexdigest() == (
        a1["verified_source_ast_hashes"][name]
    )

# 4. Verify identical original patient weights.
corrupted_record = torch.load(
    B_CORRUPT_PATH,
    map_location="cpu",
    weights_only=True,
)

assert all(
    torch.equal(
        B_LIVE_BACKEND.base_state[k],
        corrupted_record["model_state"][k]
    )
    for k in B_LIVE_BACKEND.base_state
)

assert B_LIVE_BACKEND.model.training is False
assert B_QWEN_MODEL.training is False

assert B_MODEL_REVISION == (
    B_PROTOCOL["teacher"]["revision"]
)

assert (
    B_INITIAL_MESSAGES
    == json.loads(
        B_PROMPT_PATH.read_text()
    )["initial_messages"]
)

# 5. Check session cutoff before control invocation.
now = datetime.now(
    ZoneInfo("America/Denver")
)

cutoff = datetime.fromisoformat(
    B_PROTOCOL["session"]["hard_cutoff"]
)

assert now < cutoff, (
    "Registered session launch cutoff passed. "
    "Do not launch the control."
)

# 6. Ensure this control has never been invoked.
B_REPAIR_ONLY_LIVE_DIR = (
    ROOT / "LIVE_REPAIR_ONLY_V1"
)

B_REPAIR_ONLY_INVOCATION_PATH = (
    ROOT / "LIVE_REPAIR_ONLY_INVOCATION_V1.json"
)

assert not B_REPAIR_ONLY_LIVE_DIR.exists()
assert not B_REPAIR_ONLY_INVOCATION_PATH.exists()

# 7. Freeze control invocation before inference.
control_invocation = {
    "experiment": EXPERIMENT_ID,
    "arm": "repair_only",
    "run": 1,
    "patient_sha256": B_CORRUPT_SHA,
    "teacher_data_sha256": B_TEACHER_DATA_SHA,
    "model_id": B_MODEL_ID,
    "model_revision": B_MODEL_REVISION,
    "runtime_sha256": B_RUNTIME_SHA,
    "prompt_sha256": B_PROMPT_SHA,
    "loop_sha256": B_LOOP_SHA,
    "tool_contract_sha256": B_MANIFEST[
        "tool_contract_sha256"
    ],
    "integrity_amendment_sha256": (
        B_MANIFEST["integrity_amendment_sha256"]
    ),
    "first_arm_summary_sha256": first_summary_sha,
    "maximum_diagnostic_calls": 12,
    "maximum_patch_trials": 2,
    "maximum_total_actions": 20,
    "generation_seed_role": "repair_only_generation",
    "prior_diagnosis_not_in_prompt": True,
    "private_oracle_not_in_prompt": True,
    "prompt_revisions_allowed": False,
    "automatic_retries_allowed": False,
    "sealed_evaluation": "UNOPENED",
}

B_REPAIR_ONLY_INVOCATION_SHA = b_freeze_record(
    B_REPAIR_ONLY_INVOCATION_PATH,
    control_invocation,
)

B_MANIFEST["repair_only_status"] = "IN_PROGRESS"

print("First arm: COMPLETED, EVIDENCE VERIFIED")
print("First-arm summary SHA:", first_summary_sha)
print("C06 checkpoint: VERIFIED")
print("Teacher dataset: VERIFIED")
print("Model revision: VERIFIED")
print("Frozen repair-only prompt: VERIFIED")
print("Executor source: VERIFIED")
print("A1 integrity qualification: RETAINED")
print("Sealed evaluation: UNOPENED")

print("\nControl invocation SHA-256:")
print(B_REPAIR_ONLY_INVOCATION_SHA)

print("\n🌒 Beginning autonomous repair-only trial...\n")

# 8. Execute the original runner exactly once.
try:
    B_REPAIR_ONLY_RESULT = b_run_arm(
        arm="repair_only",
        backend=B_LIVE_BACKEND,
        decide=b_live_decision,
        output_dir=B_REPAIR_ONLY_LIVE_DIR,
    )

except Exception as exc:
    B_MANIFEST["repair_only_status"] = (
        "INTERRUPTED_NO_RETRY"
    )

    crash_path = (
        ROOT / "LIVE_REPAIR_ONLY_CRASH.json"
    )

    crash_record = {
        "experiment": EXPERIMENT_ID,
        "arm": "repair_only",
        "status": "INTERRUPTED_NO_RETRY",
        "exception_type": type(exc).__name__,
        "exception_message": str(exc)[:2000],
        "traceback": traceback.format_exc(limit=8),
        "invocation_sha256": B_REPAIR_ONLY_INVOCATION_SHA,
        "sealed_evaluation": "UNOPENED",
    }

    if not crash_path.exists():
        crash_path.write_text(
            json.dumps(
                crash_record,
                indent=2,
                allow_nan=False,
            )
        )

    print("\n❌ CONTROL INTERRUPTED")
    print("Do not rerun Cell 10.")
    print("Preserve all existing evidence.")
    raise

# 9. Verify completed control evidence.
journal = (
    B_REPAIR_ONLY_LIVE_DIR / "events.jsonl"
)

summary_path = (
    B_REPAIR_ONLY_LIVE_DIR / "summary.json"
)

assert journal.exists()
assert summary_path.exists()

event_count = b_verify_journal(journal)

saved = json.loads(summary_path.read_text())

assert saved == B_REPAIR_ONLY_RESULT

assert 0 <= saved["diagnostic_calls"] <= 12
assert 0 <= saved["patch_trials"] <= 2
assert 0 <= saved["total_actions"] <= 20

if saved["status"] == "COMPLETED":
    assert saved["final_patch_committed"]

B_MANIFEST["repair_only_status"] = saved["status"]
B_MANIFEST["repair_only_diagnostic_calls"] = (
    saved["diagnostic_calls"]
)
B_MANIFEST["repair_only_patch_trials"] = (
    saved["patch_trials"]
)

print("\nREPAIR-ONLY CONTROL RESULT")
print("=" * 62)
print("Status:", saved["status"])
print(
    "Diagnostic calls:",
    saved["diagnostic_calls"], "/ 12"
)
print(
    "Patch trials:",
    saved["patch_trials"], "/ 2"
)
print(
    "Total actions:",
    saved["total_actions"], "/ 20"
)
print(
    "Final patch committed:",
    saved["final_patch_committed"]
)

if saved["final_patch_committed"]:
    print(
        "Final scalar edits:",
        len(saved["final_patch"])
    )

print("Journal events verified:", event_count)
print("Sealed evaluation accessed: NO")
print("A1 integrity qualification: RETAINED")

print("\n✅ CELL 10: CONTROL TRIAL RECORDED")


🌒 EX0004B — REPAIR-ONLY CONTROL
First arm: COMPLETED, EVIDENCE VERIFIED
First-arm summary SHA: 7dc635c9cf0f1faf7023ff04c3db4fc94ae0999438768e60fefb276dcae97475
C06 checkpoint: VERIFIED
Teacher dataset: VERIFIED
Model revision: VERIFIED
Frozen repair-only prompt: VERIFIED
Executor source: VERIFIED
A1 integrity qualification: RETAINED
Sealed evaluation: UNOPENED

Control invocation SHA-256:
c343ee9dbd14d4a01a2cc78ca53f888096e2bddc487e863864d95504d06bdad5

🌒 Beginning autonomous repair-only trial...


REPAIR-ONLY CONTROL RESULT
Status: COMPLETED
Diagnostic calls: 0 / 12
Patch trials: 2 / 2
Total actions: 3 / 20
Final patch committed: True
Final scalar edits: 2
Journal events verified: 10
Sealed evaluation accessed: NO
A1 integrity qualification: RETAINED

✅ CELL 10: CONTROL TRIAL RECORDED


#Cell 11: Lock both outcomes and preregister the sealed verdict
This cell inspects the completed action histories, permanently fingerprints both final submissions, and specifies exactly how we'll measure functional recovery and mechanistic accuracy.
It does not generate or inspect sealed data.

In [17]:

# EX0004B — Cell 11: Lock outcomes + sealed scoring plan
# No Qwen inference. No new patch trials.
# No sealed data generation or private answer-key scoring.

import json
import hashlib
from pathlib import Path

print("🌒 EX0004B — TWO-ARM EVIDENCE LOCK")
print("=" * 64)

assert B_MANIFEST["teacher_status"] == "DIAGNOSIS_FIRST_COMPLETED"
assert B_MANIFEST["repair_only_status"] == "COMPLETED"

assert b_file_sha(B_PROTOCOL_PATH) == B_PROTOCOL_SHA
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA
assert b_file_sha(B_PROMPT_PATH) == B_PROMPT_SHA
assert b_file_sha(B_LOOP_SPEC_PATH) == B_LOOP_SHA

assert b_file_sha(
    ROOT / "CODE_INTEGRITY_AMENDMENT_A1.json"
) == B_MANIFEST["integrity_amendment_sha256"]

B_ARM_LOCKS = {}
B_FINAL_PATCHES = {}

for arm, directory_name in (
    ("diagnosis_first", "LIVE_DIAGNOSIS_FIRST_V1"),
    ("repair_only", "LIVE_REPAIR_ONLY_V1"),
):
    directory = ROOT / directory_name
    journal = directory / "events.jsonl"
    summary_path = directory / "summary.json"
    transcript_path = directory / "transcript.json"

    assert all(p.exists() for p in (
        journal, summary_path, transcript_path
    ))

    journal_count = b_verify_journal(journal)
    summary = json.loads(summary_path.read_text())
    transcript = json.loads(transcript_path.read_text())

    assert summary["status"] == "COMPLETED"
    assert summary["final_patch_committed"]
    assert summary["patch_trials"] == 2
    assert summary["diagnostic_calls"] == 0
    assert summary["total_actions"] <= 20
    assert transcript["final_patch"] == summary["final_patch"]

    expected_phase = "repair" if arm == "diagnosis_first" else "repair_only"
    assert transcript["phase"] == "complete"

    # Confirm the final patch satisfies the already-frozen schema.
    b_patch(summary["final_patch"])
    B_FINAL_PATCHES[arm] = summary["final_patch"]

    if arm == "diagnosis_first":
        assert summary["diagnosis_committed"]
        assert summary["diagnosis"] is not None
        b_validate_action(summary["diagnosis"], "diagnosis")
    else:
        assert summary["diagnosis"] is None

    events = [
        json.loads(line)
        for line in journal.read_text().splitlines()
    ]

    generated = [
        e["body"]["decision"]
        for e in events
        if e["kind"] == "GENERATION"
    ]
    results = [
        e["body"]["response"]
        for e in events
        if e["kind"] == "TOOL_RESULT"
    ]

    assert len(generated) == len(results)
    assert len(generated) == summary["total_actions"]
    assert len(events) == journal_count

    B_ARM_LOCKS[arm] = {
        "summary_sha256": b_file_sha(summary_path),
        "journal_sha256": b_file_sha(journal),
        "transcript_sha256": b_file_sha(transcript_path),
        "journal_records": journal_count,
        "total_actions": summary["total_actions"],
        "diagnostic_calls": summary["diagnostic_calls"],
        "patch_trials": summary["patch_trials"],
        "diagnosis": summary["diagnosis"],
        "final_patch": summary["final_patch"],
        "status": summary["status"],
    }

    print(f"\nARM: {arm}")
    print("-" * 64)

    for index, (decision, response) in enumerate(
        zip(generated, results), 1
    ):
        try:
            action = b_parse_action(decision["text"])
            tool = action["tool"]
        except (ToolSchemaError, ValueError, KeyError):
            tool = "INVALID_ACTION"

        print(f"Action {index}: {tool}")

        if tool == "patch_trial":
            print("  Proposed edits:", len(action["edits"]))
            print("  Trial metrics:",
                  response.get("metrics", response))

        elif tool == "submit_diagnosis":
            print("  Diagnosis committed: YES")

        elif tool == "submit_patch":
            print("  Final edits:", len(action["edits"]))

        else:
            print("  Response:", response)

    print("Summary SHA-256:",
          B_ARM_LOCKS[arm]["summary_sha256"])
    print("Final patch:", json.dumps(
        summary["final_patch"], sort_keys=True
    ))

print("\nDIAGNOSIS-FIRST COMMITMENT")
print("-" * 64)
print(json.dumps(
    B_ARM_LOCKS["diagnosis_first"]["diagnosis"],
    indent=2, allow_nan=False
))

# Freeze independent scoring definitions BEFORE opening sealed data.
B_SEALED_SPEC = {
    "version": "EX0004B_SEALED_SCORING_V1",
    "experiment": EXPERIMENT_ID,
    "protocol_sha256": B_PROTOCOL_SHA,
    "corrupted_patient_sha256": B_CORRUPT_SHA,
    "arm_evidence": B_ARM_LOCKS,
    "sealed_sampling": {
        "id_count": 1000,
        "shift_count": 1000,
        "counterfactual_pair_count": 256,
        "id_seed": B_SEEDS["sealed_id"],
        "shift_seed": B_SEEDS["sealed_shift"],
        "counterfactual_seed": B_SEEDS[
            "sealed_counterfactual"
        ],
        "counterfactual_regime": "shift",
        "counterfactual_operation": "negate_x5",
    },
    "comparators": [
        "healthy", "corrupted", "zero_only",
        "oracle_A", "oracle_B", "oracle_joint",
        "diagnosis_first", "repair_only",
    ],
    "permuted_patch_controls": {
        "count_per_nonempty_patch": 3,
        "seeds": [
            B_SEEDS["permuted_control_1"],
            B_SEEDS["permuted_control_2"],
            B_SEEDS["permuted_control_3"],
        ],
        "method": (
            "For each nonempty final patch, independently "
            "relocate its deltas to other valid scalar positions "
            "within the same named tensor, without duplicate "
            "indices or changing deltas. If relocation is "
            "impossible, mark that control NOT_APPLICABLE."
        ),
        "role": "DESCRIPTIVE_NEGATIVE_CONTROLS_ONLY",
    },
    "functional_scoring": {
        "gap": "corrupted_shift_mse - healthy_shift_mse",
        "recovery": (
            "(corrupted_shift_mse - repaired_shift_mse)"
            " / gap"
        ),
        "id_excess_fraction": (
            "(repaired_id_mse - healthy_id_mse) / gap"
        ),
        "variance_ratio": (
            "variance(repaired_shift_predictions)"
            " / variance(shift_targets)"
        ),
        "minimum_recovery": B_PROTOCOL[
            "functional_gate"
        ]["minimum_recovery_fraction"],
        "maximum_id_excess_fraction": B_PROTOCOL[
            "functional_gate"
        ]["maximum_id_loss_increase_over_gap"],
        "minimum_variance_ratio": B_PROTOCOL[
            "functional_gate"
        ]["minimum_prediction_variance_ratio"],
        "nonpositive_gap": "INCONCLUSIVE",
        "all_three_thresholds_required": True,
        "no_final_patch": "EVALUATE_CORRUPTED_BASELINE",
    },
    "mechanistic_scoring": {
        "fault_A": (
            "Exact fc1 neuron and input_feature "
            "of planted A weight edge."
        ),
        "fault_B": (
            "Exact fc2 neuron and source_neuron "
            "of planted B weight edge."
        ),
        "isolated_effect_rank": (
            "Compare oracle_A-only and oracle_B-only "
            "lesion states against healthy on sealed SHIFT."
        ),
        "interaction_sign": (
            "Sign of L11-L10-L01+L00 "
            "on sealed SHIFT."
        ),
        "nuisance_restoration": (
            "Compare oracle_joint vs corrupted "
            "counterfactual nuisance sensitivity, "
            "using mean squared output difference "
            "when x5 is negated."
        ),
        "nuisance_direction_threshold": (
            "5 percent relative change; within "
            "threshold is unchanged."
        ),
        "tie_tolerance": 1e-8,
        "mechanistic_pass_requires": [
            "both_fault_regions_correct",
            "isolated_effect_rank_correct",
            "interaction_sign_correct",
            "predictions_committed_before_patch_feedback",
        ],
        "nuisance_direction_reported_separately": True,
        "repair_only_mechanistic_score": "NOT_APPLICABLE",
    },
    "interpretation": {
        "single_patient": True,
        "no_population_level_causal_claim": True,
        "zero_diagnostic_calls_in_both_arms": True,
        "A1_binary_equivalence_unresolved": True,
        "sealed_evaluation_only_once": True,
        "no_post_evaluation_patch_changes": True,
    },
}

B_SEALED_SPEC_PATH = ROOT / "SEALED_SCORING_SPEC_V1.json"
B_SEALED_SPEC_SHA = b_freeze_record(
    B_SEALED_SPEC_PATH, B_SEALED_SPEC
)

B_OUTCOME_LOCK = {
    "experiment": EXPERIMENT_ID,
    "status": "BOTH_FINAL_PATCHES_LOCKED",
    "scoring_spec_sha256": B_SEALED_SPEC_SHA,
    "arm_evidence": B_ARM_LOCKS,
    "sealed_evaluation": "NOT_GENERATED",
}

B_OUTCOME_LOCK_PATH = ROOT / "OUTCOME_LOCK_V1.json"
B_OUTCOME_LOCK_SHA = b_freeze_record(
    B_OUTCOME_LOCK_PATH, B_OUTCOME_LOCK
)

B_MANIFEST["outcome_lock_sha256"] = B_OUTCOME_LOCK_SHA
B_MANIFEST["sealed_scoring_sha256"] = B_SEALED_SPEC_SHA
B_MANIFEST["sealed_status"] = "SCORING_LOCKED_UNOPENED"

print("\n" + "=" * 64)
print("Both arm summaries verified: YES")
print("Both final patches frozen: YES")
print("All 4 patch-trial responses preserved: YES")
print("Independent sealed scoring rubric: FROZEN")
print("Private answer key scored: NO")
print("Sealed data generated: NO")
print("Further teacher inference authorized: NO")

print("\nSEALED SCORING SHA-256:")
print(B_SEALED_SPEC_SHA)

print("\nOUTCOME LOCK SHA-256:")
print(B_OUTCOME_LOCK_SHA)

print("\n✅ CELL 11: EVIDENCE AND SCORING LOCKED")


🌒 EX0004B — TWO-ARM EVIDENCE LOCK

ARM: diagnosis_first
----------------------------------------------------------------
Action 1: submit_diagnosis
  Diagnosis committed: YES
Action 2: patch_trial
  Proposed edits: 2
  Trial metrics: {'id_mse': 0.0578821, 'nuisance_mse': 0.1421946, 'shift_mse': 0.1136471, 'shift_variance': 0.158675}
Action 3: patch_trial
  Proposed edits: 2
  Trial metrics: {'id_mse': 0.0595015, 'nuisance_mse': 0.1423152, 'shift_mse': 0.1169746, 'shift_variance': 0.1550138}
Action 4: submit_patch
  Final edits: 0
Summary SHA-256: 7dc635c9cf0f1faf7023ff04c3db4fc94ae0999438768e60fefb276dcae97475
Final patch: []

ARM: repair_only
----------------------------------------------------------------
Action 1: patch_trial
  Proposed edits: 1
  Trial metrics: {'id_mse': 0.0572642, 'nuisance_mse': 0.166891, 'shift_mse': 0.1213759, 'shift_variance': 0.1595498}
Action 2: patch_trial
  Proposed edits: 1
  Trial metrics: {'id_mse': 0.0621948, 'nuisance_mse': 0.143735, 'shift_mse': 0.1

#Cell 12: Open the sealed evaluation, exactly once
This cell will evaluate both immutable final patches on the same previously unseen data.

It will also score the frozen causal predictions and compare the results with the healthy patient, corrupted patient, zero-only repair, oracle restorations, and three permuted-patch controls.

A persistent start receipt prevents accidentally repeating this evaluation. If the run is interrupted, preserve the evidence rather than rerunning the cell.

In [18]:

# EX0004B — Cell 12: One-shot sealed evaluation
# Both Qwen outcomes and scoring rules are already locked.
# DO NOT rerun after the sealed-start receipt exists.

import json
import math
import hashlib
import random
import itertools
import os
import torch
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

print("🌒 EX0004B — THE SEALED VERDICT")
print("=" * 66)

# 1. Verify the locked evidence.
assert B_MANIFEST["sealed_status"] == "SCORING_LOCKED_UNOPENED"

assert b_file_sha(B_PROTOCOL_PATH) == B_PROTOCOL_SHA
assert b_file_sha(B_HEALTHY_PATH) == B_HEALTHY_SHA
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA
assert b_file_sha(B_FACTORY_PATH) == B_FACTORY_SHA
assert b_file_sha(B_AUDIT_PATH) == (
    B_MANIFEST["construction_audit_sha256"]
)
assert b_file_sha(B_SEALED_SPEC_PATH) == B_SEALED_SPEC_SHA
assert b_file_sha(B_OUTCOME_LOCK_PATH) == B_OUTCOME_LOCK_SHA

spec = json.loads(B_SEALED_SPEC_PATH.read_text())
lock = json.loads(B_OUTCOME_LOCK_PATH.read_text())

assert lock["status"] == "BOTH_FINAL_PATCHES_LOCKED"
assert lock["scoring_spec_sha256"] == B_SEALED_SPEC_SHA
assert spec["protocol_sha256"] == B_PROTOCOL_SHA
assert spec["corrupted_patient_sha256"] == B_CORRUPT_SHA
assert spec["arm_evidence"] == lock["arm_evidence"]

assert spec["sealed_sampling"] == {
    "id_count": 1000,
    "shift_count": 1000,
    "counterfactual_pair_count": 256,
    "id_seed": B_SEEDS["sealed_id"],
    "shift_seed": B_SEEDS["sealed_shift"],
    "counterfactual_seed": B_SEEDS["sealed_counterfactual"],
    "counterfactual_regime": "shift",
    "counterfactual_operation": "negate_x5",
}

# 2. Reverify the actual frozen submissions.
arms = {}

for arm, folder in (
    ("diagnosis_first", "LIVE_DIAGNOSIS_FIRST_V1"),
    ("repair_only", "LIVE_REPAIR_ONLY_V1"),
):
    directory = ROOT / folder
    evidence = spec["arm_evidence"][arm]

    for filename, expected in (
        ("summary.json", evidence["summary_sha256"]),
        ("events.jsonl", evidence["journal_sha256"]),
        ("transcript.json", evidence["transcript_sha256"]),
    ):
        assert b_file_sha(directory / filename) == expected

    b_verify_journal(directory / "events.jsonl")

    summary = json.loads(
        (directory / "summary.json").read_text()
    )

    assert summary["status"] == "COMPLETED"
    assert summary["final_patch_committed"]
    assert summary["final_patch"] == evidence["final_patch"]
    assert summary["diagnosis"] == evidence["diagnosis"]

    b_patch(summary["final_patch"])
    arms[arm] = summary

assert arms["diagnosis_first"]["diagnosis_committed"]
assert arms["diagnosis_first"]["diagnostic_calls"] == 0
assert arms["repair_only"]["diagnostic_calls"] == 0

# 3. Verify the private lesion specification.
factory = json.loads(B_FACTORY_PATH.read_text())
audit = json.loads(B_AUDIT_PATH.read_text())

selected_id = audit["selected_candidate"]

assert selected_id == "C06"
assert audit["corrupted_checkpoint_sha256"] == B_CORRUPT_SHA

candidate = next(
    c for c in factory["candidates"]
    if c["id"] == selected_id
)

assert len(candidate["edits"]) == 4
assert len([e for e in candidate["edits"]
            if e["group"] == "A"]) == 2
assert len([e for e in candidate["edits"]
            if e["group"] == "B"]) == 2

healthy_record = torch.load(
    B_HEALTHY_PATH, map_location="cpu",
    weights_only=True
)

corrupt_record = torch.load(
    B_CORRUPT_PATH, map_location="cpu",
    weights_only=True
)

healthy = healthy_record["model_state"]
corrupt = corrupt_record["model_state"]

assert set(healthy) == set(corrupt)

# 4. Prepare all comparison states before opening data.
def b12_copy_state(state):
    return {k: v.detach().cpu().clone()
            for k, v in state.items()}

def b12_restore(groups):
    state = b12_copy_state(corrupt)

    for edit in candidate["edits"]:
        if edit["group"] in groups:
            name = edit["tensor"]
            idx = tuple(edit["index"])
            state[name][idx] = healthy[name][idx]

    return state

def b12_apply_patch(edits):
    b_patch(edits)
    state = b12_copy_state(corrupt)

    for edit in edits:
        state[edit["tensor"]][
            tuple(edit["index"])
        ] += float(edit["delta"])

    return state

zero_state = b12_copy_state(corrupt)

for edit in candidate["edits"]:
    zero_state[edit["tensor"]][
        tuple(edit["index"])
    ] = 0.0

states = {
    "healthy": b12_copy_state(healthy),
    "corrupted": b12_copy_state(corrupt),
    "zero_only": zero_state,
    # Restore A, leaving B present.
    "oracle_A": b12_restore({"A"}),
    # Restore B, leaving A present.
    "oracle_B": b12_restore({"B"}),
    "oracle_joint": b12_restore({"A", "B"}),
    "diagnosis_first": b12_apply_patch(
        arms["diagnosis_first"]["final_patch"]
    ),
    "repair_only": b12_apply_patch(
        arms["repair_only"]["final_patch"]
    ),
}

assert all(
    torch.equal(states["oracle_joint"][k], healthy[k])
    for k in healthy
)

assert all(
    torch.equal(states["diagnosis_first"][k], corrupt[k])
    for k in corrupt
), "The empty final patch must leave C06 unchanged."

# 5. Construct prospectively specified permuted controls.
def b12_permute_edits(edits, seed):
    rng = random.Random(int(seed))

    original_positions = {}
    for edit in edits:
        original_positions.setdefault(
            edit["tensor"], set()
        ).add(tuple(edit["index"]))

    occupied = {}
    relocated = []

    for edit in edits:
        name = edit["tensor"]
        shape = B_TENSOR_SHAPES[name]
        occupied.setdefault(name, set())

        possibilities = [
            tuple(idx)
            for idx in itertools.product(
                *(range(dim) for dim in shape)
            )
            if tuple(idx) not in original_positions[name]
            and tuple(idx) not in occupied[name]
        ]

        if not possibilities:
            return None

        new_index = possibilities[
            rng.randrange(len(possibilities))
        ]

        occupied[name].add(new_index)

        relocated.append({
            "tensor": name,
            "index": list(new_index),
            "delta": edit["delta"],
        })

    b_patch(relocated)
    return relocated

permutation_records = []

for arm in ("diagnosis_first", "repair_only"):
    edits = arms[arm]["final_patch"]

    if not edits:
        continue

    for number, seed in enumerate(
        spec["permuted_patch_controls"]["seeds"], 1
    ):
        permuted = b12_permute_edits(edits, seed)

        if permuted is None:
            permutation_records.append({
                "arm": arm,
                "control": number,
                "status": "NOT_APPLICABLE",
            })
            continue

        key = f"{arm}_permuted_{number}"
        states[key] = b12_apply_patch(permuted)

        permutation_records.append({
            "arm": arm,
            "control": number,
            "status": "EVALUABLE",
            "model_key": key,
            "seed": seed,
            "edits": permuted,
        })

# 6. Commit the sealed-start receipt BEFORE data generation.
start_path = ROOT / "SEALED_EVALUATION_STARTED_V1.json"
report_path = ROOT / "SEALED_EVALUATION_REPORT_V1.json"

assert not start_path.exists()
assert not report_path.exists()

now = datetime.now(ZoneInfo("America/Denver"))
cutoff = datetime.fromisoformat(
    B_PROTOCOL["session"]["hard_cutoff"]
)

assert now < cutoff, (
    "Registered midnight cutoff passed. "
    "Do not start sealed evaluation."
)

start_record = {
    "experiment": EXPERIMENT_ID,
    "status": "SEALED_EVALUATION_STARTED",
    "scoring_sha256": B_SEALED_SPEC_SHA,
    "outcome_lock_sha256": B_OUTCOME_LOCK_SHA,
    "patient_sha256": B_CORRUPT_SHA,
    "candidate_id": selected_id,
    "model_keys": sorted(states),
    "sealed_data_generated_before_start": False,
    "reexecution_permitted": False,
}

with start_path.open("x") as stream:
    json.dump(start_record, stream, indent=2)
    stream.flush()
    os.fsync(stream.fileno())

start_sha = b_file_sha(start_path)
B_MANIFEST["sealed_status"] = "EVALUATION_STARTED"

print("Sealed-start SHA-256:", start_sha)
print("Final patches: LOCKED")
print("Scoring rules: LOCKED")
print("Model comparisons:", len(states))
print("\nOpening sealed data once...\n")

# 7. Generate held-out data using frozen independent seeds.
sealed = spec["sealed_sampling"]

id_x, id_y = b_sample_world(
    sealed["id_count"],
    sealed["id_seed"], "id", "cpu"
)

shift_x, shift_y = b_sample_world(
    sealed["shift_count"],
    sealed["shift_seed"], "shift", "cpu"
)

cf_x, cf_y = b_sample_world(
    sealed["counterfactual_pair_count"],
    sealed["counterfactual_seed"], "shift", "cpu"
)

cf_changed = cf_x.clone()
cf_changed[:, 5] *= -1

assert torch.equal(
    b_ground_truth(cf_x),
    b_ground_truth(cf_changed)
)

def b12_dataset_hash(tensors):
    h = hashlib.sha256()

    for tensor in tensors:
        t = tensor.detach().cpu().contiguous()
        h.update(str(tuple(t.shape)).encode())
        h.update(str(t.dtype).encode())
        h.update(t.numpy().tobytes())

    return h.hexdigest()

sealed_data_sha = b12_dataset_hash([
    id_x, id_y, shift_x, shift_y, cf_x, cf_y
])

# 8. Evaluate every frozen state on exactly the same data.
def b12_evaluate_state(state):
    model = BPatientNet().cpu()
    model.load_state_dict(state, strict=True)
    model.eval()
    model.requires_grad_(False)

    with torch.inference_mode():
        pred_id = model(id_x)
        pred_shift = model(shift_x)
        pred_cf = model(cf_x)
        pred_changed = model(cf_changed)

        result = {
            "id_mse": torch.mean(
                (pred_id-id_y)**2
            ).item(),
            "shift_mse": torch.mean(
                (pred_shift-shift_y)**2
            ).item(),
            "shift_prediction_variance": torch.var(
                pred_shift, unbiased=False
            ).item(),
            "nuisance_mse": torch.mean(
                (pred_cf-pred_changed)**2
            ).item(),
        }

    assert all(
        math.isfinite(x) for x in result.values()
    )
    return result

metrics = {
    name: b12_evaluate_state(state)
    for name, state in states.items()
}

target_variance = torch.var(
    shift_y, unbiased=False
).item()

L00 = metrics["healthy"]["shift_mse"]
L10 = metrics["oracle_B"]["shift_mse"]
L01 = metrics["oracle_A"]["shift_mse"]
L11 = metrics["corrupted"]["shift_mse"]

gap = L11 - L00
interaction = L11 - L10 - L01 + L00

tie_tol = spec["mechanistic_scoring"]["tie_tolerance"]

# 9. Evaluate functional gate exactly as frozen.
functional_gate = B_PROTOCOL["functional_gate"]
functional_scores = {}

for arm in ("diagnosis_first", "repair_only"):
    measured = metrics[arm]

    if gap <= 0:
        functional_scores[arm] = {
            "verdict": "INCONCLUSIVE",
            "reason": "Nonpositive sealed corruption gap",
        }
        continue

    recovery = (
        L11 - measured["shift_mse"]
    ) / gap

    id_excess = (
        measured["id_mse"]
        - metrics["healthy"]["id_mse"]
    ) / gap

    variance_ratio = (
        measured["shift_prediction_variance"]
        / target_variance
    )

    checks = {
        "recovery": (
            recovery >=
            functional_gate["minimum_recovery_fraction"]
        ),
        "id_guardrail": (
            id_excess <=
            functional_gate[
                "maximum_id_loss_increase_over_gap"
            ]
        ),
        "variance_guardrail": (
            variance_ratio >=
            functional_gate[
                "minimum_prediction_variance_ratio"
            ]
        ),
    }

    functional_scores[arm] = {
        "recovery_fraction": recovery,
        "id_excess_fraction": id_excess,
        "variance_ratio": variance_ratio,
        "checks": checks,
        "verdict": (
            "PASS" if all(checks.values()) else "FAIL"
        ),
    }

# 10. Score the committed causal predictions.
diagnosis = arms["diagnosis_first"]["diagnosis"]

A_effect = L10 - L00
B_effect = L01 - L00

if abs(A_effect-B_effect) <= tie_tol:
    expected_rank = "TIE"
elif A_effect > B_effect:
    expected_rank = "A_gt_B"
else:
    expected_rank = "B_gt_A"

if abs(interaction) <= tie_tol:
    expected_interaction = "TIE"
elif interaction > 0:
    expected_interaction = "positive"
else:
    expected_interaction = "negative"

corrupt_nuisance = metrics["corrupted"]["nuisance_mse"]
restored_nuisance = metrics["oracle_joint"]["nuisance_mse"]

relative_change = (
    (restored_nuisance-corrupt_nuisance)
    / max(corrupt_nuisance, 1e-12)
)

direction_threshold = 0.05

if relative_change < -direction_threshold:
    expected_nuisance = "decrease"
elif relative_change > direction_threshold:
    expected_nuisance = "increase"
else:
    expected_nuisance = "unchanged"

true_A = {
    "layer": "fc1",
    "neuron": candidate["p"],
    "input_feature": 5,
}

true_B = {
    "layer": "fc2",
    "neuron": candidate["q"],
    "source_neuron": candidate["p"],
}

# Check commitment order from immutable journal.
journal_path = (
    ROOT / "LIVE_DIAGNOSIS_FIRST_V1/events.jsonl"
)

action_names = []

for line in journal_path.read_text().splitlines():
    event = json.loads(line)
    if event["kind"] != "GENERATION":
        continue
    action = b_parse_action(
        event["body"]["decision"]["text"]
    )
    action_names.append(action["tool"])

commit_before_feedback = (
    "submit_diagnosis" in action_names
    and "patch_trial" in action_names
    and action_names.index("submit_diagnosis")
        < action_names.index("patch_trial")
)

mechanistic_checks = {
    "both_fault_regions_correct": (
        diagnosis["fault_A"] == true_A
        and diagnosis["fault_B"] == true_B
    ),
    "isolated_effect_rank_correct": (
        diagnosis["predictions"]["isolated_effect_rank"]
        == expected_rank
    ),
    "interaction_sign_correct": (
        diagnosis["predictions"]["interaction_sign"]
        == expected_interaction
    ),
    "predictions_committed_before_patch_feedback":
        commit_before_feedback,
}

example_copied = (
    diagnosis == B_DIAGNOSIS_EXAMPLE
)

mechanistic_score = {
    "true_fault_A": true_A,
    "true_fault_B": true_B,
    "sealed_isolated_effect_A": A_effect,
    "sealed_isolated_effect_B": B_effect,
    "expected_rank": expected_rank,
    "sealed_interaction": interaction,
    "expected_interaction_sign": expected_interaction,
    "nuisance_relative_change": relative_change,
    "expected_nuisance_direction": expected_nuisance,
    "nuisance_prediction_correct": (
        diagnosis["predictions"]["nuisance_restoration"]
        == expected_nuisance
    ),
    "checks": mechanistic_checks,
    "formal_verdict": (
        "PASS"
        if all(mechanistic_checks.values())
        else "FAIL"
    ),
    "submitted_example_verbatim": example_copied,
    "evidence_qualification": (
        "The submitted diagnosis exactly reproduced "
        "the public syntax example; no diagnostic "
        "observations were requested."
        if example_copied else
        "Interpret together with the recorded "
        "diagnostic-observation history."
    ),
}

# 11. Record the final sealed report.
report = {
    "experiment": EXPERIMENT_ID,
    "status": "SEALED_EVALUATION_COMPLETED",
    "outcome_lock_sha256": B_OUTCOME_LOCK_SHA,
    "scoring_spec_sha256": B_SEALED_SPEC_SHA,
    "sealed_start_sha256": start_sha,
    "sealed_data_sha256": sealed_data_sha,
    "selected_patient": selected_id,
    "target_variance": target_variance,
    "corruption_gap": gap,
    "metrics": metrics,
    "functional_scores": functional_scores,
    "mechanistic_score": mechanistic_score,
    "permuted_controls": permutation_records,
    "diagnostic_calls": {
        "diagnosis_first": 0,
        "repair_only": 0,
    },
    "integrity_qualification": (
        "A1 source-level match; binary fingerprint "
        "discrepancy unresolved."
    ),
    "interpretation_limits": [
        "Single synthetic patient and one run per arm.",
        "Neither arm used diagnostic observations.",
        "Diagnosis-first copied the prompt example."
        if example_copied else
        "Mechanistic statements require independent checking.",
        "No population-level causal inference.",
        "Only this sealed evaluation was performed.",
    ],
}

assert not report_path.exists()

report_sha = b_freeze_record(report_path, report)

B_MANIFEST["sealed_status"] = "EVALUATION_COMPLETED"
B_MANIFEST["sealed_report_sha256"] = report_sha

# 12. Print the independent verdict.
print("SEALED FUNCTIONAL RESULTS")
print("=" * 66)
print(f"Healthy SHIFT MSE:   {L00:.6f}")
print(f"Corrupted SHIFT MSE: {L11:.6f}")
print(f"Corruption gap:      {gap:.6f}")
print()

for arm in ("diagnosis_first", "repair_only"):
    score = functional_scores[arm]

    print(arm.upper())
    print(
        "  Sealed SHIFT MSE:",
        f"{metrics[arm]['shift_mse']:.6f}"
    )
    print(
        "  Recovery:",
        f"{score.get('recovery_fraction', float('nan')):.2%}"
    )
    print("  Functional verdict:", score["verdict"])
    print()

print("MECHANISTIC RESULTS")
print("=" * 66)
print("True fault A:", true_A)
print("True fault B:", true_B)
print("Expected effect ranking:", expected_rank)
print("Expected interaction:", expected_interaction)
print("Expected nuisance direction:", expected_nuisance)

for name, passed in mechanistic_checks.items():
    print(name, "PASS" if passed else "FAIL")

print(
    "Formal mechanistic verdict:",
    mechanistic_score["formal_verdict"]
)
print(
    "Copied prompt example:",
    "YES" if example_copied else "NO"
)

print("\nSEALED RECORD")
print("Dataset SHA-256:", sealed_data_sha)
print("Report SHA-256:", report_sha)
print("Additional teacher calls: 0")
print("Sealed evaluation executions: 1")
print("Final submissions changed: NO")

print("\n✅ CELL 12: SEALED VERDICT FROZEN")


🌒 EX0004B — THE SEALED VERDICT
Sealed-start SHA-256: 7bbb210d529d8cff5499bb7eb2eb2e9fc62386e47c6dead0a48a36b2279ad467
Final patches: LOCKED
Scoring rules: LOCKED
Model comparisons: 11

Opening sealed data once...

SEALED FUNCTIONAL RESULTS
Healthy SHIFT MSE:   0.000209
Corrupted SHIFT MSE: 0.117318
Corruption gap:      0.117109

DIAGNOSIS_FIRST
  Sealed SHIFT MSE: 0.117318
  Recovery: 0.00%
  Functional verdict: FAIL

REPAIR_ONLY
  Sealed SHIFT MSE: 0.127005
  Recovery: -8.27%
  Functional verdict: FAIL

MECHANISTIC RESULTS
True fault A: {'layer': 'fc1', 'neuron': 4, 'input_feature': 5}
True fault B: {'layer': 'fc2', 'neuron': 8, 'source_neuron': 4}
Expected effect ranking: A_gt_B
Expected interaction: negative
Expected nuisance direction: decrease
both_fault_regions_correct FAIL
isolated_effect_rank_correct PASS
interaction_sign_correct FAIL
predictions_committed_before_patch_feedback PASS
Formal mechanistic verdict: FAIL
Copied prompt example: YES

SEALED RECORD
Dataset SHA-256: 0f77

#Cell 13: Final research report and evidence archive
This final cell reads the already-frozen evidence, creates a Markdown research report, verifies the source files, and packages everything into a ZIP.

It performs no new inference, patch testing, or sealed-data evaluation.

In [19]:

# EX0004B — Cell 13: Final scientific report and archive
# Archival only. NO new Qwen calls or data evaluation.
# Preserve all original evidence unchanged.

import json
import hashlib
import zipfile
from pathlib import Path

print("🌒 EX0004B — FINAL RESEARCH ARCHIVE")
print("=" * 66)

# 1. Verify the completed experiment.
assert B_MANIFEST["sealed_status"] == "EVALUATION_COMPLETED"
assert B_MANIFEST["teacher_status"] == "DIAGNOSIS_FIRST_COMPLETED"
assert B_MANIFEST["repair_only_status"] == "COMPLETED"

sealed_path = ROOT / "SEALED_EVALUATION_REPORT_V1.json"
lock_path = ROOT / "OUTCOME_LOCK_V1.json"
scoring_path = ROOT / "SEALED_SCORING_SPEC_V1.json"

assert b_file_sha(sealed_path) == (
    B_MANIFEST["sealed_report_sha256"]
)
assert b_file_sha(lock_path) == B_OUTCOME_LOCK_SHA
assert b_file_sha(scoring_path) == B_SEALED_SPEC_SHA
assert b_file_sha(B_CORRUPT_PATH) == B_CORRUPT_SHA

result = json.loads(sealed_path.read_text())
lock = json.loads(lock_path.read_text())
scoring = json.loads(scoring_path.read_text())

assert result["status"] == "SEALED_EVALUATION_COMPLETED"
assert result["outcome_lock_sha256"] == B_OUTCOME_LOCK_SHA
assert result["scoring_spec_sha256"] == B_SEALED_SPEC_SHA
assert lock["arm_evidence"] == scoring["arm_evidence"]

for arm, folder in (
    ("diagnosis_first", "LIVE_DIAGNOSIS_FIRST_V1"),
    ("repair_only", "LIVE_REPAIR_ONLY_V1"),
):
    directory = ROOT / folder
    evidence = lock["arm_evidence"][arm]

    for filename, key in (
        ("summary.json", "summary_sha256"),
        ("events.jsonl", "journal_sha256"),
        ("transcript.json", "transcript_sha256"),
    ):
        assert b_file_sha(directory / filename) == evidence[key]

    b_verify_journal(directory / "events.jsonl")

# 2. Extract already-frozen scientific results.
metrics = result["metrics"]
functional = result["functional_scores"]
mechanistic = result["mechanistic_score"]
arms = lock["arm_evidence"]

assert functional["diagnosis_first"]["verdict"] == "FAIL"
assert functional["repair_only"]["verdict"] == "FAIL"
assert mechanistic["formal_verdict"] == "FAIL"
assert mechanistic["submitted_example_verbatim"]

assert arms["diagnosis_first"]["diagnostic_calls"] == 0
assert arms["repair_only"]["diagnostic_calls"] == 0
assert arms["diagnosis_first"]["final_patch"] == []
assert len(arms["repair_only"]["final_patch"]) == 2

def fnum(x, decimals=6):
    return f"{x:.{decimals}f}"

def fpct(x):
    return f"{100*x:+.2f}%"

def mdjson(value):
    return "`" + json.dumps(
        value, sort_keys=True, allow_nan=False
    ) + "`"

# 3. Build the research report from frozen values.
lines = [
    "# ALGN_EX0004B: Land's Quiet Scalpel II",
    "",
    "**Final research report | November 12, 2026**",
    "",
    "**Disposition:** Completed negative-result case study.",
    "",
    "## Executive finding",
    "",
    "Qwen3-8B completed both experimental arms but "
    "did not successfully repair the corrupted network. "
    "The diagnosis-first arm submitted the public syntax "
    "example as its diagnosis and ultimately abstained "
    "from making parameter edits. The repair-only arm "
    "submitted two edits that worsened held-out SHIFT MSE.",
    "",
    "Neither arm requested a diagnostic observation.",
    "",
    "These results do not establish general model "
    "incapability or a population-level causal effect "
    "of the two instruction conditions.",
    "",
    "## Experiment",
    "",
    f"- Experiment: `{EXPERIMENT_ID}`",
    "- Model: Qwen/Qwen3-8B",
    f"- Pinned revision: `{B_MODEL_REVISION}`",
    "- Precision: BF16",
    "- Patient architecture: 6 → 12 → 12 → 1",
    "- Patient parameters: 253",
    f"- Selected patient: `{result['selected_patient']}`",
    "- Planted fault groups: 2",
    "- Altered patient scalars: 4",
    "- Maximum diagnostic calls per arm: 12",
    "- Maximum patch trials per arm: 2",
    "- Maximum final edits: 6",
    "- Trials per arm: 1",
    "- Teacher-development and sealed datasets: separate",
    "- Sealed evaluation: one execution after both patches locked",
    "",
    "## Held-out functional results",
    "",
    f"Healthy SHIFT MSE: **{fnum(metrics['healthy']['shift_mse'])}**.",
    "",
    f"Corrupted SHIFT MSE: **{fnum(metrics['corrupted']['shift_mse'])}**.",
    "",
    f"Corruption gap: **{fnum(result['corruption_gap'])}**.",
    "",
    "| Arm | SHIFT MSE | Recovery | Verdict |",
    "|---|---:|---:|---|",
]

for arm in ("diagnosis_first", "repair_only"):
    score = functional[arm]
    lines.append(
        f"| {arm} | "
        f"{fnum(metrics[arm]['shift_mse'])} | "
        f"{fpct(score['recovery_fraction'])} | "
        f"{score['verdict']} |"
    )

lines += [
    "",
    "Functional PASS required all three preregistered "
    "checks: at least 80% recovery, the ID-error guardrail, "
    "and the prediction-variance guardrail.",
    "",
    "### Full sealed comparison table",
    "",
    "| Model state | ID MSE | SHIFT MSE | Nuisance MSE |",
    "|---|---:|---:|---:|",
]

main_states = [
    "healthy", "corrupted", "zero_only",
    "oracle_A", "oracle_B", "oracle_joint",
    "diagnosis_first", "repair_only",
]

for name in main_states:
    m = metrics[name]
    lines.append(
        f"| {name} | "
        f"{fnum(m['id_mse'])} | "
        f"{fnum(m['shift_mse'])} | "
        f"{fnum(m['nuisance_mse'])} |"
    )

lines += [
    "",
    "## Behavioral observations",
    "",
    "| Measure | Diagnosis-first | Repair-only |",
    "|---|---:|---:|",
    "| Diagnostic calls | 0 | 0 |",
    "| Patch trials | 2 | 2 |",
    "| Total actions | 4 | 3 |",
    "| Final scalar edits | 0 | 2 |",
    "",
    "**Diagnosis-first final patch:** " +
    mdjson(arms["diagnosis_first"]["final_patch"]),
    "",
    "**Repair-only final patch:** " +
    mdjson(arms["repair_only"]["final_patch"]),
    "",
    "The diagnosis-first arm's empty patch leaves the "
    "corrupted patient unchanged. The repair-only "
    "edits yielded negative sealed recovery.",
    "",
    "## Mechanistic diagnosis",
    "",
    "**True fault A:** " + mdjson(
        mechanistic["true_fault_A"]
    ),
    "",
    "**True fault B:** " + mdjson(
        mechanistic["true_fault_B"]
    ),
    "",
    "**Committed diagnosis:** " + mdjson(
        arms["diagnosis_first"]["diagnosis"]
    ),
    "",
    "The submitted diagnosis exactly reproduced the "
    "public example, including its placeholder explanation. "
    "This is syntactically valid but does not demonstrate "
    "independent causal identification.",
    "",
    "| Mechanistic check | Result |",
    "|---|---|",
]

for key, passed in mechanistic["checks"].items():
    lines.append(
        f"| {key} | {'PASS' if passed else 'FAIL'} |"
    )

lines += [
    "",
    "**Formal mechanistic verdict: FAIL.**",
    "",
    "The isolated-effect ranking matched the sealed truth, "
    "but this was also the prediction present in the "
    "copied example. It should not be treated as evidence "
    "of an independently discovered causal relationship.",
    "",
    "**Expected isolated-effect ranking:** " +
    mechanistic["expected_rank"],
    "",
    "**Expected interaction sign:** " +
    mechanistic["expected_interaction_sign"],
    "",
    "**Expected nuisance-restoration direction:** " +
    mechanistic["expected_nuisance_direction"],
    "",
    "**Nuisance-direction prediction correct:** " +
    str(mechanistic["nuisance_prediction_correct"]),
    "",
    "## Permuted-patch negative controls",
    "",
    "These controls are descriptive only and do not "
    "establish statistical significance.",
    "",
]

for control in result["permuted_controls"]:
    name = control.get("model_key")
    if name and name in metrics:
        cm = metrics[name]
        recovery = (
            metrics["corrupted"]["shift_mse"]
            - cm["shift_mse"]
        ) / result["corruption_gap"]

        lines.append(
            f"- `{name}`: SHIFT MSE "
            f"{fnum(cm['shift_mse'])}; "
            f"recovery {fpct(recovery)}."
        )
    else:
        lines.append(
            f"- {control['arm']} control "
            f"{control['control']}: "
            f"{control['status']}."
        )

lines += [
    "",
    "## Scientific interpretation",
    "",
    "**Observed:** Both arms completed their permitted "
    "action sequences without using diagnostic calls. "
    "Neither achieved functional recovery. The "
    "diagnosis-first arm copied a public example, "
    "and the repair-only final patch worsened the patient.",
    "",
    "**Not established:** Whether the model could diagnose "
    "or repair this patient with a different interface, "
    "whether diagnosis-first instructions generally "
    "improve repair, or whether the observed behavior "
    "generalizes to other models or patients.",
    "",
    "**H1 versus H2:** This run supports neither "
    "successful mechanistic diagnosis nor successful "
    "repair without diagnosis. It does not discriminate "
    "between those positive-capability hypotheses.",
    "",
    "## Limitations and integrity qualifications",
    "",
    "1. One deliberately constructed synthetic patient "
    "and one teacher run per arm.",
    "2. Both arms requested zero diagnostic observations.",
    "3. The diagnosis example was present in the public "
    "prompt and reproduced verbatim in the commitment.",
    "4. Candidate C06 was selected using the prospectively "
    "frozen construction gates, which had been informed "
    "by the earlier EX0004 pilot.",
    "5. The original marshal-based executable fingerprint "
    "mismatched. Source AST and mock behavior were "
    "verified before exposure, but binary equivalence "
    "remains unresolved under amendment A1.",
    "6. Generated thinking tokens were not archived "
    "verbatim; action records and tool results were.",
    "7. The held-out dataset was opened once after "
    "both final submissions and scoring rules were locked.",
    "",
    "## Next experimental question",
    "",
    "A future, separately preregistered study could test "
    "whether literal schema examples cause copying and "
    "whether requiring an actual observation before "
    "diagnosis changes model behavior. That is a "
    "hypothesis for a new experiment, not an amendment "
    "to EX0004B.",
    "",
    "## Provenance",
    "",
    f"- Protocol SHA-256: `{B_PROTOCOL_SHA}`",
    f"- Patient SHA-256: `{B_CORRUPT_SHA}`",
    f"- Public prompt SHA-256: `{B_PROMPT_SHA}`",
    f"- Autonomous-loop SHA-256: `{B_LOOP_SHA}`",
    f"- Outcome-lock SHA-256: `{B_OUTCOME_LOCK_SHA}`",
    f"- Scoring-spec SHA-256: `{B_SEALED_SPEC_SHA}`",
    f"- Sealed report SHA-256: "
    f"`{B_MANIFEST['sealed_report_sha256']}`",
    f"- A1 amendment SHA-256: "
    f"`{B_MANIFEST['integrity_amendment_sha256']}`",
    "",
    "The archived JSON files, checkpoints, transcripts "
    "and event journals are the authoritative evidence. "
    "This narrative report is a derived summary.",
    "",
]

report_text = "\n".join(lines) + "\n"

REPORT_MD_PATH = ROOT / "EX0004B_FINAL_RESEARCH_REPORT.md"

def write_or_verify(path, content):
    path = Path(path)
    if path.exists():
        assert path.read_text() == content, (
            f"Existing artifact differs: {path}"
        )
    else:
        path.write_text(content)
    return b_file_sha(path)

report_md_sha = write_or_verify(
    REPORT_MD_PATH, report_text
)

# 4. Create the independent archive inventory.
inventory_path = ROOT / "EX0004B_ARCHIVE_INVENTORY.json"

source_files = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file() and p != inventory_path
)

assert all(not p.is_symlink() for p in source_files)

file_hashes = {
    p.relative_to(ROOT).as_posix(): b_file_sha(p)
    for p in source_files
}

inventory = {
    "experiment": EXPERIMENT_ID,
    "disposition": "COMPLETED_NEGATIVE_RESULT",
    "sealed_report_sha256":
        B_MANIFEST["sealed_report_sha256"],
    "research_report_sha256": report_md_sha,
    "file_count_excluding_inventory": len(file_hashes),
    "files": file_hashes,
}

inventory_text = json.dumps(
    inventory,
    indent=2,
    sort_keys=True,
    allow_nan=False
) + "\n"

inventory_sha = write_or_verify(
    inventory_path,
    inventory_text
)

# 5. Verify the archive inventory before packaging.
for rel, expected in file_hashes.items():
    assert b_file_sha(ROOT / rel) == expected

ARCHIVE_PATH = (
    ROOT.parent / "ALGN_EX0004B_FINAL_EVIDENCE.zip"
)

assert not ARCHIVE_PATH.exists(), (
    "Final archive already exists. Verify it rather "
    "than overwriting the existing package."
)

all_files = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file()
)

with zipfile.ZipFile(
    ARCHIVE_PATH,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in all_files:
        archive.write(
            path,
            arcname=(
                ROOT.name + "/" +
                path.relative_to(ROOT).as_posix()
            ),
        )

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None
    archived_names = set(archive.namelist())

    expected_names = {
        ROOT.name + "/" +
        p.relative_to(ROOT).as_posix()
        for p in all_files
    }

    assert archived_names == expected_names

    # Verify archived bytes against the inventory.
    for rel, expected in file_hashes.items():
        archived_bytes = archive.read(
            ROOT.name + "/" + rel
        )
        assert hashlib.sha256(
            archived_bytes
        ).hexdigest() == expected

archive_sha = b_file_sha(ARCHIVE_PATH)

# 6. Terminal research status.
print("Scientific disposition: COMPLETED NEGATIVE RESULT")
print("Diagnosis-first functional verdict: FAIL")
print("Repair-only functional verdict: FAIL")
print("Formal mechanistic verdict: FAIL")
print("Teacher diagnostic observations: 0 in both arms")
print("Sealed evaluation: COMPLETED ONCE")
print("Additional experimental execution: NONE")
print()
print("Research report:", REPORT_MD_PATH)
print("Report SHA-256:", report_md_sha)
print()
print("Files archived:", len(all_files))
print("Inventory SHA-256:", inventory_sha)
print("Archive:", ARCHIVE_PATH)
print("Archive SHA-256:", archive_sha)
print("Archive integrity: VERIFIED")
print()
print("✅ CELL 13: EX0004B CLOSED AND ARCHIVED")

# Download the ZIP after verifying the printed checks:
# from google.colab import files
# files.download(str(ARCHIVE_PATH))


🌒 EX0004B — FINAL RESEARCH ARCHIVE
Scientific disposition: COMPLETED NEGATIVE RESULT
Diagnosis-first functional verdict: FAIL
Repair-only functional verdict: FAIL
Formal mechanistic verdict: FAIL
Teacher diagnostic observations: 0 in both arms
Sealed evaluation: COMPLETED ONCE
Additional experimental execution: NONE

Research report: /content/ALGN_EX0004B/EX0004B_FINAL_RESEARCH_REPORT.md
Report SHA-256: db02252fae9731b4f9a5fa1dd0ca864809a353d5ff14abe75bc342076086b1cb

Files archived: 40
Inventory SHA-256: ab785f3bf1754042384553c585dde55709cc691a693e27e65ef0c41dc5b44ba1
Archive: /content/ALGN_EX0004B_FINAL_EVIDENCE.zip
Archive SHA-256: 21a2d12c013d4a173144f5a6d814f9741acefd07498d10110942db99afbe2a32
Archive integrity: VERIFIED

✅ CELL 13: EX0004B CLOSED AND ARCHIVED
